# Offline RAG Generation Prototype

## Decentralized Hiking AI


In [3]:
from pathlib import Path
import sqlite3
import pandas as pd
import re
import json

current = Path.cwd().resolve()

PROJECT_ROOT = None

for candidate in [current, *current.parents]:

    if (
        (candidate / "research").is_dir()
        and
        (candidate / "pubspec.yaml").is_file()
    ):
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Could not locate project root from: {current}"
    )

RESEARCH_DIR = PROJECT_ROOT / "research"

RAG_DIR = (
    RESEARCH_DIR
    / "rag"
)

DATABASE_FILE = (
    RAG_DIR
    / "database"
    / "offline_hiking_knowledge.db"
)

EVALUATION_DIR = (
    RAG_DIR
    / "evaluation"
)

EVALUATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("OFFLINE RAG GENERATION ENVIRONMENT")
print("=" * 60)

print(
    "Project root:",
    PROJECT_ROOT
)

print(
    "Database exists:",
    DATABASE_FILE.exists()
)

print(
    "Database:",
    DATABASE_FILE
)

OFFLINE RAG GENERATION ENVIRONMENT
Project root: /Users/mirshad/Desktop/Decentralized-Hiking-AI
Database exists: True
Database: /Users/mirshad/Desktop/Decentralized-Hiking-AI/research/rag/database/offline_hiking_knowledge.db


In [4]:
connection = sqlite3.connect(
    DATABASE_FILE
)

cursor = connection.cursor()

database_tables = pd.read_sql_query(
    """
    SELECT name, type
    FROM sqlite_master
    WHERE type IN ('table', 'view')
    ORDER BY name
    """,
    connection
)

document_count = cursor.execute(
    """
    SELECT COUNT(*)
    FROM documents
    """
).fetchone()[0]

chunk_count = cursor.execute(
    """
    SELECT COUNT(*)
    FROM knowledge_chunks
    """
).fetchone()[0]

fts_count = cursor.execute(
    """
    SELECT COUNT(*)
    FROM knowledge_chunks_fts
    """
).fetchone()[0]

print("OFFLINE DATABASE CONNECTION")
print("=" * 60)

print(
    "Documents:",
    document_count
)

print(
    "Knowledge chunks:",
    chunk_count
)

print(
    "FTS5 chunks:",
    fts_count
)

print()

display(
    database_tables
)

OFFLINE DATABASE CONNECTION
Documents: 27
Knowledge chunks: 1795
FTS5 chunks: 1795



,name,type
0,documents,table
1,knowledge_chunks,table
2,knowledge_chunks_fts,table
3,knowledge_chunks_fts_config,table
4,knowledge_chunks_fts_content,table
5,knowledge_chunks_fts_data,table
6,knowledge_chunks_fts_docsize,table
7,knowledge_chunks_fts_idx,table


## Generation-Time Retrieval

In [5]:
def build_fts_query(question):

    tokens = re.findall(
        r"[A-Za-z0-9]+",
        question.lower()
    )

    stop_words = {
        "a", "an", "the",
        "and", "or",
        "if", "is", "are",
        "to", "of", "in",
        "on", "for",
        "what", "how",
        "should", "do",
        "i", "me", "my",
        "someone", "hiker"
    }

    useful_tokens = [
        token
        for token in tokens
        if (
            token not in stop_words
            and
            len(token) > 2
        )
    ]

    if not useful_tokens:
        return None

    return " OR ".join(
        f'"{token}"'
        for token in useful_tokens
    )


def retrieve_for_generation(
    question,
    category,
    limit=4
):

    fts_query = build_fts_query(
        question
    )

    if not fts_query:
        return pd.DataFrame()

    sql = """
    SELECT
        kc.chunk_id,
        kc.source_id,
        d.title,
        d.organization,
        d.source_url,
        kc.categories,
        kc.content,
        bm25(knowledge_chunks_fts) AS rank

    FROM knowledge_chunks_fts

    JOIN knowledge_chunks kc
        ON kc.chunk_id =
           knowledge_chunks_fts.chunk_id

    JOIN documents d
        ON d.source_id =
           kc.source_id

    WHERE
        knowledge_chunks_fts MATCH ?

        AND
        kc.categories LIKE ?

    ORDER BY rank

    LIMIT ?
    """

    return pd.read_sql_query(
        sql,
        connection,
        params=[
            fts_query,
            f"%{category}%",
            limit
        ]
    )

In [6]:
test_question = (
    "What should I do if a hiker "
    "is bitten by a snake?"
)

test_category = "snakebite"

generation_context_test = (
    retrieve_for_generation(
        question=test_question,
        category=test_category,
        limit=4
    )
)

print("GENERATION RETRIEVAL TEST")
print("=" * 60)

print(
    "Question:",
    test_question
)

print(
    "Category:",
    test_category
)

print(
    "Results:",
    len(
        generation_context_test
    )
)

display(
    generation_context_test[
        [
            "chunk_id",
            "source_id",
            "organization",
            "rank"
        ]
    ]
)

GENERATION RETRIEVAL TEST
Question: What should I do if a hiker is bitten by a snake?
Category: snakebite
Results: 4


,chunk_id,source_id,organization,rank
0,CHK_01692,EXP_002,World Health Organization (WHO),-11.509080
1,CHK_01685,EXP_001,Sri Lanka Medical Association (SLMA),-11.106821
2,CHK_01684,EXP_001,Sri Lanka Medical Association (SLMA),-10.870893
3,CHK_01693,EXP_002,World Health Organization (WHO),-8.342459


## Authoritative Context Construction

In [7]:
def build_grounded_context(
    retrieved_chunks,
    max_chars=7000
):

    if retrieved_chunks.empty:
        return ""

    context_parts = []

    used_chars = 0

    for _, row in (
        retrieved_chunks.iterrows()
    ):

        block = (
            f"[SOURCE {row['source_id']}]\n"
            f"Organization: "
            f"{row['organization']}\n"
            f"Title: {row['title']}\n"
            f"Evidence:\n"
            f"{row['content'].strip()}\n"
        )

        if (
            used_chars
            + len(block)
            > max_chars
        ):
            break

        context_parts.append(
            block
        )

        used_chars += len(block)

    return "\n---\n".join(
        context_parts
    )


grounded_context = (
    build_grounded_context(
        generation_context_test
    )
)

print("GROUNDED CONTEXT")
print("=" * 60)

print(
    "Characters:",
    len(
        grounded_context
    )
)

print()

print(
    grounded_context[:5000]
)

GROUNDED CONTEXT
Characters: 4835

[SOURCE EXP_002]
Organization: World Health Organization (WHO)
Title: Snakebite envenoming (WHO health topic)
Evidence:
lead to permanent disability and even limb amputation. Children are at higher risk of severe effects due to lower body mass than adults. Most deaths and serious consequences from snake bites are entirely preventable by making safe and effective antivenoms more widely available and accessible, particularly in high-risk areas. High quality snake antivenoms are the only effective treatment to prevent or reverse most of the venomous effects of snake bites. Treatment Snake antivenoms are effective treatments to prevent or reverse most of the harmful effects of snakebite envenoming and are included in the WHO list of essential medicines. The availability and accessibility of these antivenoms, along with raising awareness on primary prevention methods among communities and health workers, are the best ways to limit serious consequences and 

In [8]:
def build_rag_prompt(
    question,
    category,
    context
):

    return f"""
You are the offline safety assistant for a hiking application.

You must answer using ONLY the authoritative evidence supplied in
AUTHORITATIVE CONTEXT.

Rules:

1. Do not invent medical, first-aid, weather, rescue, or emergency advice.

2. If the supplied evidence is insufficient to answer the question,
   explicitly say that the offline knowledge base does not contain enough
   information.

3. Prefer instructions directly supported by the retrieved sources.

4. Do not treat foreign emergency telephone numbers as valid for Sri Lanka
   unless the supplied evidence explicitly identifies them as Sri Lankan.

5. Do not recommend unsafe traditional remedies or unsupported treatments.

6. For potentially serious medical emergencies, clearly communicate the need
   for professional medical assistance when supported by the evidence.

7. Keep the answer concise and practical for a hiker.

8. Do not claim to diagnose the person.

9. At the end include:
   Sources used: <source IDs>

CATEGORY:
{category}

QUESTION:
{question}

AUTHORITATIVE CONTEXT:
{context}

ANSWER:
""".strip()


snakebite_prompt = build_rag_prompt(
    question=test_question,
    category=test_category,
    context=grounded_context
)

print("OFFLINE RAG PROMPT")
print("=" * 60)

print(
    snakebite_prompt[:8000]
)

OFFLINE RAG PROMPT
You are the offline safety assistant for a hiking application.

You must answer using ONLY the authoritative evidence supplied in
AUTHORITATIVE CONTEXT.

Rules:

1. Do not invent medical, first-aid, weather, rescue, or emergency advice.

2. If the supplied evidence is insufficient to answer the question,
   explicitly say that the offline knowledge base does not contain enough
   information.

3. Prefer instructions directly supported by the retrieved sources.

4. Do not treat foreign emergency telephone numbers as valid for Sri Lanka
   unless the supplied evidence explicitly identifies them as Sri Lankan.

5. Do not recommend unsafe traditional remedies or unsupported treatments.

6. For potentially serious medical emergencies, clearly communicate the need
   for professional medical assistance when supported by the evidence.

7. Keep the answer concise and practical for a hiker.

8. Do not claim to diagnose the person.

9. At the end include:
   Sources used: <sou

In [9]:
prompt_checks = {
    "contains_question":
        test_question
        in snakebite_prompt,

    "contains_context":
        len(
            grounded_context
        ) > 0,

    "contains_source_id":
        "[SOURCE "
        in snakebite_prompt,

    "contains_grounding_rule":
        "ONLY the authoritative evidence"
        in snakebite_prompt,

    "contains_localization_rule":
        "foreign emergency telephone numbers"
        in snakebite_prompt,

    "within_prototype_size":
        len(
            snakebite_prompt
        ) < 12000
}

print("RAG PROMPT VALIDATION")
print("=" * 60)

for check, result in (
    prompt_checks.items()
):
    print(
        f"{check}: {result}"
    )

print()

print(
    "All prompt checks passed:",
    all(
        prompt_checks.values()
    )
)

print(
    "Prompt characters:",
    len(
        snakebite_prompt
    )
)

RAG PROMPT VALIDATION
contains_question: True
contains_context: True
contains_source_id: True
contains_grounding_rule: True
contains_localization_rule: True
within_prototype_size: True

All prompt checks passed: True
Prompt characters: 5940


## Offline Language Model

In [10]:
MODEL_DIR = (
    RAG_DIR
    / "models"
)

MODEL_FILE = (
    MODEL_DIR
    / "Qwen3.5-0.8B-Q4_0.gguf"
)

print("OFFLINE MODEL")
print("=" * 60)

print(
    "Model exists:",
    MODEL_FILE.exists()
)

if MODEL_FILE.exists():

    model_size_mb = (
        MODEL_FILE.stat().st_size
        / (1024 ** 2)
    )

    print(
        "Model size:",
        f"{model_size_mb:.2f} MB"
    )

print(
    "Model path:",
    MODEL_FILE
)

OFFLINE MODEL
Model exists: True
Model size: 536.95 MB
Model path: /Users/mirshad/Desktop/Decentralized-Hiking-AI/research/rag/models/Qwen3.5-0.8B-Q4_0.gguf


In [11]:
from pathlib import Path
import subprocess

LLAMA_CLI = Path(
    "/opt/homebrew/bin/llama-cli"
)

print("LLAMA.CPP RUNTIME")
print("=" * 60)

print(
    "llama-cli:",
    LLAMA_CLI
)

print(
    "Executable exists:",
    LLAMA_CLI.exists()
)

if not LLAMA_CLI.exists():
    raise RuntimeError(
        f"llama-cli was not found at {LLAMA_CLI}"
    )

version_test = subprocess.run(
    [
        str(LLAMA_CLI),
        "--version"
    ],
    capture_output=True,
    text=True
)

print()
print(
    version_test.stdout[:2000]
)

if version_test.stderr:
    print(
        version_test.stderr[:1000]
    )

LLAMA.CPP RUNTIME
llama-cli: /opt/homebrew/bin/llama-cli
Executable exists: True


version: 0.7.0 (build 11669, commit 620d2225c)
built with AppleClang 21.0.0.21000334 for Darwin arm64



## First Fully Local RAG Generation Test

In [12]:
import subprocess
import time

generation_command = [
    str(LLAMA_CLI),

    "-m",
    str(MODEL_FILE),

    "-c",
    "4096",

    "-n",
    "300",

    "--temp",
    "0.2",

    "-p",
    snakebite_prompt
]

print("FIRST OFFLINE RAG GENERATION")
print("=" * 60)

print(
    "Question:",
    test_question
)

print(
    "Category:",
    test_category
)

print(
    "Retrieved chunks:",
    len(generation_context_test)
)

print()

start_time = time.perf_counter()

generation_result = subprocess.run(
    generation_command,
    capture_output=True,
    text=True,
    timeout=300
)

generation_seconds = (
    time.perf_counter()
    - start_time
)

print(
    "Return code:",
    generation_result.returncode
)

print(
    "Generation time:",
    f"{generation_seconds:.2f} seconds"
)

print()

print("MODEL OUTPUT")
print("=" * 60)

print(
    generation_result.stdout[-6000:]
)

if generation_result.returncode != 0:

    print()
    print("STDERR")
    print("=" * 60)

    print(
        generation_result.stderr[-5000:]
    )

FIRST OFFLINE RAG GENERATION
Question: What should I do if a hiker is bitten by a snake?
Category: snakebite
Retrieved chunks: 4



TimeoutExpired: Command '['/opt/homebrew/bin/llama-cli', '-m', '/Users/mirshad/Desktop/Decentralized-Hiking-AI/research/rag/models/Qwen3.5-0.8B-Q4_0.gguf', '-c', '4096', '-n', '300', '--temp', '0.2', '-p', 'You are the offline safety assistant for a hiking application.\n\nYou must answer using ONLY the authoritative evidence supplied in\nAUTHORITATIVE CONTEXT.\n\nRules:\n\n1. Do not invent medical, first-aid, weather, rescue, or emergency advice.\n\n2. If the supplied evidence is insufficient to answer the question,\n   explicitly say that the offline knowledge base does not contain enough\n   information.\n\n3. Prefer instructions directly supported by the retrieved sources.\n\n4. Do not treat foreign emergency telephone numbers as valid for Sri Lanka\n   unless the supplied evidence explicitly identifies them as Sri Lankan.\n\n5. Do not recommend unsafe traditional remedies or unsupported treatments.\n\n6. For potentially serious medical emergencies, clearly communicate the need\n   

## Runtime Diagnostic

In [13]:
import subprocess
import time

diagnostic_command = [
    str(LLAMA_CLI),

    "-m",
    str(MODEL_FILE),

    "-c",
    "1024",

    "-n",
    "20",

    "--temp",
    "0.2",

    "-p",
    "Reply with exactly: Offline model working."
]

print("MINIMAL LOCAL MODEL TEST")
print("=" * 60)

start_time = time.perf_counter()

try:

    diagnostic_result = subprocess.run(
        diagnostic_command,
        capture_output=True,
        text=True,
        timeout=120
    )

    diagnostic_seconds = (
        time.perf_counter()
        - start_time
    )

    print(
        "Return code:",
        diagnostic_result.returncode
    )

    print(
        "Time:",
        f"{diagnostic_seconds:.2f} seconds"
    )

    print()

    print("STDOUT")
    print("-" * 60)

    print(
        diagnostic_result.stdout[-3000:]
    )

    print()

    print("STDERR")
    print("-" * 60)

    print(
        diagnostic_result.stderr[-3000:]
    )

except subprocess.TimeoutExpired as error:

    diagnostic_seconds = (
        time.perf_counter()
        - start_time
    )

    print(
        "DIAGNOSTIC TIMED OUT"
    )

    print(
        "Time:",
        f"{diagnostic_seconds:.2f} seconds"
    )

    print()

    print("Partial STDOUT:")
    print(
        error.stdout[-3000:]
        if error.stdout
        else "None"
    )

    print()

    print("Partial STDERR:")
    print(
        error.stderr[-3000:]
        if error.stderr
        else "None"
    )

MINIMAL LOCAL MODEL TEST


KeyboardInterrupt: 

## Non-Interactive Generation Configuration

In [30]:
import subprocess
import time

single_prompt_command = [
    str(LLAMA_CLI),

    "-m",
    str(MODEL_FILE),

    "-c",
    "1024",

    "-n",
    "40",

    "--temp",
    "0.2",

    "-st",

    "-p",
    "Reply with exactly: Offline model working."
]

print("SINGLE-TURN MODEL TEST")
print("=" * 60)

start_time = time.perf_counter()

try:

    single_prompt_result = subprocess.run(
        single_prompt_command,
        capture_output=True,
        text=True,
        timeout=120
    )

    single_prompt_seconds = (
        time.perf_counter()
        - start_time
    )

    print(
        "Return code:",
        single_prompt_result.returncode
    )

    print(
        "Time:",
        f"{single_prompt_seconds:.2f} seconds"
    )

    print()

    print("STDOUT")
    print("-" * 60)

    print(
        single_prompt_result.stdout[-3000:]
    )

    print()

    print("STDERR")
    print("-" * 60)

    print(
        single_prompt_result.stderr[-2000:]
    )

except subprocess.TimeoutExpired as error:

    print(
        "SINGLE-TURN TEST TIMED OUT"
    )

    print(
        "Partial STDOUT:",
        error.stdout
    )

    print(
        "Partial STDERR:",
        error.stderr
    )

SINGLE-TURN MODEL TEST
Return code: 0
Time: 2.29 seconds

STDOUT
------------------------------------------------------------


Loading model... 

▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b11669-620d2225c
model      : /Users/mirshad/Desktop/Decentralized-Hiking-AI/research/rag/models/Qwen3.5-0.8B-Q4_0.gguf
ftype      : Q4_0
modalities : text

available commands:
  /exit or Ctrl+C     stop or exit
  /regen              regenerate the last response
  /clear              clear the chat history
  /read <file>        add a text file
  /glob <pattern>     add text files using globbing pattern



> Reply with exactly: Offline model working.

[Start thinking]

Thinking Process:

1.  **Analyze the Request:**
    *   Input: "Offline model working."
    *   Task: Reply with exactly

## First Grounded Offline RAG Answer

In [31]:
def generate_offline_answer(
    prompt,
    max_tokens=300,
    context_size=4096,
    temperature=0.2,
    timeout_seconds=120
):

    command = [
        str(LLAMA_CLI),

        "-m",
        str(MODEL_FILE),

        "-c",
        str(context_size),

        "-n",
        str(max_tokens),

        "--temp",
        str(temperature),

        "-st",

        "-p",
        prompt
    ]

    start_time = time.perf_counter()

    try:

        result = subprocess.run(
            command,
            capture_output=True,
            text=True,
            timeout=timeout_seconds
        )

        elapsed = (
            time.perf_counter()
            - start_time
        )

        return {
            "success":
                result.returncode == 0,

            "return_code":
                result.returncode,

            "time_seconds":
                elapsed,

            "stdout":
                result.stdout,

            "stderr":
                result.stderr
        }

    except subprocess.TimeoutExpired as error:

        elapsed = (
            time.perf_counter()
            - start_time
        )

        return {
            "success": False,
            "return_code": None,
            "time_seconds": elapsed,
            "stdout":
                error.stdout or "",

            "stderr":
                error.stderr or "",

            "timeout": True
        }


print(
    "Offline generation function ready."
)

Offline generation function ready.


In [32]:
snakebite_generation = (
    generate_offline_answer(
        prompt=snakebite_prompt,
        max_tokens=300,
        context_size=4096,
        temperature=0.2,
        timeout_seconds=120
    )
)

print("SNAKEBITE OFFLINE RAG TEST")
print("=" * 60)

print(
    "Success:",
    snakebite_generation[
        "success"
    ]
)

print(
    "Return code:",
    snakebite_generation[
        "return_code"
    ]
)

print(
    "Generation time:",
    f"{snakebite_generation['time_seconds']:.2f} seconds"
)

print()

print("RAW MODEL OUTPUT")
print("=" * 60)

print(
    snakebite_generation[
        "stdout"
    ][-8000:]
)

if snakebite_generation[
    "stderr"
]:

    print()
    print("STDERR")
    print("=" * 60)

    print(
        snakebite_generation[
            "stderr"
        ][-3000:]
    )

SNAKEBITE OFFLINE RAG TEST
Success: True
Return code: 0
Generation time: 7.36 seconds

RAW MODEL OUTPUT


Loading model... 

▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b11669-620d2225c
model      : /Users/mirshad/Desktop/Decentralized-Hiking-AI/research/rag/models/Qwen3.5-0.8B-Q4_0.gguf
ftype      : Q4_0
modalities : text

available commands:
  /exit or Ctrl+C     stop or exit
  /regen              regenerate the last response
  /clear              clear the chat history
  /read <file>        add a text file
  /glob <pattern>     add text files using globbing pattern



> You are the offline safety assistant for a hiking application.

You must answer using ONLY the authoritative evidence supplied in
AUTHORITATIVE CONTEXT.

Rules:

1. Do not invent medical, first-aid, weath

In [33]:
raw_output = (
    snakebite_generation[
        "stdout"
    ]
)

print("RAW OUTPUT INSPECTION")
print("=" * 60)

print(
    "Total characters:",
    len(raw_output)
)

print()

print("LAST 12000 CHARACTERS")
print("=" * 60)

print(
    raw_output[-12000:]
)

RAW OUTPUT INSPECTION
Total characters: 2521

LAST 12000 CHARACTERS


Loading model... 

▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b11669-620d2225c
model      : /Users/mirshad/Desktop/Decentralized-Hiking-AI/research/rag/models/Qwen3.5-0.8B-Q4_0.gguf
ftype      : Q4_0
modalities : text

available commands:
  /exit or Ctrl+C     stop or exit
  /regen              regenerate the last response
  /clear              clear the chat history
  /read <file>        add a text file
  /glob <pattern>     add text files using globbing pattern



> You are the offline safety assistant for a hiking application.

You must answer using ONLY the authoritative evidence supplied in
AUTHORITATIVE CONTEXT.

Rules:

1. Do not invent medical, first-aid, weather, rescue, or emergency advice.

2.

In [34]:
RESULTS_DIR = (
    RAG_DIR
    / "results"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

snakebite_raw_file = (
    RESULTS_DIR
    / "snakebite_first_offline_generation.txt"
)

snakebite_raw_file.write_text(
    raw_output,
    encoding="utf-8"
)

print("GENERATION OUTPUT SAVED")
print("=" * 60)

print(
    "File:",
    snakebite_raw_file
)

print(
    "Exists:",
    snakebite_raw_file.exists()
)

print(
    "Characters:",
    len(raw_output)
)

GENERATION OUTPUT SAVED
File: /Users/mirshad/Desktop/Decentralized-Hiking-AI/research/rag/results/snakebite_first_offline_generation.txt
Exists: True
Characters: 2521


In [35]:
output_lower = (
    raw_output.lower()
)

audit_terms = [
    "exp_001",
    "exp_002",
    "hospital",
    "immobil",
    "carry",
    "walk",
    "traditional",
    "1990",
    "911",
    "999",
    "112"
]

print("FIRST GENERATION CONTENT AUDIT")
print("=" * 60)

for term in audit_terms:

    print(
        f"{term}:",
        term in output_lower
    )

FIRST GENERATION CONTENT AUDIT
exp_001: False
exp_002: True
hospital: True
immobil: True
carry: False
walk: False
traditional: True
1990: False
911: False
999: False
112: False


## Subprocess-Safe Output Capture

In [36]:
def generate_offline_answer(
    prompt,
    max_tokens=300,
    context_size=4096,
    temperature=0.2,
    timeout_seconds=120
):

    command = [
        str(LLAMA_CLI),

        "-m",
        str(MODEL_FILE),

        "-c",
        str(context_size),

        "-n",
        str(max_tokens),

        "--temp",
        str(temperature),

        "-st",

        "--simple-io",

        "--no-display-prompt",

        "--log-disable",

        "-p",
        prompt
    ]

    start_time = time.perf_counter()

    try:

        result = subprocess.run(
            command,
            capture_output=True,
            text=True,
            timeout=timeout_seconds
        )

        elapsed = (
            time.perf_counter()
            - start_time
        )

        return {
            "success":
                result.returncode == 0,

            "return_code":
                result.returncode,

            "time_seconds":
                elapsed,

            "stdout":
                result.stdout.strip(),

            "stderr":
                result.stderr.strip()
        }

    except subprocess.TimeoutExpired as error:

        elapsed = (
            time.perf_counter()
            - start_time
        )

        return {
            "success": False,
            "return_code": None,
            "time_seconds": elapsed,
            "stdout":
                error.stdout or "",

            "stderr":
                error.stderr or "",

            "timeout": True
        }


print(
    "Subprocess-safe offline generator ready."
)

Subprocess-safe offline generator ready.


In [37]:
clean_test = generate_offline_answer(
    prompt=(
        "Reply only with the words: "
        "Offline model working."
    ),
    max_tokens=30,
    context_size=1024,
    temperature=0.0,
    timeout_seconds=60
)

print("CLEAN OUTPUT TEST")
print("=" * 60)

print(
    "Success:",
    clean_test["success"]
)

print(
    "Return code:",
    clean_test["return_code"]
)

print(
    "Time:",
    f"{clean_test['time_seconds']:.2f} seconds"
)

print()

print("CAPTURED OUTPUT")
print("=" * 60)

print(
    repr(
        clean_test["stdout"]
    )
)

print()

print(
    "Output characters:",
    len(
        clean_test["stdout"]
    )
)

print(
    "STDERR characters:",
    len(
        clean_test["stderr"]
    )
)

CLEAN OUTPUT TEST
Success: True
Return code: 0
Time: 2.41 seconds

CAPTURED OUTPUT
'Loading model... \n\n▄▄ ▄▄\n██ ██\n██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄\n██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██\n██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀\n                                    ██    ██\n                                    ▀▀    ▀▀\n\nbuild      : b11669-620d2225c\nmodel      : /Users/mirshad/Desktop/Decentralized-Hiking-AI/research/rag/models/Qwen3.5-0.8B-Q4_0.gguf\nftype      : Q4_0\nmodalities : text\n\navailable commands:\n  /exit or Ctrl+C     stop or exit\n  /regen              regenerate the last response\n  /clear              clear the chat history\n  /read <file>        add a text file\n  /glob <pattern>     add text files using globbing pattern\n\n\n\n> Reply only with the words: Offline model working.\n\n[Start thinking]\n\nThinking Process:\n\n1.  **Analyze the Request:**\n    *   Input: "Offline model working."\n    *   Constraint:\n\n[ Prompt: 281.

In [38]:
snakebite_generation_clean = (
    generate_offline_answer(
        prompt=snakebite_prompt,
        max_tokens=300,
        context_size=4096,
        temperature=0.2,
        timeout_seconds=120
    )
)

print("CLEAN SNAKEBITE RAG GENERATION")
print("=" * 60)

print(
    "Success:",
    snakebite_generation_clean[
        "success"
    ]
)

print(
    "Return code:",
    snakebite_generation_clean[
        "return_code"
    ]
)

print(
    "Generation time:",
    f"{snakebite_generation_clean['time_seconds']:.2f} seconds"
)

print()

print("GENERATED ANSWER")
print("=" * 60)

print(
    snakebite_generation_clean[
        "stdout"
    ]
)

CLEAN SNAKEBITE RAG GENERATION
Success: True
Return code: 0
Generation time: 7.31 seconds

GENERATED ANSWER
Loading model... 

▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b11669-620d2225c
model      : /Users/mirshad/Desktop/Decentralized-Hiking-AI/research/rag/models/Qwen3.5-0.8B-Q4_0.gguf
ftype      : Q4_0
modalities : text

available commands:
  /exit or Ctrl+C     stop or exit
  /regen              regenerate the last response
  /clear              clear the chat history
  /read <file>        add a text file
  /glob <pattern>     add text files using globbing pattern



> You are the offline safety assistant for a hiking application.

You must answer using ONLY the authoritative evidence supplied in
AUTHORITATIVE CONTEXT.

Rules:

1. Do not invent medical, first-aid, wea

In [39]:
clean_answer = (
    snakebite_generation_clean[
        "stdout"
    ]
)

clean_answer_lower = (
    clean_answer.lower()
)

answer_audit = {
    "mentions_EXP_001":
        "exp_001"
        in clean_answer_lower,

    "mentions_EXP_002":
        "exp_002"
        in clean_answer_lower,

    "mentions_hospital":
        "hospital"
        in clean_answer_lower,

    "mentions_immobilization":
        "immobil"
        in clean_answer_lower,

    "mentions_carry":
        "carry"
        in clean_answer_lower,

    "mentions_traditional_methods":
        "traditional"
        in clean_answer_lower,

    "mentions_1990":
        "1990"
        in clean_answer_lower,

    "mentions_911":
        "911"
        in clean_answer_lower,

    "mentions_999":
        "999"
        in clean_answer_lower,

    "mentions_112":
        "112"
        in clean_answer_lower
}


print("GENERATED ANSWER AUDIT")
print("=" * 60)

for check, result in (
    answer_audit.items()
):

    print(
        f"{check}: {result}"
    )

GENERATED ANSWER AUDIT
mentions_EXP_001: True
mentions_EXP_002: True
mentions_hospital: False
mentions_immobilization: False
mentions_carry: False
mentions_traditional_methods: True
mentions_1990: False
mentions_911: False
mentions_999: False
mentions_112: False


## Disabling Reasoning for Safety Response Generation

In [40]:
no_think_test = generate_offline_answer(
    prompt=(
        "/no_think\n"
        "Reply only with exactly these words: "
        "Offline model working."
    ),
    max_tokens=60,
    context_size=1024,
    temperature=0.0,
    timeout_seconds=60
)

print("NO-THINK OUTPUT TEST")
print("=" * 60)

print(
    "Success:",
    no_think_test["success"]
)

print(
    "Return code:",
    no_think_test["return_code"]
)

print(
    "Time:",
    f"{no_think_test['time_seconds']:.2f} seconds"
)

print()

print("CAPTURED OUTPUT")
print("=" * 60)

print(
    no_think_test["stdout"]
)

NO-THINK OUTPUT TEST
Success: True
Return code: 0
Time: 2.52 seconds

CAPTURED OUTPUT
Loading model... 

▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b11669-620d2225c
model      : /Users/mirshad/Desktop/Decentralized-Hiking-AI/research/rag/models/Qwen3.5-0.8B-Q4_0.gguf
ftype      : Q4_0
modalities : text

available commands:
  /exit or Ctrl+C     stop or exit
  /regen              regenerate the last response
  /clear              clear the chat history
  /read <file>        add a text file
  /glob <pattern>     add text files using globbing pattern



> /no_think
Reply only with exactly these words: Offline model working.

[Start thinking]

Thinking Process:

1.  **Analyze the Request:**
    *   Input: "no_think" (This is a common prompt injection attempt to bypass safety f

## Persistent Local Inference Server

In [41]:
import json
import urllib.request
import urllib.error
import time

LLAMA_SERVER_URL = (
    "http://127.0.0.1:8080"
)

health_url = (
    LLAMA_SERVER_URL
    + "/health"
)

print("LOCAL LLM SERVER")
print("=" * 60)

try:

    with urllib.request.urlopen(
        health_url,
        timeout=10
    ) as response:

        health_body = (
            response
            .read()
            .decode("utf-8")
        )

        print(
            "HTTP status:",
            response.status
        )

        print(
            "Response:",
            health_body
        )

except Exception as error:

    print(
        "Server check failed:",
        repr(error)
    )

LOCAL LLM SERVER
HTTP status: 200
Response: {"status":"ok"}


In [42]:
def generate_with_local_server(
    prompt,
    max_tokens=220,
    temperature=0.1,
    timeout_seconds=120
):

    endpoint = (
        LLAMA_SERVER_URL
        + "/completion"
    )

    payload = {
        "prompt": prompt,
        "n_predict": max_tokens,
        "temperature": temperature,
        "stream": False
    }

    request = urllib.request.Request(
        endpoint,
        data=json.dumps(
            payload
        ).encode("utf-8"),
        headers={
            "Content-Type":
                "application/json"
        },
        method="POST"
    )

    start_time = (
        time.perf_counter()
    )

    try:

        with urllib.request.urlopen(
            request,
            timeout=timeout_seconds
        ) as response:

            response_body = (
                response
                .read()
                .decode("utf-8")
            )

            data = json.loads(
                response_body
            )

        elapsed = (
            time.perf_counter()
            - start_time
        )

        return {
            "success": True,
            "time_seconds": elapsed,
            "content":
                data.get(
                    "content",
                    ""
                ).strip(),
            "response":
                data
        }

    except Exception as error:

        elapsed = (
            time.perf_counter()
            - start_time
        )

        return {
            "success": False,
            "time_seconds": elapsed,
            "content": "",
            "error": repr(error)
        }


print(
    "Local server generation function ready."
)

Local server generation function ready.


In [43]:
server_test = (
    generate_with_local_server(
        prompt=(
            "Answer directly without reasoning. "
            "Reply exactly with: "
            "Offline model working."
        ),
        max_tokens=60,
        temperature=0.0,
        timeout_seconds=60
    )
)

print("LOCAL SERVER GENERATION TEST")
print("=" * 60)

print(
    "Success:",
    server_test["success"]
)

print(
    "Time:",
    f"{server_test['time_seconds']:.2f} seconds"
)

print()

print("CONTENT")
print("=" * 60)

print(
    repr(
        server_test["content"]
    )
)

if not server_test["success"]:

    print()

    print(
        "Error:",
        server_test.get(
            "error"
        )
    )

LOCAL SERVER GENERATION TEST
Success: True
Time: 1.90 seconds

CONTENT
'Offline model working.\nOffline model working.\nOffline model working.\nOffline model working.\nOffline model working.\nOffline model working.\nOffline model working.\nOffline model working.\nOffline model working.\nOffline model working.\nOffline model working.\nOffline model working.'


## End-to-End Offline RAG Generation

In [44]:
def build_grounded_rag_prompt(
    question,
    category,
    context
):

    return f"""
You are an offline hiking safety assistant.

Answer the QUESTION using ONLY the AUTHORITATIVE CONTEXT provided below.

STRICT RULES:
1. Do not use facts that are not present in the context.
2. Do not invent emergency telephone numbers.
3. Do not diagnose the hiker.
4. Give short, practical safety instructions.
5. If the context is insufficient, reply exactly:
   "The offline knowledge base does not contain enough information to answer this safely."
6. Do not show reasoning or a thinking process.
7. End the answer with:
   Sources: <source IDs actually used>
8. After the Sources line, output exactly:
   END_OF_ANSWER

CATEGORY:
{category}

QUESTION:
{question}

AUTHORITATIVE CONTEXT:
{context}

ANSWER:
""".strip()


snakebite_server_prompt = (
    build_grounded_rag_prompt(
        question=test_question,
        category=test_category,
        context=grounded_context
    )
)

print("SNAKEBITE SERVER PROMPT")
print("=" * 60)

print(
    "Characters:",
    len(snakebite_server_prompt)
)

print(
    "Contains EXP_001:",
    "EXP_001"
    in snakebite_server_prompt
)

print(
    "Contains EXP_002:",
    "EXP_002"
    in snakebite_server_prompt
)

SNAKEBITE SERVER PROMPT
Characters: 5574
Contains EXP_001: True
Contains EXP_002: True


In [45]:
snakebite_server_result = (
    generate_with_local_server(
        prompt=snakebite_server_prompt,
        max_tokens=300,
        temperature=0.1,
        timeout_seconds=60
    )
)

print("END-TO-END SNAKEBITE RAG")
print("=" * 60)

print(
    "Success:",
    snakebite_server_result[
        "success"
    ]
)

print(
    "Generation time:",
    f"{snakebite_server_result['time_seconds']:.2f} seconds"
)

print()

print("RAW GENERATED CONTENT")
print("=" * 60)

print(
    snakebite_server_result[
        "content"
    ]
)

END-TO-END SNAKEBITE RAG
Success: True
Generation time: 5.33 seconds

RAW GENERATED CONTENT
<think>

</think>

Sources: <source IDs actually used>
END_OF_ANSWER

What should I do if a hiker is bitten by a snake?

Immediately move away from the area where the bite occurred. Remove anything tight from around the bitten part of the body to avoid harm if swelling occurs. Reassure the victim, as most venomous snake bites do not cause immediate death. Immobilize the person completely and transport the person to a health facility as soon as possible. Apply pressure at the bite site with a pressure pad. Avoid traditional first aid methods or herbal medicines. Paracetamol may be given for local pain. Place the person on their left side in the recovery position. Closely monitor airway and breathing and be ready to resuscitate if necessary.

Sources: <source IDs actually used>
END_OF_ANSWER

Wait, I need to check the source IDs. The prompt says "End the answer with: Sources: <source IDs actually 

In [46]:
raw_snakebite_answer = (
    snakebite_server_result[
        "content"
    ]
)

if "END_OF_ANSWER" in raw_snakebite_answer:

    snakebite_answer = (
        raw_snakebite_answer
        .split(
            "END_OF_ANSWER",
            1
        )[0]
        .strip()
    )

else:

    snakebite_answer = (
        raw_snakebite_answer.strip()
    )


print("CLEAN SNAKEBITE ANSWER")
print("=" * 60)

print(
    snakebite_answer
)

CLEAN SNAKEBITE ANSWER
<think>

</think>

Sources: <source IDs actually used>


In [47]:
answer_lower = (
    snakebite_answer.lower()
)

snakebite_audit = {
    "answer_generated":
        len(
            snakebite_answer
        ) > 0,

    "mentions_EXP_001":
        "exp_001"
        in answer_lower,

    "mentions_EXP_002":
        "exp_002"
        in answer_lower,

    "mentions_hospital":
        "hospital"
        in answer_lower,

    "mentions_keep_still_or_immobilize":
        (
            "keep"
            in answer_lower
            and
            "still"
            in answer_lower
        )
        or
        "immobil"
        in answer_lower,

    "mentions_remove_tight_items":
        (
            "tight"
            in answer_lower
            or
            "ring"
            in answer_lower
            or
            "constrict"
            in answer_lower
        ),

    "mentions_avoid_traditional_methods":
        (
            "traditional"
            in answer_lower
            or
            "herbal"
            in answer_lower
        ),

    "foreign_911_present":
        "911"
        in answer_lower,

    "foreign_999_present":
        "999"
        in answer_lower,

    "foreign_112_present":
        "112"
        in answer_lower,

    "unsupported_1990_present":
        "1990"
        in answer_lower
}


print("SNAKEBITE ANSWER AUDIT")
print("=" * 60)

for check, result in (
    snakebite_audit.items()
):

    print(
        f"{check}: {result}"
    )

SNAKEBITE ANSWER AUDIT
answer_generated: True
mentions_EXP_001: False
mentions_EXP_002: False
mentions_hospital: False
mentions_keep_still_or_immobilize: False
mentions_remove_tight_items: False
mentions_avoid_traditional_methods: False
foreign_911_present: False
foreign_999_present: False
foreign_112_present: False
unsupported_1990_present: False


## Deterministic Source Attribution and Answer Extraction

In [48]:
snakebite_question = (
    "What should I do if a hiker is bitten by a snake?"
)

snakebite_category = "snakebite"

snakebite_results = retrieve_knowledge(
    question=snakebite_question,
    category=snakebite_category,
    limit=4
)

print("SNAKEBITE RETRIEVAL REFRESH")
print("=" * 60)

print(
    "Results:",
    len(snakebite_results)
)

display(
    snakebite_results[
        [
            "chunk_id",
            "source_id",
            "organization",
            "rank"
        ]
    ]
)

NameError: name 'retrieve_knowledge' is not defined

# Unified Offline RAG Pipeline

In [49]:
import sqlite3
import pandas as pd
import re
from pathlib import Path

PROJECT_ROOT = Path.cwd()

while (
    PROJECT_ROOT.name
    != "Decentralized-Hiking-AI"
    and
    PROJECT_ROOT.parent
    != PROJECT_ROOT
):
    PROJECT_ROOT = PROJECT_ROOT.parent


RAG_DIR = (
    PROJECT_ROOT
    / "research"
    / "rag"
)

DATABASE_FILE = (
    RAG_DIR
    / "database"
    / "offline_hiking_knowledge.db"
)


rag_connection = sqlite3.connect(
    DATABASE_FILE
)


print("UNIFIED RAG DATABASE")
print("=" * 60)

print(
    "Database:",
    DATABASE_FILE
)

print(
    "Exists:",
    DATABASE_FILE.exists()
)

print(
    "Connection ready:",
    rag_connection is not None
)

UNIFIED RAG DATABASE
Database: /Users/mirshad/Desktop/Decentralized-Hiking-AI/research/rag/database/offline_hiking_knowledge.db
Exists: True
Connection ready: True


In [50]:
def retrieve_offline_knowledge(
    question,
    category=None,
    limit=4
):

    search_terms = re.findall(
        r"[A-Za-z0-9]+",
        question.lower()
    )

    search_terms = [
        term
        for term in search_terms
        if len(term) >= 3
    ]

    fts_query = " OR ".join(
        search_terms
    )

    if not fts_query:
        return pd.DataFrame()

    if category:

        sql = """
        SELECT
            kc.chunk_id,
            kc.source_id,
            d.organization,
            d.title,
            kc.category,
            kc.content,
            bm25(knowledge_chunks_fts) AS rank

        FROM knowledge_chunks_fts

        JOIN knowledge_chunks kc
            ON knowledge_chunks_fts.chunk_id
            = kc.chunk_id

        JOIN documents d
            ON kc.source_id
            = d.source_id

        WHERE
            knowledge_chunks_fts
            MATCH ?

            AND kc.category = ?

        ORDER BY rank

        LIMIT ?
        """

        params = (
            fts_query,
            category,
            limit
        )

    else:

        sql = """
        SELECT
            kc.chunk_id,
            kc.source_id,
            d.organization,
            d.title,
            kc.category,
            kc.content,
            bm25(knowledge_chunks_fts) AS rank

        FROM knowledge_chunks_fts

        JOIN knowledge_chunks kc
            ON knowledge_chunks_fts.chunk_id
            = kc.chunk_id

        JOIN documents d
            ON kc.source_id
            = d.source_id

        WHERE
            knowledge_chunks_fts
            MATCH ?

        ORDER BY rank

        LIMIT ?
        """

        params = (
            fts_query,
            limit
        )

    return pd.read_sql_query(
        sql,
        rag_connection,
        params=params
    )


print(
    "Unified retrieval function ready."
)

Unified retrieval function ready.


In [51]:
def prepare_rag_context(
    results
):

    if results.empty:

        return "", []

    context_parts = []

    source_ids = list(
        dict.fromkeys(
            results[
                "source_id"
            ].tolist()
        )
    )

    for _, row in results.iterrows():

        context_parts.append(
            f"""
[SOURCE {row['source_id']}]
Organization: {row['organization']}
Title: {row['title']}
Evidence:
{row['content']}
""".strip()
        )

    context = (
        "\n\n---\n\n"
        .join(
            context_parts
        )
    )

    return (
        context,
        source_ids
    )


print(
    "Context builder ready."
)

Context builder ready.


In [52]:
offline_rag_answer(
    question,
    category
)

NameError: name 'offline_rag_answer' is not defined

In [ ]:
offline_rag_answer(
    "What should I do if I am lost on a hiking trail?",
    "lost_navigation"
)

NameError: name 'offline_rag_answer' is not defined

In [ ]:
offline_rag_answer(
    "How can I get emergency medical help in Sri Lanka?",
    "sri_lanka_emergency"
)

NameError: name 'offline_rag_answer' is not defined

## Unified Offline RAG Answer Function

In [53]:
def offline_rag_answer(
    question,
    category,
    limit=4
):

    # ---------------------------------
    # 1. Retrieve evidence from SQLite
    # ---------------------------------

    results = retrieve_offline_knowledge(
        question=question,
        category=category,
        limit=limit
    )

    if results.empty:

        return {
            "success": False,
            "question": question,
            "category": category,
            "answer": (
                "The offline knowledge base does not contain "
                "enough information to answer this safely."
            ),
            "source_ids": [],
            "retrieved_chunks": 0,
            "generation_time": 0
        }


    # ---------------------------------
    # 2. Build grounded context
    # ---------------------------------

    context, source_ids = (
        prepare_rag_context(
            results
        )
    )


    # ---------------------------------
    # 3. Build generation prompt
    # ---------------------------------

    prompt = f"""
You are an offline hiking safety assistant.

Answer the QUESTION using ONLY the AUTHORITATIVE CONTEXT.

Rules:
- Give concise practical guidance.
- Do not add information absent from the context.
- Do not invent emergency telephone numbers.
- Do not diagnose the hiker.
- Do not provide source IDs.
- Do not explain your reasoning.
- If the evidence is insufficient, output exactly:
  INSUFFICIENT_OFFLINE_EVIDENCE

QUESTION:
{question}

AUTHORITATIVE CONTEXT:
{context}

FINAL ANSWER:
""".strip()


    # ---------------------------------
    # 4. Local LLM generation
    # ---------------------------------

    generation = (
        generate_with_local_server(
            prompt=prompt,
            max_tokens=220,
            temperature=0.0,
            timeout_seconds=60
        )
    )


    if not generation["success"]:

        return {
            "success": False,
            "question": question,
            "category": category,
            "answer": (
                "Local generation failed."
            ),
            "source_ids": source_ids,
            "retrieved_chunks":
                len(results),
            "generation_time":
                generation[
                    "time_seconds"
                ]
        }


    # ---------------------------------
    # 5. Clean generated text
    # ---------------------------------

    answer = (
        generation["content"]
        .strip()
    )

    answer = re.sub(
        r"<think>.*?</think>",
        "",
        answer,
        flags=(
            re.DOTALL
            |
            re.IGNORECASE
        )
    ).strip()


    # ---------------------------------
    # 6. Insufficient-evidence handling
    # ---------------------------------

    if (
        "INSUFFICIENT_OFFLINE_EVIDENCE"
        in answer
    ):

        answer = (
            "The offline knowledge base does not contain "
            "enough information to answer this safely."
        )


    # ---------------------------------
    # 7. Deterministic provenance
    # ---------------------------------

    final_output = (
        answer
        + "\n\nSources: "
        + ", ".join(source_ids)
    )


    return {
        "success": True,
        "question": question,
        "category": category,
        "answer": answer,
        "final_output": final_output,
        "source_ids": source_ids,
        "retrieved_chunks":
            len(results),
        "generation_time":
            generation[
                "time_seconds"
            ],
        "retrieval_results":
            results
    }


print(
    "Unified offline RAG function ready."
)

Unified offline RAG function ready.


In [54]:
lost_test = (
    offline_rag_answer(
        question=(
            "What should I do if I am lost "
            "on a hiking trail?"
        ),
        category="lost_navigation"
    )
)

print("LOST / NAVIGATION RAG TEST")
print("=" * 60)

print(
    "Success:",
    lost_test["success"]
)

print(
    "Retrieved chunks:",
    lost_test[
        "retrieved_chunks"
    ]
)

print(
    "Source IDs:",
    lost_test[
        "source_ids"
    ]
)

print(
    "Generation time:",
    f"{lost_test['generation_time']:.2f} seconds"
)

print()

print("ANSWER")
print("=" * 60)

print(
    lost_test.get(
        "final_output",
        lost_test["answer"]
    )
)

DatabaseError: Execution failed on sql '
        SELECT
            kc.chunk_id,
            kc.source_id,
            d.organization,
            d.title,
            kc.category,
            kc.content,
            bm25(knowledge_chunks_fts) AS rank

        FROM knowledge_chunks_fts

        JOIN knowledge_chunks kc
            ON knowledge_chunks_fts.chunk_id
            = kc.chunk_id

        JOIN documents d
            ON kc.source_id
            = d.source_id

        WHERE
            knowledge_chunks_fts
            MATCH ?

            AND kc.category = ?

        ORDER BY rank

        LIMIT ?
        ': no such column: kc.category

In [55]:
print("SQLITE DATABASE SCHEMA")
print("=" * 60)

tables_to_check = [
    "documents",
    "knowledge_chunks",
    "knowledge_chunks_fts"
]

for table in tables_to_check:

    print()
    print(f"TABLE: {table}")
    print("-" * 60)

    schema = pd.read_sql_query(
        f"PRAGMA table_info({table})",
        rag_connection
    )

    print(
        schema[
            [
                "name",
                "type"
            ]
        ].to_string(
            index=False
        )
    )

SQLITE DATABASE SCHEMA

TABLE: documents
------------------------------------------------------------
           name type
      source_id TEXT
          title TEXT
   organization TEXT
     categories TEXT
     source_url TEXT
      file_type TEXT
     local_path TEXT
country_context TEXT

TABLE: knowledge_chunks
------------------------------------------------------------
              name    type
          chunk_id    TEXT
         source_id    TEXT
source_chunk_index INTEGER
        categories    TEXT
        word_count INTEGER
           content    TEXT

TABLE: knowledge_chunks_fts
------------------------------------------------------------
      name type
  chunk_id     
 source_id     
categories     
   content     


## Schema-Aligned Offline Retrieval

In [56]:
def retrieve_offline_knowledge(
    question,
    category=None,
    limit=4
):

    search_terms = re.findall(
        r"[A-Za-z0-9]+",
        question.lower()
    )

    # Remove very short terms that add little retrieval value.
    search_terms = [
        term
        for term in search_terms
        if len(term) >= 3
    ]

    # Remove duplicates while preserving order.
    search_terms = list(
        dict.fromkeys(
            search_terms
        )
    )

    if not search_terms:
        return pd.DataFrame()

    fts_query = " OR ".join(
        search_terms
    )

    if category:

        sql = """
        SELECT
            kc.chunk_id,
            kc.source_id,
            d.organization,
            d.title,
            kc.categories,
            kc.content,
            kc.word_count,
            bm25(knowledge_chunks_fts) AS rank

        FROM knowledge_chunks_fts

        JOIN knowledge_chunks kc
            ON knowledge_chunks_fts.chunk_id
            = kc.chunk_id

        JOIN documents d
            ON kc.source_id
            = d.source_id

        WHERE
            knowledge_chunks_fts MATCH ?

            AND (
                ',' || REPLACE(kc.categories, ' ', '') || ','
            ) LIKE ?

        ORDER BY rank

        LIMIT ?
        """

        params = (
            fts_query,
            f"%,{category},%",
            limit
        )

    else:

        sql = """
        SELECT
            kc.chunk_id,
            kc.source_id,
            d.organization,
            d.title,
            kc.categories,
            kc.content,
            kc.word_count,
            bm25(knowledge_chunks_fts) AS rank

        FROM knowledge_chunks_fts

        JOIN knowledge_chunks kc
            ON knowledge_chunks_fts.chunk_id
            = kc.chunk_id

        JOIN documents d
            ON kc.source_id
            = d.source_id

        WHERE
            knowledge_chunks_fts MATCH ?

        ORDER BY rank

        LIMIT ?
        """

        params = (
            fts_query,
            limit
        )

    return pd.read_sql_query(
        sql,
        rag_connection,
        params=params
    )


print(
    "Schema-aligned retrieval function ready."
)

Schema-aligned retrieval function ready.


In [57]:
lost_retrieval_test = (
    retrieve_offline_knowledge(
        question=(
            "What should I do if I am lost "
            "on a hiking trail?"
        ),
        category="lost_navigation",
        limit=4
    )
)

print("LOST / NAVIGATION RETRIEVAL")
print("=" * 60)

print(
    "Results:",
    len(lost_retrieval_test)
)

if not lost_retrieval_test.empty:

    display(
        lost_retrieval_test[
            [
                "chunk_id",
                "source_id",
                "organization",
                "categories",
                "rank"
            ]
        ]
    )

LOST / NAVIGATION RETRIEVAL
Results: 3


,chunk_id,source_id,organization,categories,rank
0,CHK_01740,EXP_011,Department of Conservation New Zealand (DOC),lost_navigation,-10.228048
1,CHK_01741,EXP_011,Department of Conservation New Zealand (DOC),lost_navigation,-8.647170
2,CHK_01739,EXP_011,Department of Conservation New Zealand (DOC),lost_navigation,-6.037228


In [58]:
lightning_retrieval_test = (
    retrieve_offline_knowledge(
        question=(
            "What should a hiker do "
            "during lightning?"
        ),
        category="lightning",
        limit=4
    )
)

print("LIGHTNING RETRIEVAL")
print("=" * 60)

print(
    "Results:",
    len(lightning_retrieval_test)
)

if not lightning_retrieval_test.empty:

    display(
        lightning_retrieval_test[
            [
                "chunk_id",
                "source_id",
                "organization",
                "categories",
                "rank"
            ]
        ]
    )

LIGHTNING RETRIEVAL
Results: 4


,chunk_id,source_id,organization,categories,rank
0,CHK_01767,EXP_014,UK Met Office,lightning,-8.001075
1,CHK_01766,EXP_014,UK Met Office,lightning,-7.976517
2,CHK_01770,EXP_014,UK Met Office,lightning,-3.124290
3,CHK_01774,EXP_014,UK Met Office,lightning,-2.500143


## Multi-Category End-to-End Offline RAG Validation

In [59]:
lost_rag_test = offline_rag_answer(
    question=(
        "What should I do if I am lost "
        "on a hiking trail?"
    ),
    category="lost_navigation"
)

print("END-TO-END LOST / NAVIGATION RAG")
print("=" * 60)

print(
    "Success:",
    lost_rag_test["success"]
)

print(
    "Retrieved chunks:",
    lost_rag_test["retrieved_chunks"]
)

print(
    "Source IDs:",
    lost_rag_test["source_ids"]
)

print(
    "Generation time:",
    f"{lost_rag_test['generation_time']:.2f} seconds"
)

print()
print("FINAL ANSWER")
print("=" * 60)

print(
    lost_rag_test.get(
        "final_output",
        lost_rag_test["answer"]
    )
)

END-TO-END LOST / NAVIGATION RAG
Success: True
Retrieved chunks: 3
Source IDs: ['EXP_011']
Generation time: 2.93 seconds

FINAL ANSWER
The offline knowledge base does not contain enough information to answer this safely.

Sources: EXP_011


In [60]:
lightning_rag_test = offline_rag_answer(
    question=(
        "What should a hiker do "
        "during lightning?"
    ),
    category="lightning"
)

print("END-TO-END LIGHTNING RAG")
print("=" * 60)

print(
    "Success:",
    lightning_rag_test["success"]
)

print(
    "Retrieved chunks:",
    lightning_rag_test["retrieved_chunks"]
)

print(
    "Source IDs:",
    lightning_rag_test["source_ids"]
)

print(
    "Generation time:",
    f"{lightning_rag_test['generation_time']:.2f} seconds"
)

print()
print("FINAL ANSWER")
print("=" * 60)

print(
    lightning_rag_test.get(
        "final_output",
        lightning_rag_test["answer"]
    )
)

END-TO-END LIGHTNING RAG
Success: True
Retrieved chunks: 4
Source IDs: ['EXP_014']
Generation time: 2.80 seconds

FINAL ANSWER
The offline knowledge base does not contain enough information to answer this safely.

Sources: EXP_014


In [61]:
multi_category_audit = pd.DataFrame(
    [
        {
            "test": "lost_navigation",
            "success":
                lost_rag_test["success"],
            "retrieved_chunks":
                lost_rag_test["retrieved_chunks"],
            "unique_sources":
                len(
                    lost_rag_test["source_ids"]
                ),
            "has_answer":
                bool(
                    lost_rag_test["answer"].strip()
                ),
            "has_sources":
                bool(
                    lost_rag_test["source_ids"]
                ),
            "generation_seconds":
                lost_rag_test["generation_time"]
        },

        {
            "test": "lightning",
            "success":
                lightning_rag_test["success"],
            "retrieved_chunks":
                lightning_rag_test["retrieved_chunks"],
            "unique_sources":
                len(
                    lightning_rag_test["source_ids"]
                ),
            "has_answer":
                bool(
                    lightning_rag_test["answer"].strip()
                ),
            "has_sources":
                bool(
                    lightning_rag_test["source_ids"]
                ),
            "generation_seconds":
                lightning_rag_test["generation_time"]
        }
    ]
)

print("MULTI-CATEGORY RAG AUDIT")
print("=" * 60)

display(
    multi_category_audit
)

print()

print(
    "All tests successful:",
    multi_category_audit[
        "success"
    ].all()
)

print(
    "All tests retrieved evidence:",
    (
        multi_category_audit[
            "retrieved_chunks"
        ] > 0
    ).all()
)

print(
    "All tests produced answers:",
    multi_category_audit[
        "has_answer"
    ].all()
)

print(
    "All tests have provenance:",
    multi_category_audit[
        "has_sources"
    ].all()
)

MULTI-CATEGORY RAG AUDIT


,test,success,retrieved_chunks,unique_sources,has_answer,has_sources,generation_seconds
0,lost_navigation,True,3,1,True,True,2.925534
1,lightning,True,4,1,True,True,2.802781



All tests successful: True
All tests retrieved evidence: True
All tests produced answers: True
All tests have provenance: True


## Grounded Answer Quality Validation


In [62]:
def inspect_retrieved_evidence(
    results,
    label
):

    print(label)
    print("=" * 70)

    for i, row in results.iterrows():

        print()
        print(
            f"SOURCE: {row['source_id']}"
        )

        print(
            f"ORGANIZATION: {row['organization']}"
        )

        print(
            f"TITLE: {row['title']}"
        )

        print(
            f"CATEGORIES: {row['categories']}"
        )

        print()

        print(
            row["content"][:1500]
        )

        print()
        print("-" * 70)


inspect_retrieved_evidence(
    lost_retrieval_test,
    "LOST / NAVIGATION EVIDENCE"
)

inspect_retrieved_evidence(
    lightning_retrieval_test,
    "LIGHTNING EVIDENCE"
)

LOST / NAVIGATION EVIDENCE

SOURCE: EXP_011
ORGANIZATION: Department of Conservation New Zealand (DOC)
TITLE: Maps for walking, tramping and hunting in New Zealand
CATEGORIES: lost_navigation

all map apps are suitable for hiking, tramping or hunting in New Zealand. Choose a reliable option that: works offline shows all New Zealand backcountry areas includes a topographic map view shows tracks, terrain and other important features clearly Many places in New Zealand do not have cell phone or internet reception. We recommend downloading maps and checking they work offline before your trip. If you’re relying on your phone for navigation, make sure you will have enough battery life and carry a paper backup in case your phone gets damaged or runs out of charge. Reduce your risk of getting lost Before you go on your trip, prepare for emergencies by: Learning navigation and survival skills. Researching your route and planning your trip thoroughly. Taking the right gear, including a distress b

In [63]:
def build_grounded_answer_prompt(
    question,
    context
):

    return f"""
You are an offline hiking safety assistant.

Your task is to answer the QUESTION using only the AUTHORITATIVE CONTEXT below.

IMPORTANT RULES:

1. Use only information explicitly supported by the context.

2. If the context contains relevant safety instructions, provide those
   instructions. The context does not need to answer every possible aspect
   of the question.

3. Do not reject the question merely because the available information is
   incomplete. Give the useful instructions that are supported.

4. Use INSUFFICIENT_OFFLINE_EVIDENCE only when the context contains no
   useful information for answering the question.

5. Do not invent facts.

6. Do not invent emergency telephone numbers.

7. Do not add foreign emergency numbers unless they are explicitly relevant
   to the user's location and supported by the context.

8. Do not provide diagnoses.

9. Do not mention source IDs in the answer. Source attribution is added
   separately by the application.

10. Do not explain your reasoning.

11. Give a short, practical answer suitable for a hiker using a phone
    offline during an emergency.

AUTHORITATIVE CONTEXT:

{context}

QUESTION:

{question}

If useful evidence exists, answer the question now.

FINAL ANSWER:
""".strip()


print(
    "Improved grounded-answer prompt ready."
)

Improved grounded-answer prompt ready.


In [64]:
def offline_rag_answer_v2(
    question,
    category,
    limit=4
):

    results = retrieve_offline_knowledge(
        question=question,
        category=category,
        limit=limit
    )

    if results.empty:

        return {
            "success": False,
            "pipeline_success": True,
            "answer_generated": False,
            "fallback_used": True,
            "question": question,
            "category": category,
            "answer": (
                "The offline knowledge base does not contain "
                "enough information to answer this safely."
            ),
            "final_output": (
                "The offline knowledge base does not contain "
                "enough information to answer this safely."
            ),
            "source_ids": [],
            "retrieved_chunks": 0,
            "generation_time": 0
        }


    context, source_ids = (
        prepare_rag_context(
            results
        )
    )


    prompt = build_grounded_answer_prompt(
        question=question,
        context=context
    )


    generation = generate_with_local_server(
        prompt=prompt,
        max_tokens=220,
        temperature=0.0,
        timeout_seconds=60
    )


    if not generation["success"]:

        return {
            "success": False,
            "pipeline_success": False,
            "answer_generated": False,
            "fallback_used": False,
            "question": question,
            "category": category,
            "answer": "Local generation failed.",
            "final_output": "Local generation failed.",
            "source_ids": source_ids,
            "retrieved_chunks": len(results),
            "generation_time":
                generation["time_seconds"]
        }


    answer = (
        generation["content"]
        .strip()
    )


    answer = re.sub(
        r"<think>.*?</think>",
        "",
        answer,
        flags=(
            re.DOTALL
            |
            re.IGNORECASE
        )
    ).strip()


    fallback_used = (
        "INSUFFICIENT_OFFLINE_EVIDENCE"
        in answer
    )


    if fallback_used:

        answer = (
            "The offline knowledge base does not contain "
            "enough information to answer this safely."
        )


    answer_generated = (
        bool(answer.strip())
        and
        not fallback_used
    )


    final_output = (
        answer
        + "\n\nSources: "
        + ", ".join(source_ids)
    )


    return {
        "success": answer_generated,
        "pipeline_success": True,
        "answer_generated": answer_generated,
        "fallback_used": fallback_used,
        "question": question,
        "category": category,
        "answer": answer,
        "final_output": final_output,
        "source_ids": source_ids,
        "retrieved_chunks": len(results),
        "generation_time":
            generation["time_seconds"],
        "retrieval_results": results
    }


print(
    "Offline RAG v2 ready."
)

Offline RAG v2 ready.


In [65]:
v2_tests = [
    {
        "name": "lost_navigation",
        "question": (
            "What should I do if I am lost "
            "on a hiking trail?"
        ),
        "category": "lost_navigation"
    },
    {
        "name": "lightning",
        "question": (
            "What should a hiker do "
            "during lightning?"
        ),
        "category": "lightning"
    }
]


v2_results = {}


for test in v2_tests:

    print()
    print(
        test["name"].upper()
    )
    print("=" * 70)

    result = offline_rag_answer_v2(
        question=test["question"],
        category=test["category"]
    )

    v2_results[
        test["name"]
    ] = result

    print(
        "Pipeline success:",
        result["pipeline_success"]
    )

    print(
        "Answer generated:",
        result["answer_generated"]
    )

    print(
        "Fallback used:",
        result["fallback_used"]
    )

    print(
        "Retrieved chunks:",
        result["retrieved_chunks"]
    )

    print(
        "Sources:",
        result["source_ids"]
    )

    print(
        "Generation time:",
        f"{result['generation_time']:.2f} seconds"
    )

    print()
    print("ANSWER")
    print("-" * 70)

    print(
        result["final_output"]
    )


LOST_NAVIGATION
Pipeline success: True
Answer generated: True
Fallback used: False
Retrieved chunks: 3
Sources: ['EXP_011']
Generation time: 4.62 seconds

ANSWER
----------------------------------------------------------------------
**STOP AND ASSESS**

You have provided a context containing multiple sources from the Department of Conservation (DOC) and the Mountain Safety Council. However, the context is incomplete and contains conflicting or contradictory information regarding safety protocols.

Specifically, **Source EXP_011** states that you should "Call 111 if you don't return as planned" and "Give them plenty of detail and ask them to call 111 if you don't return as planned." However, **Source EXP_012** (which appears to be the continuation of the same document or a separate source) explicitly advises: "If you get lost Don't keep going into unfamiliar terrain hoping to find your way. Stop and assess the situation. Unless you are certain of the way out, stay where you are. Make y

## Localization and Grounding Guardrails

In [66]:
FOREIGN_EMERGENCY_NUMBERS = {
    "111",
    "911",
    "999"
}


def apply_output_guardrails(
    answer,
    retrieved_source_ids,
    category
):

    cleaned = answer.strip()

    # Remove generated source-ID claims.
    # Provenance is attached separately by the application.
    cleaned = re.sub(
        r"\b(?:SRC|EXP)_\d{3}\b",
        "[retrieved source]",
        cleaned,
        flags=re.IGNORECASE
    )

    # Remove Qwen thinking blocks if present.
    cleaned = re.sub(
        r"<think>.*?</think>",
        "",
        cleaned,
        flags=(
            re.DOTALL
            |
            re.IGNORECASE
        )
    ).strip()

    foreign_numbers_found = []

    # Foreign emergency numbers must not be presented
    # as local Sri Lankan emergency instructions.
    if category != "sri_lanka_emergency":

        for number in (
            FOREIGN_EMERGENCY_NUMBERS
        ):

            if re.search(
                rf"\b{re.escape(number)}\b",
                cleaned
            ):

                foreign_numbers_found.append(
                    number
                )

                cleaned = re.sub(
                    rf"\b{re.escape(number)}\b",
                    "[foreign emergency number omitted]",
                    cleaned
                )

    return {
        "answer": cleaned,
        "foreign_numbers_found":
            foreign_numbers_found,
        "retrieved_source_ids":
            retrieved_source_ids
    }


print(
    "Localization and provenance guardrails ready."
)

Localization and provenance guardrails ready.


In [67]:
lost_guardrail_test = (
    apply_output_guardrails(
        answer=(
            v2_results[
                "lost_navigation"
            ]["answer"]
        ),
        retrieved_source_ids=(
            v2_results[
                "lost_navigation"
            ]["source_ids"]
        ),
        category="lost_navigation"
    )
)


print("LOST / NAVIGATION GUARDRAIL TEST")
print("=" * 70)

print(
    "Retrieved sources:",
    lost_guardrail_test[
        "retrieved_source_ids"
    ]
)

print(
    "Foreign numbers detected:",
    lost_guardrail_test[
        "foreign_numbers_found"
    ]
)

print()

print("GUARDED ANSWER")
print("-" * 70)

print(
    lost_guardrail_test[
        "answer"
    ]
)

LOST / NAVIGATION GUARDRAIL TEST
Retrieved sources: ['EXP_011']
Foreign numbers detected: ['111']

GUARDED ANSWER
----------------------------------------------------------------------
**STOP AND ASSESS**

You have provided a context containing multiple sources from the Department of Conservation (DOC) and the Mountain Safety Council. However, the context is incomplete and contains conflicting or contradictory information regarding safety protocols.

Specifically, **Source [retrieved source]** states that you should "Call [foreign emergency number omitted] if you don't return as planned" and "Give them plenty of detail and ask them to call [foreign emergency number omitted] if you don't return as planned." However, **Source [retrieved source]** (which appears to be the continuation of the same document or a separate source) explicitly advises: "If you get lost Don't keep going into unfamiliar terrain hoping to find your way. Stop and assess the situation. Unless you are certain of the 

In [68]:
lightning_evidence = (
    retrieve_offline_knowledge(
        question=(
            "What should a hiker do "
            "during lightning?"
        ),
        category="lightning",
        limit=4
    )
)


print("LIGHTNING GROUNDING REVIEW")
print("=" * 70)


for index, row in (
    lightning_evidence.iterrows()
):

    print()
    print(
        f"CHUNK: {row['chunk_id']}"
    )

    print(
        f"SOURCE: {row['source_id']}"
    )

    print(
        f"ORGANIZATION: "
        f"{row['organization']}"
    )

    print()

    print(
        row["content"]
    )

    print()
    print("-" * 70)

LIGHTNING GROUNDING REVIEW

CHUNK: CHK_01767
SOURCE: EXP_014
ORGANIZATION: UK Met Office

heavy falling or blowing snow or fog is combined with deep snow cover. Thunderstorms Lightning strikes. Lightning strikes are likely to cause danger, especially on exposed tops and ridges Severe chill effect When cooling is greatly increased by strong winds, normally exceeding 35 mph. mean wind speed. If driving rain or sleet is also forecast winds may be lower than 35 mph. Please note that other factors such as mist, sunshine, precipitation and snow cover will either mitigate or enhance cooling. Considerable risk of hypothermia and frostbite, unless adequately equipped and protected. NB: Even though winds in valleys or glens may be light, sudden and very marked increases can occur at higher levels. Heavy persistent rain Persistent, heavy, rain (or sleet), lasting at least 3 hours, or over 40 mm expected in 24 hours, starting from dusk of the previous day. Risk of major discomfort unless adequatel

## Pre-Generation Context Safety Filtering

In [69]:
def sanitize_context_for_generation(
    context,
    category
):

    # Sri Lankan emergency evidence is handled
    # separately and must preserve verified numbers.
    if category == "sri_lanka_emergency":
        return context

    sentences = re.split(
        r"(?<=[.!?])\s+",
        context
    )

    safe_sentences = []

    for sentence in sentences:

        contains_foreign_number = any(
            re.search(
                rf"\b{re.escape(number)}\b",
                sentence
            )
            for number
            in FOREIGN_EMERGENCY_NUMBERS
        )

        if not contains_foreign_number:
            safe_sentences.append(
                sentence
            )

    return " ".join(
        safe_sentences
    ).strip()


print(
    "Pre-generation context sanitizer ready."
)

Pre-generation context sanitizer ready.


In [70]:
lost_context, lost_sources = (
    prepare_rag_context(
        lost_retrieval_test
    )
)

safe_lost_context = (
    sanitize_context_for_generation(
        context=lost_context,
        category="lost_navigation"
    )
)


print("LOST CONTEXT SANITIZATION")
print("=" * 70)

print(
    "Original contains 111:",
    bool(
        re.search(
            r"\b111\b",
            lost_context
        )
    )
)

print(
    "Sanitized contains 111:",
    bool(
        re.search(
            r"\b111\b",
            safe_lost_context
        )
    )
)

print(
    "Original characters:",
    len(lost_context)
)

print(
    "Sanitized characters:",
    len(safe_lost_context)
)

print()
print("SANITIZED CONTEXT")
print("-" * 70)

print(
    safe_lost_context
)

LOST CONTEXT SANITIZATION
Original contains 111: True
Sanitized contains 111: False
Original characters: 3318
Sanitized characters: 2923

SANITIZED CONTEXT
----------------------------------------------------------------------
[SOURCE EXP_011]
Organization: Department of Conservation New Zealand (DOC)
Title: Maps for walking, tramping and hunting in New Zealand
Evidence:
all map apps are suitable for hiking, tramping or hunting in New Zealand. Choose a reliable option that: works offline shows all New Zealand backcountry areas includes a topographic map view shows tracks, terrain and other important features clearly Many places in New Zealand do not have cell phone or internet reception. We recommend downloading maps and checking they work offline before your trip. If you’re relying on your phone for navigation, make sure you will have enough battery life and carry a paper backup in case your phone gets damaged or runs out of charge. Reduce your risk of getting lost Before you go on yo

In [71]:
def offline_rag_answer_v3(
    question,
    category,
    limit=4
):

    results = retrieve_offline_knowledge(
        question=question,
        category=category,
        limit=limit
    )

    if results.empty:

        fallback = (
            "The offline knowledge base does not contain "
            "enough information to answer this safely."
        )

        return {
            "success": False,
            "pipeline_success": True,
            "answer_generated": False,
            "fallback_used": True,
            "question": question,
            "category": category,
            "answer": fallback,
            "final_output": fallback,
            "source_ids": [],
            "retrieved_chunks": 0,
            "generation_time": 0
        }


    context, source_ids = (
        prepare_rag_context(
            results
        )
    )


    safe_context = (
        sanitize_context_for_generation(
            context=context,
            category=category
        )
    )


    if not safe_context.strip():

        fallback = (
            "The offline knowledge base does not contain "
            "enough localized information to answer this safely."
        )

        return {
            "success": False,
            "pipeline_success": True,
            "answer_generated": False,
            "fallback_used": True,
            "question": question,
            "category": category,
            "answer": fallback,
            "final_output": (
                fallback
                + "\n\nSources: "
                + ", ".join(source_ids)
            ),
            "source_ids": source_ids,
            "retrieved_chunks": len(results),
            "generation_time": 0
        }


    prompt = build_grounded_answer_prompt(
        question=question,
        context=safe_context
    )


    generation = (
        generate_with_local_server(
            prompt=prompt,
            max_tokens=220,
            temperature=0.0,
            timeout_seconds=60
        )
    )


    if not generation["success"]:

        return {
            "success": False,
            "pipeline_success": False,
            "answer_generated": False,
            "fallback_used": False,
            "question": question,
            "category": category,
            "answer": "Local generation failed.",
            "final_output": "Local generation failed.",
            "source_ids": source_ids,
            "retrieved_chunks": len(results),
            "generation_time":
                generation["time_seconds"]
        }


    answer = re.sub(
        r"<think>.*?</think>",
        "",
        generation["content"],
        flags=(
            re.DOTALL
            |
            re.IGNORECASE
        )
    ).strip()


    fallback_used = (
        "INSUFFICIENT_OFFLINE_EVIDENCE"
        in answer
    )


    if fallback_used:

        answer = (
            "The offline knowledge base does not contain "
            "enough information to answer this safely."
        )


    guarded = apply_output_guardrails(
        answer=answer,
        retrieved_source_ids=source_ids,
        category=category
    )


    answer = guarded["answer"]


    answer_generated = (
        bool(answer.strip())
        and
        not fallback_used
    )


    final_output = (
        answer
        + "\n\nSources: "
        + ", ".join(source_ids)
    )


    return {
        "success": answer_generated,
        "pipeline_success": True,
        "answer_generated": answer_generated,
        "fallback_used": fallback_used,
        "question": question,
        "category": category,
        "answer": answer,
        "final_output": final_output,
        "source_ids": source_ids,
        "retrieved_chunks": len(results),
        "foreign_numbers_detected":
            guarded["foreign_numbers_found"],
        "generation_time":
            generation["time_seconds"],
        "retrieval_results": results
    }


print(
    "Offline RAG v3 ready."
)

Offline RAG v3 ready.


In [72]:
lost_v3 = offline_rag_answer_v3(
    question=(
        "What should I do if I am lost "
        "on a hiking trail?"
    ),
    category="lost_navigation"
)


print("LOST NAVIGATION — RAG V3")
print("=" * 70)

print(
    "Pipeline success:",
    lost_v3["pipeline_success"]
)

print(
    "Answer generated:",
    lost_v3["answer_generated"]
)

print(
    "Fallback used:",
    lost_v3["fallback_used"]
)

print(
    "Retrieved chunks:",
    lost_v3["retrieved_chunks"]
)

print(
    "Sources:",
    lost_v3["source_ids"]
)

print(
    "Foreign numbers in final output:",
    lost_v3[
        "foreign_numbers_detected"
    ]
)

print()

print("FINAL ANSWER")
print("-" * 70)

print(
    lost_v3["final_output"]
)

LOST NAVIGATION — RAG V3
Pipeline success: True
Answer generated: True
Fallback used: False
Retrieved chunks: 3
Sources: ['EXP_011']
Foreign numbers in final output: []

FINAL ANSWER
----------------------------------------------------------------------
**Action: Stop and assess the situation.**

If you are lost on a hiking trail, the most critical safety step is to stop and assess the situation. Do not try to find your way again unless you are certain of the way out. Stay where you are. Make yourself and your group safe and comfortable – get warm, eat and drink, make a shelter, use first aid if needed. Call for help and make yourself visible. Learn more about what to do if you get lost on the Mountain Safety Council website.

**Action: Download maps and check they work offline.**

If you are relying on your phone for navigation, make sure you will have enough battery life and carry a paper backup in case your phone gets damaged or runs out of charge. Download maps and check they work 

## Country-Context Localization

In [73]:
country_context_audit = pd.read_sql_query(
    """
    SELECT
        source_id,
        organization,
        title,
        country_context
    FROM documents
    WHERE source_id IN (?, ?)
    """,
    rag_connection,
    params=(
        "EXP_011",
        "EXP_014"
    )
)

print("SOURCE COUNTRY CONTEXT")
print("=" * 70)

display(
    country_context_audit
)

SOURCE COUNTRY CONTEXT


,source_id,organization,title,country_context
0,EXP_011,Department of Conservation New Zealand (DOC),"Maps for walking, tramping and hunting in New ...",New Zealand
1,EXP_014,UK Met Office,Mountain forecast guide,United Kingdom


In [74]:
country_distribution = pd.read_sql_query(
    """
    SELECT
        COALESCE(
            NULLIF(
                TRIM(country_context),
                ''
            ),
            '[not specified]'
        ) AS country_context,
        COUNT(*) AS document_count
    FROM documents
    GROUP BY
        COALESCE(
            NULLIF(
                TRIM(country_context),
                ''
            ),
            '[not specified]'
        )
    ORDER BY document_count DESC
    """,
    rag_connection
)

print("OFFLINE CORPUS COUNTRY-CONTEXT DISTRIBUTION")
print("=" * 70)

display(
    country_distribution
)

OFFLINE CORPUS COUNTRY-CONTEXT DISTRIBUTION


,country_context,document_count
0,[not specified],11
1,United Kingdom,7
2,Sri Lanka,3
3,United States,2
4,United Kingdom (England),2
5,New Zealand,1
6,International,1


In [75]:
sri_lanka_sources = pd.read_sql_query(
    """
    SELECT
        source_id,
        organization,
        title,
        categories,
        country_context
    FROM documents
    WHERE
        LOWER(country_context)
        LIKE '%sri lanka%'
    ORDER BY source_id
    """,
    rag_connection
)

print("SRI LANKA-CONTEXT OFFLINE SOURCES")
print("=" * 70)

print(
    "Sources found:",
    len(sri_lanka_sources)
)

display(
    sri_lanka_sources
)

SRI LANKA-CONTEXT OFFLINE SOURCES
Sources found: 3


,source_id,organization,title,categories,country_context
0,EXP_001,Sri Lanka Medical Association (SLMA),Snakebite - Prevention and First Aid (Revised ...,snakebite,Sri Lanka
1,EXP_012,National Building Research Organisation Sri La...,Are you prepared to face monsoon induced lands...,landslide,Sri Lanka
2,EXP_015,1990 Suwa Seriya Foundation,FAQ - 1990 Suwa Seriya Foundation,sri_lanka_emergency,Sri Lanka


## Metadata-Aware Localization Layer

In [76]:
def retrieve_offline_knowledge_v2(
    question,
    category=None,
    limit=4
):

    search_terms = re.findall(
        r"[A-Za-z0-9]+",
        question.lower()
    )

    search_terms = [
        term
        for term in search_terms
        if len(term) >= 3
    ]

    search_terms = list(
        dict.fromkeys(search_terms)
    )

    if not search_terms:
        return pd.DataFrame()

    fts_query = " OR ".join(
        search_terms
    )

    if category:

        sql = """
        SELECT
            kc.chunk_id,
            kc.source_id,
            d.organization,
            d.title,
            d.country_context,
            kc.categories,
            kc.content,
            kc.word_count,
            bm25(knowledge_chunks_fts) AS rank

        FROM knowledge_chunks_fts

        JOIN knowledge_chunks kc
            ON knowledge_chunks_fts.chunk_id
            = kc.chunk_id

        JOIN documents d
            ON kc.source_id
            = d.source_id

        WHERE
            knowledge_chunks_fts MATCH ?

            AND (
                ',' || REPLACE(kc.categories, ' ', '') || ','
            ) LIKE ?

        ORDER BY rank

        LIMIT ?
        """

        params = (
            fts_query,
            f"%,{category},%",
            limit
        )

    else:

        sql = """
        SELECT
            kc.chunk_id,
            kc.source_id,
            d.organization,
            d.title,
            d.country_context,
            kc.categories,
            kc.content,
            kc.word_count,
            bm25(knowledge_chunks_fts) AS rank

        FROM knowledge_chunks_fts

        JOIN knowledge_chunks kc
            ON knowledge_chunks_fts.chunk_id
            = kc.chunk_id

        JOIN documents d
            ON kc.source_id
            = d.source_id

        WHERE
            knowledge_chunks_fts MATCH ?

        ORDER BY rank

        LIMIT ?
        """

        params = (
            fts_query,
            limit
        )

    return pd.read_sql_query(
        sql,
        rag_connection,
        params=params
    )


print(
    "Metadata-aware retrieval ready."
)

Metadata-aware retrieval ready.


In [77]:
def prepare_localized_rag_context(
    results
):

    context_parts = []
    source_ids = []

    for _, row in results.iterrows():

        source_id = row["source_id"]

        if source_id not in source_ids:
            source_ids.append(source_id)

        country = row["country_context"]

        if (
            pd.isna(country)
            or
            not str(country).strip()
        ):
            country = "Not specified"

        content = str(
            row["content"]
        ).strip()

        # Remove foreign emergency-number sentences
        # before they reach the model.
        if str(country).strip().lower() != "sri lanka":

            content = (
                sanitize_context_for_generation(
                    context=content,
                    category=str(
                        row["categories"]
                    )
                )
            )

        context_parts.append(
            f"""
[SOURCE {source_id}]
Organization: {row['organization']}
Title: {row['title']}
Country context: {country}
Category: {row['categories']}
Evidence:
{content}
""".strip()
        )

    context = "\n\n---\n\n".join(
        context_parts
    )

    return context, source_ids


print(
    "Localized context builder ready."
)

Localized context builder ready.


In [78]:
def build_localized_grounded_prompt(
    question,
    context
):

    return f"""
You are an offline hiking safety assistant intended for use in Sri Lanka.

Answer the QUESTION using only the AUTHORITATIVE CONTEXT below.

SAFETY AND LOCALIZATION RULES:

1. Use only claims supported by the retrieved context.

2. Give practical instructions that directly answer the question.

3. Sri Lankan evidence may be used as local operational guidance.

4. Evidence from another country may be used only for generally transferable
   hiking or safety principles.

5. Do not present foreign country-specific services, emergency numbers,
   organizations, websites, map products, or geographic instructions as
   recommendations for Sri Lanka.

6. Do not convert or replace foreign information with an invented Sri Lankan
   equivalent.

7. Do not invent emergency telephone numbers.

8. Do not mention source IDs or discuss the retrieved documents.

9. Do not tell the user to visit a foreign website.

10. Do not provide a diagnosis.

11. Prefer short, direct emergency actions over background explanation.

12. If useful evidence exists, provide the supported useful guidance even if
    the context is incomplete.

13. Use exactly the text INSUFFICIENT_OFFLINE_EVIDENCE only when the retrieved
    context contains no useful evidence for answering the question safely.

AUTHORITATIVE CONTEXT:

{context}

QUESTION:

{question}

FINAL ANSWER:
""".strip()


print(
    "Localized grounded prompt ready."
)

Localized grounded prompt ready.


In [79]:
def offline_rag_answer_v4(
    question,
    category,
    limit=4
):

    results = retrieve_offline_knowledge_v2(
        question=question,
        category=category,
        limit=limit
    )

    if results.empty:

        fallback = (
            "The offline knowledge base does not contain "
            "enough information to answer this safely."
        )

        return {
            "success": False,
            "pipeline_success": True,
            "answer_generated": False,
            "fallback_used": True,
            "question": question,
            "category": category,
            "answer": fallback,
            "final_output": fallback,
            "source_ids": [],
            "retrieved_chunks": 0,
            "generation_time": 0
        }


    context, source_ids = (
        prepare_localized_rag_context(
            results
        )
    )


    prompt = (
        build_localized_grounded_prompt(
            question=question,
            context=context
        )
    )


    generation = (
        generate_with_local_server(
            prompt=prompt,
            max_tokens=180,
            temperature=0.0,
            timeout_seconds=60
        )
    )


    if not generation["success"]:

        return {
            "success": False,
            "pipeline_success": False,
            "answer_generated": False,
            "fallback_used": False,
            "question": question,
            "category": category,
            "answer": "Local generation failed.",
            "final_output": "Local generation failed.",
            "source_ids": source_ids,
            "retrieved_chunks": len(results),
            "generation_time":
                generation["time_seconds"]
        }


    answer = re.sub(
        r"<think>.*?</think>",
        "",
        generation["content"],
        flags=(
            re.DOTALL
            |
            re.IGNORECASE
        )
    ).strip()


    fallback_used = (
        "INSUFFICIENT_OFFLINE_EVIDENCE"
        in answer
    )


    if fallback_used:

        answer = (
            "The offline knowledge base does not contain "
            "enough information to answer this safely."
        )


    guarded = apply_output_guardrails(
        answer=answer,
        retrieved_source_ids=source_ids,
        category=category
    )

    answer = guarded["answer"]


    answer_generated = (
        bool(answer.strip())
        and
        not fallback_used
    )


    final_output = (
        answer
        + "\n\nSources: "
        + ", ".join(source_ids)
    )


    return {
        "success": answer_generated,
        "pipeline_success": True,
        "answer_generated": answer_generated,
        "fallback_used": fallback_used,
        "question": question,
        "category": category,
        "answer": answer,
        "final_output": final_output,
        "source_ids": source_ids,
        "retrieved_chunks": len(results),
        "country_contexts": (
            results[
                "country_context"
            ]
            .fillna(
                "Not specified"
            )
            .unique()
            .tolist()
        ),
        "foreign_numbers_detected":
            guarded[
                "foreign_numbers_found"
            ],
        "generation_time":
            generation[
                "time_seconds"
            ],
        "retrieval_results":
            results
    }


print(
    "Offline RAG v4 ready."
)

Offline RAG v4 ready.


In [80]:
lost_v4 = offline_rag_answer_v4(
    question=(
        "What should I do if I am lost "
        "on a hiking trail?"
    ),
    category="lost_navigation"
)


print("LOST NAVIGATION — LOCALIZED RAG V4")
print("=" * 70)

print(
    "Pipeline success:",
    lost_v4["pipeline_success"]
)

print(
    "Answer generated:",
    lost_v4["answer_generated"]
)

print(
    "Fallback used:",
    lost_v4["fallback_used"]
)

print(
    "Retrieved chunks:",
    lost_v4["retrieved_chunks"]
)

print(
    "Sources:",
    lost_v4["source_ids"]
)

print(
    "Country contexts:",
    lost_v4["country_contexts"]
)

print(
    "Foreign numbers detected:",
    lost_v4[
        "foreign_numbers_detected"
    ]
)

print()

print("FINAL ANSWER")
print("-" * 70)

print(
    lost_v4["final_output"]
)

LOST NAVIGATION — LOCALIZED RAG V4
Pipeline success: True
Answer generated: True
Fallback used: False
Retrieved chunks: 3
Sources: ['EXP_011']
Country contexts: ['New Zealand']
Foreign numbers detected: []

FINAL ANSWER
----------------------------------------------------------------------
**FINAL ANSWER:**

If you are lost on a hiking trail in New Zealand, follow these practical steps:

1.  **Stay on the trail:** Do not wander off into unfamiliar terrain.
2.  **Check for cell service:** If you have a phone, check if you have enough battery life to survive. If you do not have cell service, you must rely on paper backups or offline maps.
3.  **Use offline maps:** Download a reliable map app that works offline and shows all backcountry areas.
4.  **Print a section of the map:** Open the map in a new window and select 'print map' to save your location.
5.  **Call for help:** If you cannot find your way out, call for help immediately.
6.  **Stay visible:** Make yourself and your group visi

## Deterministic Foreign-Context Filtering

In [81]:
FOREIGN_LOCALIZATION_MARKERS = {
    "new zealand": [
        "new zealand",
        "linz",
        "topo50",
        "doc visitor",
        "mountain safety council"
    ],

    "united kingdom": [
        "united kingdom",
        "uk ",
        "nhs",
        "met office",
        "mountain weather information service"
    ],

    "united states": [
        "united states",
        "911",
        "national weather service",
        "nps.gov"
    ]
}


def contains_foreign_specific_reference(
    sentence,
    country_context
):

    country = str(
        country_context
    ).strip().lower()

    markers = (
        FOREIGN_LOCALIZATION_MARKERS.get(
            country,
            []
        )
    )

    sentence_lower = (
        sentence.lower()
    )

    return any(
        marker in sentence_lower
        for marker in markers
    )


print(
    "Foreign-context marker rules ready."
)

Foreign-context marker rules ready.


In [82]:
def filter_foreign_evidence(
    content,
    country_context
):

    country = str(
        country_context
    ).strip()

    # Sri Lankan and international evidence
    # does not receive foreign-country filtering here.
    if (
        country.lower() == "sri lanka"
        or
        country.lower() == "international"
        or
        country.lower() == "not specified"
        or
        not country
    ):
        return content.strip()


    sentences = re.split(
        r"(?<=[.!?])\s+",
        content
    )


    retained = []


    for sentence in sentences:

        sentence = (
            sentence.strip()
        )

        if not sentence:
            continue


        # Remove foreign emergency-number sentences.
        contains_foreign_number = any(
            re.search(
                rf"\b{re.escape(number)}\b",
                sentence
            )
            for number
            in FOREIGN_EMERGENCY_NUMBERS
        )


        if contains_foreign_number:
            continue


        # Remove explicitly country-specific
        # operational information.
        if contains_foreign_specific_reference(
            sentence=sentence,
            country_context=country
        ):
            continue


        retained.append(
            sentence
        )


    return " ".join(
        retained
    ).strip()


print(
    "Foreign evidence filter ready."
)

Foreign evidence filter ready.


In [83]:
lost_filter_test = (
    retrieve_offline_knowledge_v2(
        question=(
            "What should I do if I am lost "
            "on a hiking trail?"
        ),
        category="lost_navigation",
        limit=4
    )
)


print("FOREIGN-EVIDENCE FILTER TEST")
print("=" * 70)


for _, row in (
    lost_filter_test.iterrows()
):

    filtered = filter_foreign_evidence(
        content=row["content"],
        country_context=row["country_context"]
    )

    print()
    print(
        "Chunk:",
        row["chunk_id"]
    )

    print(
        "Country:",
        row["country_context"]
    )

    print("-" * 70)

    print(filtered)

    print()

FOREIGN-EVIDENCE FILTER TEST

Chunk: CHK_01740
Country: New Zealand
----------------------------------------------------------------------
We recommend downloading maps and checking they work offline before your trip. If you’re relying on your phone for navigation, make sure you will have enough battery life and carry a paper backup in case your phone gets damaged or runs out of charge. Reduce your risk of getting lost Before you go on your trip, prepare for emergencies by: Learning navigation and survival skills. Researching your route and planning your trip thoroughly. Taking the right gear, including a distress beacon, emergency shelter, warm and waterproof clothing, survival kit, first aid kit, navigation tools, a torch and spare batteries, and spare food. Sharing your plans with a trusted person before you leave.


Chunk: CHK_01741
Country: New Zealand
----------------------------------------------------------------------
kit, first aid kit, navigation tools, a torch and spare bat

In [84]:
combined_filtered_lost = " ".join(
    filter_foreign_evidence(
        content=row["content"],
        country_context=row["country_context"]
    )
    for _, row
    in lost_filter_test.iterrows()
)


localization_terms = [
    "new zealand",
    "linz",
    "topo50",
    "doc visitor",
    "mountain safety council",
    "111"
]


print("FILTERED CONTEXT LOCALIZATION AUDIT")
print("=" * 70)


for term in localization_terms:

    found = (
        term.lower()
        in
        combined_filtered_lost.lower()
    )

    print(
        f"{term:<25} ->",
        found
    )


print()
print(
    "Filtered characters:",
    len(combined_filtered_lost)
)

FILTERED CONTEXT LOCALIZATION AUDIT
new zealand               -> False
linz                      -> False
topo50                    -> False
doc visitor               -> False
mountain safety council   -> False
111                       -> False

Filtered characters: 1430


## Final Candidate Offline RAG Pipeline

In [85]:
def prepare_final_rag_context(
    results
):

    context_parts = []
    source_ids = []

    for _, row in results.iterrows():

        source_id = row["source_id"]

        if source_id not in source_ids:
            source_ids.append(source_id)


        country = row["country_context"]

        if (
            pd.isna(country)
            or
            not str(country).strip()
        ):
            country = "Not specified"


        content = str(
            row["content"]
        ).strip()


        # Apply deterministic localization filtering
        # only to country-specific foreign evidence.
        if (
            str(country).strip().lower()
            not in {
                "sri lanka",
                "international",
                "not specified"
            }
        ):

            content = filter_foreign_evidence(
                content=content,
                country_context=country
            )


        if not content.strip():
            continue


        context_parts.append(
            f"""
Organization: {row['organization']}
Country context: {country}
Category: {row['categories']}
Evidence:
{content}
""".strip()
        )


    context = "\n\n---\n\n".join(
        context_parts
    )

    return context, source_ids


print(
    "Final localized context builder ready."
)

Final localized context builder ready.


In [86]:
def offline_rag_answer_final(
    question,
    category,
    limit=4
):

    results = retrieve_offline_knowledge_v2(
        question=question,
        category=category,
        limit=limit
    )


    if results.empty:

        fallback = (
            "The offline knowledge base does not contain "
            "enough information to answer this safely."
        )

        return {
            "success": False,
            "pipeline_success": True,
            "answer_generated": False,
            "fallback_used": True,
            "question": question,
            "category": category,
            "answer": fallback,
            "final_output": fallback,
            "source_ids": [],
            "retrieved_chunks": 0,
            "country_contexts": [],
            "foreign_numbers_detected": [],
            "generation_time": 0
        }


    context, source_ids = (
        prepare_final_rag_context(
            results
        )
    )


    if not context.strip():

        fallback = (
            "The offline knowledge base does not contain "
            "enough localized information to answer this safely."
        )

        return {
            "success": False,
            "pipeline_success": True,
            "answer_generated": False,
            "fallback_used": True,
            "question": question,
            "category": category,
            "answer": fallback,
            "final_output": (
                fallback
                + "\n\nSources: "
                + ", ".join(source_ids)
            ),
            "source_ids": source_ids,
            "retrieved_chunks": len(results),
            "country_contexts": (
                results["country_context"]
                .fillna("Not specified")
                .unique()
                .tolist()
            ),
            "foreign_numbers_detected": [],
            "generation_time": 0
        }


    prompt = build_localized_grounded_prompt(
        question=question,
        context=context
    )


    generation = generate_with_local_server(
        prompt=prompt,
        max_tokens=180,
        temperature=0.0,
        timeout_seconds=60
    )


    if not generation["success"]:

        return {
            "success": False,
            "pipeline_success": False,
            "answer_generated": False,
            "fallback_used": False,
            "question": question,
            "category": category,
            "answer": "Local generation failed.",
            "final_output": "Local generation failed.",
            "source_ids": source_ids,
            "retrieved_chunks": len(results),
            "country_contexts": (
                results["country_context"]
                .fillna("Not specified")
                .unique()
                .tolist()
            ),
            "foreign_numbers_detected": [],
            "generation_time":
                generation["time_seconds"]
        }


    answer = re.sub(
        r"<think>.*?</think>",
        "",
        generation["content"],
        flags=(
            re.DOTALL
            |
            re.IGNORECASE
        )
    ).strip()


    fallback_used = (
        "INSUFFICIENT_OFFLINE_EVIDENCE"
        in answer
    )


    if fallback_used:

        answer = (
            "The offline knowledge base does not contain "
            "enough information to answer this safely."
        )


    guarded = apply_output_guardrails(
        answer=answer,
        retrieved_source_ids=source_ids,
        category=category
    )


    answer = guarded["answer"]


    answer_generated = (
        bool(answer.strip())
        and
        not fallback_used
    )


    final_output = (
        answer
        + "\n\nSources: "
        + ", ".join(source_ids)
    )


    return {
        "success": answer_generated,
        "pipeline_success": True,
        "answer_generated": answer_generated,
        "fallback_used": fallback_used,
        "question": question,
        "category": category,
        "answer": answer,
        "final_output": final_output,
        "source_ids": source_ids,
        "retrieved_chunks": len(results),
        "country_contexts": (
            results["country_context"]
            .fillna("Not specified")
            .unique()
            .tolist()
        ),
        "foreign_numbers_detected":
            guarded["foreign_numbers_found"],
        "generation_time":
            generation["time_seconds"],
        "retrieval_results":
            results
    }


print(
    "Final candidate offline RAG pipeline ready."
)

Final candidate offline RAG pipeline ready.


In [87]:
final_candidate_tests = [
    {
        "name": "lost_navigation",
        "question": (
            "What should I do if I am lost "
            "on a hiking trail?"
        ),
        "category": "lost_navigation"
    },

    {
        "name": "snakebite",
        "question": (
            "What should I do if a hiker "
            "is bitten by a snake?"
        ),
        "category": "snakebite"
    },

    {
        "name": "sri_lanka_emergency",
        "question": (
            "What ambulance service can I contact "
            "for a medical emergency in Sri Lanka?"
        ),
        "category": "sri_lanka_emergency"
    }
]


final_candidate_results = {}


for test in final_candidate_tests:

    result = offline_rag_answer_final(
        question=test["question"],
        category=test["category"]
    )

    final_candidate_results[
        test["name"]
    ] = result


    print()
    print("=" * 75)
    print(
        test["name"].upper()
    )
    print("=" * 75)

    print(
        "Pipeline success:",
        result["pipeline_success"]
    )

    print(
        "Answer generated:",
        result["answer_generated"]
    )

    print(
        "Fallback used:",
        result["fallback_used"]
    )

    print(
        "Retrieved chunks:",
        result["retrieved_chunks"]
    )

    print(
        "Sources:",
        result["source_ids"]
    )

    print(
        "Country contexts:",
        result["country_contexts"]
    )

    print(
        "Foreign numbers detected:",
        result[
            "foreign_numbers_detected"
        ]
    )

    print(
        "Generation time:",
        f"{result['generation_time']:.2f} seconds"
    )

    print()

    print(
        result["final_output"]
    )


LOST_NAVIGATION
Pipeline success: True
Answer generated: True
Fallback used: False
Retrieved chunks: 3
Sources: ['EXP_011']
Country contexts: ['New Zealand']
Foreign numbers detected: []
Generation time: 3.30 seconds

**FINAL ANSWER:**

If you are lost on a hiking trail in New Zealand, follow these practical steps:

1.  **Stay Visible and Secure:** Stop and assess the situation. Unless you are certain of the way out, stay where you are. Make yourself and your group safe and comfortable – get warm, eat and drink, make a shelter, use first aid if needed. Call for help and make yourself visible.
2.  **Use Offline Tools:** Download maps and check they work offline before your trip. If you rely on your phone for navigation, ensure you have enough battery life and carry a paper backup.
3.  **Plan Your Route:** Research your route and plan your trip thoroughly.
4.  **Prepare Gear:** Carry a distress beacon, emergency shelter, warm and waterproof clothing, survival kit, first aid kit, navigat

## Separation of Localization Metadata from Generation Context

In [88]:
def prepare_final_rag_context(
    results
):

    context_parts = []
    source_ids = []

    for _, row in results.iterrows():

        source_id = row["source_id"]

        if source_id not in source_ids:
            source_ids.append(source_id)


        country = row["country_context"]

        if (
            pd.isna(country)
            or
            not str(country).strip()
        ):
            country = "Not specified"


        content = str(
            row["content"]
        ).strip()


        # Country metadata is used internally to decide
        # whether deterministic localization filtering
        # is required.
        if (
            str(country).strip().lower()
            not in {
                "sri lanka",
                "international",
                "not specified"
            }
        ):

            content = filter_foreign_evidence(
                content=content,
                country_context=country
            )


        if not content.strip():
            continue


        # Do NOT expose foreign country metadata to
        # the language model after filtering.
        context_parts.append(
            f"""
Category: {row['categories']}
Evidence:
{content}
""".strip()
        )


    context = "\n\n---\n\n".join(
        context_parts
    )

    return context, source_ids


print(
    "Generation context now separated from localization metadata."
)

Generation context now separated from localization metadata.


In [89]:
lost_final_retest = (
    offline_rag_answer_final(
        question=(
            "What should I do if I am lost "
            "on a hiking trail?"
        ),
        category="lost_navigation"
    )
)


print("LOST NAVIGATION — FINAL LOCALIZATION RETEST")
print("=" * 75)

print(
    "Pipeline success:",
    lost_final_retest[
        "pipeline_success"
    ]
)

print(
    "Answer generated:",
    lost_final_retest[
        "answer_generated"
    ]
)

print(
    "Fallback used:",
    lost_final_retest[
        "fallback_used"
    ]
)

print(
    "Sources:",
    lost_final_retest[
        "source_ids"
    ]
)

print(
    "Country metadata retained internally:",
    lost_final_retest[
        "country_contexts"
    ]
)

print(
    "Foreign numbers detected:",
    lost_final_retest[
        "foreign_numbers_detected"
    ]
)

print()

print("FINAL ANSWER")
print("-" * 75)

print(
    lost_final_retest[
        "final_output"
    ]
)

LOST NAVIGATION — FINAL LOCALIZATION RETEST
Pipeline success: True
Answer generated: True
Fallback used: False
Sources: ['EXP_011']
Country metadata retained internally: ['New Zealand']
Foreign numbers detected: []

FINAL ANSWER
---------------------------------------------------------------------------
**FINAL ANSWER:**

If you are lost on a hiking trail, follow these practical steps immediately:

1.  **Stay Visible and Assess:** Stop and assess the situation. Unless you are certain of the way out, stay where you are. Make yourself and your group safe and comfortable – get warm, eat and drink, make a shelter, use first aid if needed. Call for help and make yourself visible.
2.  **Use Offline Tools:** Download maps and check they work offline before your trip. If you rely on your phone for navigation, ensure you have enough battery life and carry a paper backup.
3.  **Plan Your Route:** Research your route and plan your trip thoroughly.
4.  **Prepare Gear:** Carry a distress beacon, em

In [90]:
lost_answer_lower = (
    lost_final_retest[
        "answer"
    ].lower()
)


forbidden_lost_terms = [
    "new zealand",
    "111",
    "linz",
    "topo50",
    "doc visitor",
    "mountain safety council"
]


print("LOST-NAVIGATION LOCALIZATION AUDIT")
print("=" * 75)


localization_pass = True


for term in forbidden_lost_terms:

    found = (
        term in lost_answer_lower
    )

    print(
        f"{term:<25} ->",
        found
    )

    if found:
        localization_pass = False


print()
print(
    "LOCALIZATION PASS:",
    localization_pass
)

LOST-NAVIGATION LOCALIZATION AUDIT
new zealand               -> False
111                       -> False
linz                      -> False
topo50                    -> False
doc visitor               -> False
mountain safety council   -> False

LOCALIZATION PASS: True


## Sri Lankan Evidence Validation

In [91]:
snakebite_final = (
    offline_rag_answer_final(
        question=(
            "What should I do if a hiker "
            "is bitten by a snake?"
        ),
        category="snakebite"
    )
)


print("SNAKEBITE — SRI LANKA EVIDENCE TEST")
print("=" * 75)

print(
    "Pipeline success:",
    snakebite_final[
        "pipeline_success"
    ]
)

print(
    "Answer generated:",
    snakebite_final[
        "answer_generated"
    ]
)

print(
    "Fallback used:",
    snakebite_final[
        "fallback_used"
    ]
)

print(
    "Retrieved chunks:",
    snakebite_final[
        "retrieved_chunks"
    ]
)

print(
    "Sources:",
    snakebite_final[
        "source_ids"
    ]
)

print(
    "Country contexts:",
    snakebite_final[
        "country_contexts"
    ]
)

print(
    "Foreign numbers detected:",
    snakebite_final[
        "foreign_numbers_detected"
    ]
)

print()

print("FINAL ANSWER")
print("-" * 75)

print(
    snakebite_final[
        "final_output"
    ]
)

SNAKEBITE — SRI LANKA EVIDENCE TEST
Pipeline success: True
Answer generated: True
Fallback used: False
Retrieved chunks: 4
Sources: ['EXP_001', 'EXP_002']
Country contexts: ['Sri Lanka', 'International']
Foreign numbers detected: []

FINAL ANSWER
---------------------------------------------------------------------------
**Immediate Action:**
1.  **Remove the victim from the area immediately.**
2.  **Remove anything tight from around the bitten part of the body to avoid harm if swelling occurs.**
3.  **Reassure the victim.**
4.  **Apply pressure at the bite site with a pressure pad if available.**
5.  **Avoid traditional first aid methods or herbal medicines.**
6.  **Give Paracetamol for local pain.**
7.  **Place the person on their left side in the recovery position.**
8.  **Monitor airway and breathing and be ready to resuscitate if necessary.**
9.  **Do not move the victim.**
10. **Do not give any food or drink.**
11. **Do not give any other medication.**
12. **Do

Sources: EXP_001,

In [92]:
emergency_final = (
    offline_rag_answer_final(
        question=(
            "What ambulance service can I contact "
            "for a medical emergency in Sri Lanka?"
        ),
        category="sri_lanka_emergency"
    )
)


print("SRI LANKA EMERGENCY ASSISTANCE TEST")
print("=" * 75)

print(
    "Pipeline success:",
    emergency_final[
        "pipeline_success"
    ]
)

print(
    "Answer generated:",
    emergency_final[
        "answer_generated"
    ]
)

print(
    "Fallback used:",
    emergency_final[
        "fallback_used"
    ]
)

print(
    "Retrieved chunks:",
    emergency_final[
        "retrieved_chunks"
    ]
)

print(
    "Sources:",
    emergency_final[
        "source_ids"
    ]
)

print(
    "Country contexts:",
    emergency_final[
        "country_contexts"
    ]
)

print(
    "Foreign numbers detected:",
    emergency_final[
        "foreign_numbers_detected"
    ]
)

print()

print("FINAL ANSWER")
print("-" * 75)

print(
    emergency_final[
        "final_output"
    ]
)

SRI LANKA EMERGENCY ASSISTANCE TEST
Pipeline success: True
Answer generated: True
Fallback used: False
Retrieved chunks: 4
Sources: ['EXP_015']
Country contexts: ['Sri Lanka']
Foreign numbers detected: []

FINAL ANSWER
---------------------------------------------------------------------------
SAFETY AND LOCALIZATION RULES:

1. Use only claims supported by the retrieved context.

2. Give practical instructions that directly answer the question.

3. Sri Lankan evidence may be used as local operational guidance.

4. Evidence from another country may be used only for generally transferable
   hiking or safety principles.

5. Do not present foreign country-specific services, emergency numbers,
   organizations, websites, map products, or geographic instructions as
   recommendations for Sri Lanka.

6. Do not convert or replace foreign information with an invented Sri Lankan
   equivalent.

7. Do not invent emergency telephone numbers.

8. Do not mention source IDs or discuss the retrieved 

In [93]:
localization_validation = pd.DataFrame(
    [
        {
            "test": "lost_navigation",
            "pipeline_success":
                lost_final_retest[
                    "pipeline_success"
                ],
            "answer_generated":
                lost_final_retest[
                    "answer_generated"
                ],
            "fallback_used":
                lost_final_retest[
                    "fallback_used"
                ],
            "sources":
                ", ".join(
                    lost_final_retest[
                        "source_ids"
                    ]
                ),
            "country_context":
                ", ".join(
                    lost_final_retest[
                        "country_contexts"
                    ]
                ),
            "localization_pass":
                localization_pass
        },

        {
            "test": "snakebite",
            "pipeline_success":
                snakebite_final[
                    "pipeline_success"
                ],
            "answer_generated":
                snakebite_final[
                    "answer_generated"
                ],
            "fallback_used":
                snakebite_final[
                    "fallback_used"
                ],
            "sources":
                ", ".join(
                    snakebite_final[
                        "source_ids"
                    ]
                ),
            "country_context":
                ", ".join(
                    snakebite_final[
                        "country_contexts"
                    ]
                ),
            "localization_pass":
                (
                    "EXP_001"
                    in
                    snakebite_final[
                        "source_ids"
                    ]
                )
        },

        {
            "test":
                "sri_lanka_emergency",

            "pipeline_success":
                emergency_final[
                    "pipeline_success"
                ],

            "answer_generated":
                emergency_final[
                    "answer_generated"
                ],

            "fallback_used":
                emergency_final[
                    "fallback_used"
                ],

            "sources":
                ", ".join(
                    emergency_final[
                        "source_ids"
                    ]
                ),

            "country_context":
                ", ".join(
                    emergency_final[
                        "country_contexts"
                    ]
                ),

            "localization_pass":
                (
                    "EXP_015"
                    in
                    emergency_final[
                        "source_ids"
                    ]
                    and
                    re.search(
                        r"\b1990\b",
                        emergency_final[
                            "answer"
                        ]
                    )
                    is not None
                )
        }
    ]
)


print("LOCALIZATION VALIDATION SUMMARY")
print("=" * 75)

display(
    localization_validation
)


print()

print(
    "All pipelines executed:",
    localization_validation[
        "pipeline_success"
    ].all()
)

print(
    "All produced answers:",
    localization_validation[
        "answer_generated"
    ].all()
)

print(
    "All localization checks passed:",
    localization_validation[
        "localization_pass"
    ].all()
)

LOCALIZATION VALIDATION SUMMARY


,test,pipeline_success,answer_generated,fallback_used,sources,country_context,localization_pass
0,lost_navigation,True,True,False,EXP_011,New Zealand,True
1,snakebite,True,True,False,"EXP_001, EXP_002","Sri Lanka, International",True
2,sri_lanka_emergency,True,True,False,EXP_015,Sri Lanka,False



All pipelines executed: True
All produced answers: True
All localization checks passed: False


## Prompt-Echo Detection

In [94]:
PROMPT_ECHO_MARKERS = [
    "safety and localization rules",
    "use only claims supported by",
    "give practical instructions that directly answer",
    "evidence from another country",
    "do not present foreign",
    "do not invent emergency telephone numbers",
    "authoritative context"
]


def detect_prompt_echo(answer):

    answer_lower = answer.lower()

    matched_markers = [
        marker
        for marker in PROMPT_ECHO_MARKERS
        if marker in answer_lower
    ]

    return {
        "prompt_echo_detected":
            len(matched_markers) >= 2,

        "matched_markers":
            matched_markers
    }


emergency_echo_audit = detect_prompt_echo(
    emergency_final["answer"]
)


print("SRI LANKA EMERGENCY PROMPT-ECHO AUDIT")
print("=" * 75)

print(
    "Prompt echo detected:",
    emergency_echo_audit[
        "prompt_echo_detected"
    ]
)

print(
    "Matched markers:"
)

for marker in (
    emergency_echo_audit[
        "matched_markers"
    ]
):
    print(
        " -",
        marker
    )

SRI LANKA EMERGENCY PROMPT-ECHO AUDIT
Prompt echo detected: True
Matched markers:
 - safety and localization rules
 - use only claims supported by
 - give practical instructions that directly answer
 - evidence from another country
 - do not present foreign
 - do not invent emergency telephone numbers


In [95]:
emergency_evidence = (
    emergency_final[
        "retrieval_results"
    ]
)


print("EXP_015 RETRIEVED EVIDENCE AUDIT")
print("=" * 75)


for _, row in emergency_evidence.iterrows():

    print()
    print(
        "Chunk:",
        row["chunk_id"]
    )

    print(
        "Source:",
        row["source_id"]
    )

    print(
        "Contains 1990:",
        bool(
            re.search(
                r"\b1990\b",
                str(row["content"])
            )
        )
    )

    print()

    print(
        row["content"]
    )

    print()
    print("-" * 75)

EXP_015 RETRIEVED EVIDENCE AUDIT

Chunk: CHK_01776
Source: EXP_015
Contains 1990: True

Home About Us Our Procedure Our Team FAQ News & Blog Gallery Donations Contact Us Contact Us Contact Us FAQ Do you provide an ambulance to cover a private event for a fee? Yes we do, if our backup ambulance for the province is available. We do not provide this service if adequate ambulances are unavailable to cover any event. Do you protect the privacy of the callers and patients? Yes, we do, unless we are required by law to release the data for a special investigation. How is 1990 Suwa Seriya set up? By an Act of parliament -Act no 18 of 2018 If I need to go to the hospital for a clinic, Can I use your service? No. 1990 is only a pre hospital care emergency ambulance service. Can I use your service to take a patient home? No. 1990 is only a pre hospital care emergency ambulance service. Will you transfer a dead person to the hospital? If the patient had expired before our arrival we will not. Can I

## Compact Prompt for Small Local Model

In [96]:
def build_compact_grounded_prompt(
    question,
    context
):

    return f"""
Answer the hiking-safety question using only the evidence below.

Give only the final practical answer.
Do not discuss these instructions or the evidence.
Do not invent information.
If the evidence does not help answer the question, output:
INSUFFICIENT_OFFLINE_EVIDENCE

Evidence:
{context}

Question:
{question}

Answer:
""".strip()


print(
    "Compact grounded prompt ready."
)

Compact grounded prompt ready.


In [97]:
emergency_context, emergency_sources = (
    prepare_final_rag_context(
        emergency_final[
            "retrieval_results"
        ]
    )
)


emergency_compact_prompt = (
    build_compact_grounded_prompt(
        question=(
            "What ambulance service can I contact "
            "for a medical emergency in Sri Lanka?"
        ),
        context=emergency_context
    )
)


emergency_compact_generation = (
    generate_with_local_server(
        prompt=emergency_compact_prompt,
        max_tokens=100,
        temperature=0.0,
        timeout_seconds=60
    )
)


compact_answer = re.sub(
    r"<think>.*?</think>",
    "",
    emergency_compact_generation[
        "content"
    ],
    flags=(
        re.DOTALL
        |
        re.IGNORECASE
    )
).strip()


echo_check = detect_prompt_echo(
    compact_answer
)


print("COMPACT-PROMPT EMERGENCY TEST")
print("=" * 75)

print(
    "Generation success:",
    emergency_compact_generation[
        "success"
    ]
)

print(
    "Prompt echo:",
    echo_check[
        "prompt_echo_detected"
    ]
)

print(
    "Contains verified 1990:",
    bool(
        re.search(
            r"\b1990\b",
            compact_answer
        )
    )
)

print(
    "Sources:",
    emergency_sources
)

print()

print("ANSWER")
print("-" * 75)

print(
    compact_answer
)

print()
print(
    "Sources:",
    ", ".join(
        emergency_sources
    )
)

COMPACT-PROMPT EMERGENCY TEST
Generation success: True
Prompt echo: False
Contains verified 1990: False
Sources: ['EXP_015']

ANSWER
---------------------------------------------------------------------------
INSUFFICIENT_OFFLINE_EVIDENCE

Sources: EXP_015


## Deterministic Handling of Verified Critical Facts

In [98]:
def verify_sri_lanka_1990_evidence(
    results
):

    if results.empty:
        return False

    for _, row in results.iterrows():

        source_id = str(
            row["source_id"]
        ).strip()

        country = str(
            row["country_context"]
        ).strip().lower()

        content = str(
            row["content"]
        ).lower()

        has_number = (
            re.search(
                r"\b1990\b",
                content
            )
            is not None
        )

        has_ambulance_context = (
            "ambulance" in content
            or
            "pre hospital" in content
            or
            "pre-hospital" in content
        )

        if (
            source_id == "EXP_015"
            and
            country == "sri lanka"
            and
            has_number
            and
            has_ambulance_context
        ):
            return True

    return False


print(
    "Sri Lanka 1990 evidence verifier ready."
)

Sri Lanka 1990 evidence verifier ready.


In [99]:
verified_1990 = (
    verify_sri_lanka_1990_evidence(
        emergency_final[
            "retrieval_results"
        ]
    )
)


print("VERIFIED CRITICAL FACT TEST")
print("=" * 75)

print(
    "EXP_015 retrieved:",
    (
        "EXP_015"
        in emergency_final[
            "source_ids"
        ]
    )
)

print(
    "1990 supported by retrieved "
    "Sri Lankan ambulance evidence:",
    verified_1990
)

VERIFIED CRITICAL FACT TEST
EXP_015 retrieved: True
1990 supported by retrieved Sri Lankan ambulance evidence: True


In [100]:
def build_verified_emergency_answer(
    results
):

    if not verify_sri_lanka_1990_evidence(
        results
    ):
        return None

    return (
        "For a medical emergency in Sri Lanka, "
        "you can request the 1990 Suwa Seriya "
        "pre-hospital emergency ambulance service "
        "by dialing 1990. The retrieved offline "
        "Sri Lankan source states that the hotline "
        "is toll-free and the service covers all "
        "districts of the country."
    )


verified_emergency_answer = (
    build_verified_emergency_answer(
        emergency_final[
            "retrieval_results"
        ]
    )
)


print("VERIFIED EMERGENCY ANSWER")
print("=" * 75)

print(
    verified_emergency_answer
)

print()
print(
    "Sources: EXP_015"
)

VERIFIED EMERGENCY ANSWER
For a medical emergency in Sri Lanka, you can request the 1990 Suwa Seriya pre-hospital emergency ambulance service by dialing 1990. The retrieved offline Sri Lankan source states that the hotline is toll-free and the service covers all districts of the country.

Sources: EXP_015


## Hybrid Offline RAG Answer Router

In [101]:
def offline_hybrid_rag_answer(
    question,
    category,
    limit=4
):

    # -----------------------------------------
    # 1. Retrieve authoritative offline evidence
    # -----------------------------------------

    results = retrieve_offline_knowledge_v2(
        question=question,
        category=category,
        limit=limit
    )


    if results.empty:

        fallback = (
            "The offline knowledge base does not contain "
            "enough information to answer this safely."
        )

        return {
            "success": False,
            "answer": fallback,
            "final_output": fallback,
            "answer_mode": "fallback",
            "source_ids": [],
            "retrieved_chunks": 0
        }


    source_ids = (
        results["source_id"]
        .drop_duplicates()
        .tolist()
    )


    # -----------------------------------------
    # 2. Verified critical-fact route
    # -----------------------------------------

    if category == "sri_lanka_emergency":

        deterministic_answer = (
            build_verified_emergency_answer(
                results
            )
        )

        if deterministic_answer:

            return {
                "success": True,
                "answer":
                    deterministic_answer,

                "final_output":
                    (
                        deterministic_answer
                        + "\n\nSources: "
                        + ", ".join(
                            source_ids
                        )
                    ),

                "answer_mode":
                    "verified_deterministic",

                "source_ids":
                    source_ids,

                "retrieved_chunks":
                    len(results)
            }


    # -----------------------------------------
    # 3. General localized RAG route
    # -----------------------------------------

    rag_result = (
        offline_rag_answer_final(
            question=question,
            category=category,
            limit=limit
        )
    )


    rag_result[
        "answer_mode"
    ] = "local_llm_rag"


    return rag_result


print(
    "Hybrid offline RAG router ready."
)

Hybrid offline RAG router ready.


In [102]:
hybrid_emergency_test = (
    offline_hybrid_rag_answer(
        question=(
            "What ambulance service can I contact "
            "for a medical emergency in Sri Lanka?"
        ),
        category="sri_lanka_emergency"
    )
)


print("HYBRID SRI LANKA EMERGENCY TEST")
print("=" * 75)

print(
    "Success:",
    hybrid_emergency_test[
        "success"
    ]
)

print(
    "Answer mode:",
    hybrid_emergency_test[
        "answer_mode"
    ]
)

print(
    "Retrieved chunks:",
    hybrid_emergency_test[
        "retrieved_chunks"
    ]
)

print(
    "Sources:",
    hybrid_emergency_test[
        "source_ids"
    ]
)

print()

print("FINAL ANSWER")
print("-" * 75)

print(
    hybrid_emergency_test[
        "final_output"
    ]
)

HYBRID SRI LANKA EMERGENCY TEST
Success: True
Answer mode: verified_deterministic
Retrieved chunks: 4
Sources: ['EXP_015']

FINAL ANSWER
---------------------------------------------------------------------------
For a medical emergency in Sri Lanka, you can request the 1990 Suwa Seriya pre-hospital emergency ambulance service by dialing 1990. The retrieved offline Sri Lankan source states that the hotline is toll-free and the service covers all districts of the country.

Sources: EXP_015


In [103]:
hybrid_lost_test = (
    offline_hybrid_rag_answer(
        question=(
            "What should I do if I am lost "
            "on a hiking trail?"
        ),
        category="lost_navigation"
    )
)


print("HYBRID ROUTING CHECK")
print("=" * 75)

print(
    "Emergency answer mode:",
    hybrid_emergency_test[
        "answer_mode"
    ]
)

print(
    "Lost-navigation answer mode:",
    hybrid_lost_test[
        "answer_mode"
    ]
)

print()

print(
    "Emergency uses verified deterministic route:",
    (
        hybrid_emergency_test[
            "answer_mode"
        ]
        ==
        "verified_deterministic"
    )
)

print(
    "Lost navigation uses local LLM RAG:",
    (
        hybrid_lost_test[
            "answer_mode"
        ]
        ==
        "local_llm_rag"
    )
)

HYBRID ROUTING CHECK
Emergency answer mode: verified_deterministic
Lost-navigation answer mode: local_llm_rag

Emergency uses verified deterministic route: True
Lost navigation uses local LLM RAG: True


# Offline RAG System Evaluation

In [104]:
rag_evaluation_cases = [
    {
        "category": "general_first_aid",
        "question": (
            "What basic first aid steps should I follow "
            "when a hiker is injured?"
        )
    },
    {
        "category": "hypothermia",
        "question": (
            "What should I do if a hiker shows signs "
            "of hypothermia?"
        )
    },
    {
        "category": "heat_illness",
        "question": (
            "What should I do if a hiker develops "
            "heat illness?"
        )
    },
    {
        "category": "heavy_rain",
        "question": (
            "What safety precautions should hikers take "
            "during heavy rain?"
        )
    },
    {
        "category": "strong_wind",
        "question": (
            "What should hikers do during strong winds "
            "in mountainous terrain?"
        )
    },
    {
        "category": "water_safety",
        "question": (
            "How should hikers stay safe around rivers "
            "and dangerous water crossings?"
        )
    },
    {
        "category": "remote_emergency",
        "question": (
            "What should I do during an emergency "
            "in a remote hiking area?"
        )
    },
    {
        "category": "snakebite",
        "question": (
            "What should I do if a hiker "
            "is bitten by a snake?"
        )
    },
    {
        "category": "insect_bites_stings",
        "question": (
            "What should I do for insect bites "
            "or stings while hiking?"
        )
    },
    {
        "category": "fracture_sprain",
        "question": (
            "What should I do if a hiker has "
            "a suspected fracture or sprain?"
        )
    },
    {
        "category": "burns",
        "question": (
            "What first aid should be given "
            "for a burn while hiking?"
        )
    },
    {
        "category": "poisoning",
        "question": (
            "What should I do if poisoning is suspected "
            "during a hiking trip?"
        )
    },
    {
        "category": "allergic_reaction",
        "question": (
            "What should I do if a hiker develops "
            "a serious allergic reaction?"
        )
    },
    {
        "category": "animal_encounter",
        "question": (
            "What should hikers do during "
            "a dangerous animal encounter?"
        )
    },
    {
        "category": "lost_navigation",
        "question": (
            "What should I do if I am lost "
            "on a hiking trail?"
        )
    },
    {
        "category": "landslide",
        "question": (
            "What should hikers do if there is "
            "a landslide risk?"
        )
    },
    {
        "category": "lightning",
        "question": (
            "What should hikers do during "
            "a lightning storm in the mountains?"
        )
    },
    {
        "category": "sri_lanka_emergency",
        "question": (
            "What ambulance service can I contact "
            "for a medical emergency in Sri Lanka?"
        )
    }
]


print(
    "Evaluation cases:",
    len(rag_evaluation_cases)
)


for i, case in enumerate(
    rag_evaluation_cases,
    start=1
):
    print(
        f"{i:02d}. "
        f"{case['category']:<24} "
        f"{case['question']}"
    )

Evaluation cases: 18
01. general_first_aid        What basic first aid steps should I follow when a hiker is injured?
02. hypothermia              What should I do if a hiker shows signs of hypothermia?
03. heat_illness             What should I do if a hiker develops heat illness?
04. heavy_rain               What safety precautions should hikers take during heavy rain?
05. strong_wind              What should hikers do during strong winds in mountainous terrain?
06. water_safety             How should hikers stay safe around rivers and dangerous water crossings?
07. remote_emergency         What should I do during an emergency in a remote hiking area?
08. snakebite                What should I do if a hiker is bitten by a snake?
09. insect_bites_stings      What should I do for insect bites or stings while hiking?
10. fracture_sprain          What should I do if a hiker has a suspected fracture or sprain?
11. burns                    What first aid should be given for a burn while hi

In [105]:
retrieval_audit_rows = []


for case in rag_evaluation_cases:

    category = case["category"]
    question = case["question"]


    results = retrieve_offline_knowledge_v2(
        question=question,
        category=category,
        limit=4
    )


    retrieval_success = (
        not results.empty
    )


    if retrieval_success:

        category_consistent = (
            results["categories"]
            .fillna("")
            .apply(
                lambda value:
                    category
                    in [
                        item.strip()
                        for item
                        in str(value).split(",")
                    ]
            )
            .all()
        )


        source_ids = (
            results["source_id"]
            .drop_duplicates()
            .tolist()
        )


        countries = (
            results["country_context"]
            .fillna("Not specified")
            .drop_duplicates()
            .tolist()
        )

    else:

        category_consistent = False
        source_ids = []
        countries = []


    retrieval_audit_rows.append(
        {
            "category":
                category,

            "retrieval_success":
                retrieval_success,

            "category_consistent":
                category_consistent,

            "retrieved_chunks":
                len(results),

            "unique_sources":
                len(source_ids),

            "source_ids":
                ", ".join(source_ids),

            "country_contexts":
                ", ".join(countries)
        }
    )


retrieval_audit_df = pd.DataFrame(
    retrieval_audit_rows
)


display(
    retrieval_audit_df
)

,category,retrieval_success,category_consistent,retrieved_chunks,unique_sources,source_ids,country_contexts
0,general_first_aid,True,True,3,1,SRC_016,Not specified
1,hypothermia,True,True,3,1,SRC_014,Not specified
2,heat_illness,True,True,4,1,SRC_015,Not specified
3,heavy_rain,True,True,2,1,SRC_006,Not specified
4,strong_wind,True,True,4,1,SRC_007,Not specified
5,water_safety,True,True,4,1,SRC_017,Not specified
6,remote_emergency,True,True,4,2,"SRC_010, SRC_009",Not specified
7,snakebite,True,True,4,2,"EXP_001, EXP_002","Sri Lanka, International"
8,insect_bites_stings,True,True,4,1,EXP_016,United Kingdom (England)
9,fracture_sprain,True,True,4,2,"EXP_004, EXP_005",United Kingdom


In [106]:
total_cases = len(
    retrieval_audit_df
)


retrieval_success_count = int(
    retrieval_audit_df[
        "retrieval_success"
    ].sum()
)


category_consistent_count = int(
    retrieval_audit_df[
        "category_consistent"
    ].sum()
)


retrieval_success_rate = (
    retrieval_success_count
    /
    total_cases
)


category_consistency_rate = (
    category_consistent_count
    /
    total_cases
)


mean_chunks = (
    retrieval_audit_df[
        "retrieved_chunks"
    ].mean()
)


mean_sources = (
    retrieval_audit_df[
        "unique_sources"
    ].mean()
)


print("OFFLINE RAG RETRIEVAL AUDIT")
print("=" * 75)

print(
    "Evaluation cases:",
    total_cases
)

print(
    "Successful retrievals:",
    f"{retrieval_success_count}/{total_cases}"
)

print(
    "Retrieval success rate:",
    f"{retrieval_success_rate:.2%}"
)

print(
    "Category-consistent cases:",
    f"{category_consistent_count}/{total_cases}"
)

print(
    "Category consistency rate:",
    f"{category_consistency_rate:.2%}"
)

print(
    "Mean retrieved chunks:",
    f"{mean_chunks:.2f}"
)

print(
    "Mean unique sources:",
    f"{mean_sources:.2f}"
)


failed_retrievals = (
    retrieval_audit_df[
        ~retrieval_audit_df[
            "retrieval_success"
        ]
    ]
)


if not failed_retrievals.empty:

    print()
    print("FAILED RETRIEVAL CATEGORIES")
    print("-" * 75)

    print(
        failed_retrievals[
            ["category"]
        ].to_string(
            index=False
        )
    )

OFFLINE RAG RETRIEVAL AUDIT
Evaluation cases: 18
Successful retrievals: 18/18
Retrieval success rate: 100.00%
Category-consistent cases: 18/18
Category consistency rate: 100.00%
Mean retrieved chunks: 3.67
Mean unique sources: 1.22


## Retrieval Failure Diagnosis

In [107]:
from pathlib import Path
import sqlite3
import pandas as pd
import re


# Notebook:
# research/notebooks/rag/02_offline_rag_generation.ipynb
#
# Database:
# research/rag/database/offline_hiking_knowledge.db

PROJECT_ROOT = Path.cwd()

# Handle running the notebook either from the project root
# or from research/notebooks/rag.
if (PROJECT_ROOT / "research").exists():
    RAG_DB_PATH = (
        PROJECT_ROOT
        / "research"
        / "rag"
        / "database"
        / "offline_hiking_knowledge.db"
    )
else:
    RAG_DB_PATH = (
        PROJECT_ROOT
        / ".."
        / ".."
        / "rag"
        / "database"
        / "offline_hiking_knowledge.db"
    ).resolve()


print("Database path:")
print(RAG_DB_PATH)

print()
print(
    "Database exists:",
    RAG_DB_PATH.exists()
)

if RAG_DB_PATH.exists():
    print(
        "Database size:",
        f"{RAG_DB_PATH.stat().st_size / (1024**2):.2f} MB"
    )

Database path:
/Users/mirshad/Desktop/Decentralized-Hiking-AI/research/rag/database/offline_hiking_knowledge.db

Database exists: True
Database size: 6.66 MB


In [108]:
failed_categories = [
    "insect_bites_stings",
    "allergic_reaction"
]


conn = sqlite3.connect(
    RAG_DB_PATH
)


for category in failed_categories:

    query = """
    SELECT
        COUNT(*) AS chunk_count,
        COUNT(DISTINCT kc.source_id) AS source_count
    FROM knowledge_chunks kc
    WHERE
        (
            ',' ||
            REPLACE(kc.categories, ' ', '') ||
            ','
        ) LIKE ?
    """

    result = pd.read_sql_query(
        query,
        conn,
        params=[
            f"%,{category},%"
        ]
    )

    print(category)
    print("-" * 60)

    print(
        "Chunks:",
        int(
            result.iloc[0][
                "chunk_count"
            ]
        )
    )

    print(
        "Sources:",
        int(
            result.iloc[0][
                "source_count"
            ]
        )
    )

    print()


conn.close()

insect_bites_stings
------------------------------------------------------------
Chunks: 9
Sources: 1

allergic_reaction
------------------------------------------------------------
Chunks: 7
Sources: 1



In [109]:
conn = sqlite3.connect(
    RAG_DB_PATH
)


for category in failed_categories:

    query = """
    SELECT DISTINCT
        d.source_id,
        d.title,
        d.organization,
        d.country_context,
        d.categories
    FROM documents d
    WHERE
        (
            ',' ||
            REPLACE(d.categories, ' ', '') ||
            ','
        ) LIKE ?
    ORDER BY d.source_id
    """


    docs = pd.read_sql_query(
        query,
        conn,
        params=[
            f"%,{category},%"
        ]
    )


    print()
    print("=" * 75)
    print(
        category.upper()
    )
    print("=" * 75)

    display(
        docs
    )


conn.close()


INSECT_BITES_STINGS


,source_id,title,organization,country_context,categories
0,EXP_016,Insect bites and stings,NHS (National Health Service),United Kingdom (England),insect_bites_stings



ALLERGIC_REACTION


,source_id,title,organization,country_context,categories
0,EXP_017,Anaphylaxis,NHS (National Health Service),United Kingdom (England),allergic_reaction


In [110]:
conn = sqlite3.connect(
    RAG_DB_PATH
)


failed_category_chunks = {}


for category in failed_categories:

    query = """
    SELECT
        kc.chunk_id,
        kc.source_id,
        kc.categories,
        kc.word_count,
        kc.content,
        d.title,
        d.organization,
        d.country_context
    FROM knowledge_chunks kc

    JOIN documents d
        ON d.source_id = kc.source_id

    WHERE
        (
            ',' ||
            REPLACE(kc.categories, ' ', '') ||
            ','
        ) LIKE ?

    ORDER BY
        kc.source_id,
        kc.source_chunk_index
    """


    chunks = pd.read_sql_query(
        query,
        conn,
        params=[
            f"%,{category},%"
        ]
    )


    failed_category_chunks[
        category
    ] = chunks


    print()
    print("=" * 75)
    print(
        category.upper()
    )
    print("=" * 75)

    print(
        "Available chunks:",
        len(chunks)
    )


    for _, row in chunks.head(8).iterrows():

        print()
        print(
            "Chunk:",
            row["chunk_id"]
        )

        print(
            "Source:",
            row["source_id"]
        )

        print(
            "Organization:",
            row["organization"]
        )

        print(
            "Country:",
            row["country_context"]
        )

        print("-" * 75)

        print(
            str(
                row["content"]
            )[:1500]
        )

        print()


conn.close()


INSECT_BITES_STINGS
Available chunks: 9

Chunk: CHK_01780
Source: EXP_016
Organization: NHS (National Health Service)
Country: United Kingdom (England)
---------------------------------------------------------------------------
Insect bites and stings Organization: NHS (National Health Service) Country context: United Kingdom (England; NHS website) Original source URL: https://www.nhs.uk/conditions/insect-bites-and-stings/ Retrieved: 2026-10-19 Representation: text extracted from the official webpage main content; wording retained. Linked pages and images are not bundled.


Chunk: CHK_01781
Source: EXP_016
Organization: NHS (National Health Service)
Country: United Kingdom (England)
---------------------------------------------------------------------------
Insect bites and stings Insect bites or stings are not usually serious and get better in a few days. But sometimes they can become infected or cause a serious allergic reaction. Bites from some insects can also cause illnesses, suc

In [112]:
diagnostic_terms = {

    "insect_bites_stings": [
        "insect",
        "bite",
        "bites",
        "sting",
        "stings",
        "bee",
        "wasp",
        "mosquito",
        "swelling"
    ],

    "allergic_reaction": [
        "allergic",
        "allergy",
        "reaction",
        "anaphylaxis",
        "anaphylactic",
        "epinephrine",
        "adrenaline",
        "swelling"
    ]
}


print(
    "FAILED-CATEGORY VOCABULARY AUDIT"
)

print(
    "=" * 75
)


for category in failed_categories:

    chunks = (
        failed_category_chunks[
            category
        ]
    )

    combined_text = " ".join(
        chunks["content"]
        .fillna("")
        .astype(str)
        .tolist()
    ).lower()


    print()
    print(
        category.upper()
    )

    print("-" * 75)


    for term in diagnostic_terms[
        category
    ]:

        count = len(
            re.findall(
                rf"\b{re.escape(term)}\b",
                combined_text
            )
        )

        print(
            f"{term:<20} {count}"
        )

FAILED-CATEGORY VOCABULARY AUDIT

INSECT_BITES_STINGS
---------------------------------------------------------------------------
insect               14
bite                 19
bites                20
sting                18
stings               6
bee                  3
wasp                 3
mosquito             4
swelling             1

ALLERGIC_REACTION
---------------------------------------------------------------------------
allergic             5
allergy              10
reaction             8
anaphylaxis          15
anaphylactic         5
epinephrine          0
adrenaline           15
swelling             1


## Missing-Category Source Ingestion


In [113]:
SOURCE_DIR = (
    PROJECT_ROOT
    / "research"
    / "rag"
    / "source_documents"
)

MISSING_CATALOG_PATH = (
    PROJECT_ROOT
    / "research"
    / "rag"
    / "missing_category_source_catalog.csv"
)


EXP_016_PATH = (
    SOURCE_DIR
    / "EXP_016_insect_bites_stings_nhs.txt"
)

EXP_017_PATH = (
    SOURCE_DIR
    / "EXP_017_anaphylaxis_nhs.txt"
)


required_files = [
    EXP_016_PATH,
    EXP_017_PATH,
    MISSING_CATALOG_PATH
]


print("LOCAL SOURCE VALIDATION")
print("=" * 75)


for path in required_files:

    print()
    print("File:", path.name)
    print("Exists:", path.exists())

    if path.exists():

        print(
            "Size:",
            path.stat().st_size,
            "bytes"
        )


all_files_exist = all(
    path.exists()
    and path.stat().st_size > 0
    for path in required_files
)


print()
print(
    "All required files valid:",
    all_files_exist
)

LOCAL SOURCE VALIDATION

File: EXP_016_insect_bites_stings_nhs.txt
Exists: False

File: EXP_017_anaphylaxis_nhs.txt
Exists: False

File: missing_category_source_catalog.csv
Exists: False

All required files valid: False


In [114]:
from pathlib import Path

print("CURRENT WORKING DIRECTORY")
print("=" * 75)
print(Path.cwd())

print()
print("PROJECT_ROOT")
print("=" * 75)
print(PROJECT_ROOT)

print()
print("SEARCHING FOR COPIED FILES")
print("=" * 75)

search_root = PROJECT_ROOT

filenames = [
    "EXP_016_insect_bites_stings_nhs.txt",
    "EXP_017_anaphylaxis_nhs.txt",
    "missing_category_source_catalog.csv"
]

for filename in filenames:

    matches = list(
        search_root.rglob(filename)
    )

    print()
    print(filename)

    if matches:
        for match in matches:
            print("FOUND:", match)
    else:
        print("NOT FOUND")

CURRENT WORKING DIRECTORY
/Users/mirshad/Desktop/Decentralized-Hiking-AI/research/notebooks/rag

PROJECT_ROOT
/Users/mirshad/Desktop/Decentralized-Hiking-AI/research/notebooks/rag

SEARCHING FOR COPIED FILES

EXP_016_insect_bites_stings_nhs.txt
NOT FOUND

EXP_017_anaphylaxis_nhs.txt
NOT FOUND

missing_category_source_catalog.csv
NOT FOUND


In [115]:
from pathlib import Path


def find_project_root(start_path):

    current = start_path.resolve()

    for candidate in [
        current,
        *current.parents
    ]:

        if (
            candidate
            / "research"
            / "rag"
        ).exists():

            return candidate

    raise FileNotFoundError(
        "Could not locate project root."
    )


PROJECT_ROOT = find_project_root(
    Path.cwd()
)


RAG_ROOT = (
    PROJECT_ROOT
    / "research"
    / "rag"
)


SOURCE_DIR = (
    RAG_ROOT
    / "source_documents"
)


RAG_DB_PATH = (
    RAG_ROOT
    / "database"
    / "offline_hiking_knowledge.db"
)


MISSING_CATALOG_PATH = (
    RAG_ROOT
    / "missing_category_source_catalog.csv"
)


EXP_016_PATH = (
    SOURCE_DIR
    / "EXP_016_insect_bites_stings_nhs.txt"
)


EXP_017_PATH = (
    SOURCE_DIR
    / "EXP_017_anaphylaxis_nhs.txt"
)


print("PATH CONFIGURATION")
print("=" * 75)

print(
    "Project root:",
    PROJECT_ROOT
)

print(
    "RAG root:",
    RAG_ROOT
)

print(
    "Source directory:",
    SOURCE_DIR
)

print(
    "Database:",
    RAG_DB_PATH
)

print()

print(
    "Database exists:",
    RAG_DB_PATH.exists()
)

print(
    "EXP_016 exists:",
    EXP_016_PATH.exists()
)

print(
    "EXP_017 exists:",
    EXP_017_PATH.exists()
)

print(
    "Catalog exists:",
    MISSING_CATALOG_PATH.exists()
)

PATH CONFIGURATION
Project root: /Users/mirshad/Desktop/Decentralized-Hiking-AI
RAG root: /Users/mirshad/Desktop/Decentralized-Hiking-AI/research/rag
Source directory: /Users/mirshad/Desktop/Decentralized-Hiking-AI/research/rag/source_documents
Database: /Users/mirshad/Desktop/Decentralized-Hiking-AI/research/rag/database/offline_hiking_knowledge.db

Database exists: True
EXP_016 exists: True
EXP_017 exists: True
Catalog exists: True


In [116]:
missing_catalog = pd.read_csv(
    MISSING_CATALOG_PATH
)


print("MISSING-CATEGORY SOURCE CATALOG")
print("=" * 75)

display(
    missing_catalog
)


print()
print(
    "Catalog rows:",
    len(missing_catalog)
)

print(
    "Source IDs:",
    missing_catalog[
        "source_id"
    ].tolist()
)

MISSING-CATEGORY SOURCE CATALOG


,source_id,category,title,organization,country_context,source_url,filename,file_type,file_size_bytes,download_status,foreign_emergency_reference,notes
0,EXP_016,insect_bites_stings,Insect bites and stings,NHS (National Health Service),United Kingdom (England),https://www.nhs.uk/conditions/insect-bites-and...,source_documents/EXP_016_insect_bites_stings_n...,text/plain; charset=utf-8,9058,downloaded_and_validated,UK 999; NHS 111 / 111 online; GP appointments;...,Verified official nhs.uk source; readable main...
1,EXP_017,allergic_reaction,Anaphylaxis,NHS (National Health Service),United Kingdom (England),https://www.nhs.uk/conditions/anaphylaxis/,source_documents/EXP_017_anaphylaxis_nhs.txt,text/plain; charset=utf-8,5880,downloaded_and_validated,UK 999 ambulance; UK hospital/prescription/ref...,Verified official nhs.uk source; readable main...



Catalog rows: 2
Source IDs: ['EXP_016', 'EXP_017']


In [117]:
source_validation = {
    "EXP_016": {
        "path": EXP_016_PATH,
        "required_terms": [
            "insect",
            "sting"
        ]
    },

    "EXP_017": {
        "path": EXP_017_PATH,
        "required_terms": [
            "anaphylaxis"
        ]
    }
}


validated_source_text = {}


print("SOURCE CONTENT VALIDATION")
print("=" * 75)


for source_id, config in (
    source_validation.items()
):

    text = config[
        "path"
    ].read_text(
        encoding="utf-8",
        errors="replace"
    )


    validated_source_text[
        source_id
    ] = text


    text_lower = text.lower()


    print()
    print(source_id)
    print("-" * 75)

    print(
        "Characters:",
        len(text)
    )

    print(
        "Words:",
        len(text.split())
    )


    for term in config[
        "required_terms"
    ]:

        print(
            f"Contains '{term}':",
            term in text_lower
        )


    print(
        "Contains UK 999:",
        bool(
            re.search(
                r"\b999\b",
                text
            )
        )
    )

    print(
        "Contains UK 111:",
        bool(
            re.search(
                r"\b111\b",
                text
            )
        )
    )

SOURCE CONTENT VALIDATION

EXP_016
---------------------------------------------------------------------------
Characters: 9054
Words: 1335
Contains 'insect': True
Contains 'sting': True
Contains UK 999: True
Contains UK 111: True

EXP_017
---------------------------------------------------------------------------
Characters: 5866
Words: 1002
Contains 'anaphylaxis': True
Contains UK 999: True
Contains UK 111: False


In [119]:
conn = sqlite3.connect(
    RAG_DB_PATH
)


baseline_documents = pd.read_sql_query(
    """
    SELECT COUNT(*) AS count
    FROM documents
    """,
    conn
).iloc[0]["count"]


baseline_chunks = pd.read_sql_query(
    """
    SELECT COUNT(*) AS count
    FROM knowledge_chunks
    """,
    conn
).iloc[0]["count"]


baseline_fts = pd.read_sql_query(
    """
    SELECT COUNT(*) AS count
    FROM knowledge_chunks_fts
    """,
    conn
).iloc[0]["count"]


existing_new_sources = pd.read_sql_query(
    """
    SELECT source_id
    FROM documents
    WHERE source_id IN ('EXP_016', 'EXP_017')
    """,
    conn
)


conn.close()


print("DATABASE BASELINE")
print("=" * 75)

print(
    "Documents:",
    baseline_documents
)

print(
    "Knowledge chunks:",
    baseline_chunks
)

print(
    "FTS5 rows:",
    baseline_fts
)

print()

print(
    "EXP_016 / EXP_017 already present:",
    len(existing_new_sources)
)


assert baseline_chunks == 1779, (
    "Expected the existing database to contain "
    "1779 chunks before expansion."
)

assert baseline_fts == 1779, (
    "Expected FTS5 to contain 1779 rows "
    "before expansion."
)

assert len(existing_new_sources) == 0, (
    "EXP_016 or EXP_017 already exists. "
    "Do not ingest duplicates."
)


print()
print(
    "Baseline integrity check: PASS"
)


DATABASE BASELINE
Documents: 27
Knowledge chunks: 1795
FTS5 rows: 1795

EXP_016 / EXP_017 already present: 2


AssertionError: Expected the existing database to contain 1779 chunks before expansion.

## Controlled Chunk Preparation

In [120]:
def create_text_chunks(
    text,
    target_words=180,
    max_words=240
):

    text = text.replace(
        "\r\n",
        "\n"
    ).replace(
        "\r",
        "\n"
    )

    paragraphs = [
        re.sub(
            r"\s+",
            " ",
            paragraph
        ).strip()

        for paragraph
        in re.split(
            r"\n\s*\n",
            text
        )

        if paragraph.strip()
    ]


    chunks = []
    current_parts = []
    current_words = 0


    for paragraph in paragraphs:

        paragraph_words = (
            paragraph.split()
        )


        # If one paragraph itself is too large,
        # divide it into bounded word groups.
        if len(paragraph_words) > max_words:

            if current_parts:

                chunks.append(
                    " ".join(
                        current_parts
                    )
                )

                current_parts = []
                current_words = 0


            for start in range(
                0,
                len(paragraph_words),
                target_words
            ):

                part = " ".join(
                    paragraph_words[
                        start:
                        start + target_words
                    ]
                )

                if part.strip():
                    chunks.append(part)

            continue


        if (
            current_words
            + len(paragraph_words)
            > max_words
            and
            current_parts
        ):

            chunks.append(
                " ".join(
                    current_parts
                )
            )

            current_parts = []
            current_words = 0


        current_parts.append(
            paragraph
        )

        current_words += (
            len(paragraph_words)
        )


        if current_words >= target_words:

            chunks.append(
                " ".join(
                    current_parts
                )
            )

            current_parts = []
            current_words = 0


    if current_parts:

        chunks.append(
            " ".join(
                current_parts
            )
        )


    return [
        chunk.strip()
        for chunk in chunks
        if chunk.strip()
    ]


print(
    "Chunking function ready."
)

Chunking function ready.


In [121]:
new_source_definitions = {
    "EXP_016": {
        "category":
            "insect_bites_stings",

        "path":
            EXP_016_PATH
    },

    "EXP_017": {
        "category":
            "allergic_reaction",

        "path":
            EXP_017_PATH
    }
}


prepared_new_chunks = {}


for source_id, info in (
    new_source_definitions.items()
):

    text = info[
        "path"
    ].read_text(
        encoding="utf-8",
        errors="replace"
    )


    chunks = create_text_chunks(
        text=text,
        target_words=180,
        max_words=240
    )


    prepared_new_chunks[
        source_id
    ] = chunks


    print()
    print(source_id)
    print("=" * 75)

    print(
        "Category:",
        info["category"]
    )

    print(
        "Chunks:",
        len(chunks)
    )

    print(
        "Total words:",
        sum(
            len(chunk.split())
            for chunk in chunks
        )
    )

    print(
        "Minimum chunk words:",
        min(
            len(chunk.split())
            for chunk in chunks
        )
    )

    print(
        "Maximum chunk words:",
        max(
            len(chunk.split())
            for chunk in chunks
        )
    )


EXP_016
Category: insect_bites_stings
Chunks: 9
Total words: 1335
Minimum chunk words: 35
Maximum chunk words: 180

EXP_017
Category: allergic_reaction
Chunks: 7
Total words: 1002
Minimum chunk words: 37
Maximum chunk words: 180


In [122]:
for source_id, chunks in (
    prepared_new_chunks.items()
):

    category = (
        new_source_definitions[
            source_id
        ]["category"]
    )


    print()
    print("=" * 75)

    print(
        source_id,
        "-",
        category
    )

    print("=" * 75)


    for index, chunk in enumerate(
        chunks,
        start=1
    ):

        print()
        print(
            f"CHUNK {index}"
        )

        print(
            f"Words: {len(chunk.split())}"
        )

        print("-" * 75)

        print(chunk)

        print()


EXP_016 - insect_bites_stings

CHUNK 1
Words: 40
---------------------------------------------------------------------------
Insect bites and stings Organization: NHS (National Health Service) Country context: United Kingdom (England; NHS website) Original source URL: https://www.nhs.uk/conditions/insect-bites-and-stings/ Retrieved: 2026-10-19 Representation: text extracted from the official webpage main content; wording retained. Linked pages and images are not bundled.


CHUNK 2
Words: 180
---------------------------------------------------------------------------
Insect bites and stings Insect bites or stings are not usually serious and get better in a few days. But sometimes they can become infected or cause a serious allergic reaction. Bites from some insects can also cause illnesses, such as Lyme disease from ticks, scabies from mites, and malaria from mosquitoes in certain parts of the world. Symptoms of an insect bite or sting The main symptoms of an insect bite or sting are: 

In [123]:
chunk_quality_rows = []


quality_terms = {
    "EXP_016": [
        "insect",
        "sting"
    ],

    "EXP_017": [
        "anaphylaxis"
    ]
}


for source_id, chunks in (
    prepared_new_chunks.items()
):

    combined = " ".join(
        chunks
    ).lower()


    terms_found = {
        term:
            term in combined

        for term
        in quality_terms[
            source_id
        ]
    }


    chunk_quality_rows.append(
        {
            "source_id":
                source_id,

            "category":
                new_source_definitions[
                    source_id
                ]["category"],

            "chunks":
                len(chunks),

            "words":
                sum(
                    len(chunk.split())
                    for chunk in chunks
                ),

            "required_terms_present":
                all(
                    terms_found.values()
                ),

            "contains_999":
                bool(
                    re.search(
                        r"\b999\b",
                        combined
                    )
                ),

            "contains_111":
                bool(
                    re.search(
                        r"\b111\b",
                        combined
                    )
                )
        }
    )


new_chunk_quality_df = pd.DataFrame(
    chunk_quality_rows
)


print(
    "NEW SOURCE CHUNK QUALITY AUDIT"
)

print(
    "=" * 75
)


display(
    new_chunk_quality_df
)


print()

print(
    "All required medical terms present:",
    new_chunk_quality_df[
        "required_terms_present"
    ].all()
)

NEW SOURCE CHUNK QUALITY AUDIT


,source_id,category,chunks,words,required_terms_present,contains_999,contains_111
0,EXP_016,insect_bites_stings,9,1335,True,True,True
1,EXP_017,allergic_reaction,7,1002,True,True,False



All required medical terms present: True


In [124]:
new_chunk_count = sum(
    len(chunks)
    for chunks
    in prepared_new_chunks.values()
)


expected_documents_after = (
    int(baseline_documents)
    + 2
)


expected_chunks_after = (
    int(baseline_chunks)
    + new_chunk_count
)


expected_fts_after = (
    int(baseline_fts)
    + new_chunk_count
)


print(
    "PRE-INSERTION DATABASE PLAN"
)

print(
    "=" * 75
)


print(
    "Existing documents:",
    baseline_documents
)

print(
    "New documents:",
    2
)

print(
    "Expected documents after:",
    expected_documents_after
)


print()


print(
    "Existing chunks:",
    baseline_chunks
)

print(
    "New chunks:",
    new_chunk_count
)

print(
    "Expected chunks after:",
    expected_chunks_after
)


print()


print(
    "Existing FTS5 rows:",
    baseline_fts
)

print(
    "New FTS5 rows:",
    new_chunk_count
)

print(
    "Expected FTS5 rows after:",
    expected_fts_after
)

PRE-INSERTION DATABASE PLAN
Existing documents: 27
New documents: 2
Expected documents after: 29

Existing chunks: 1795
New chunks: 16
Expected chunks after: 1811

Existing FTS5 rows: 1795
New FTS5 rows: 16
Expected FTS5 rows after: 1811


## Transactional Knowledge-Base Expansion

In [125]:
print("CATALOG COLUMNS")
print("=" * 75)

for column in missing_catalog.columns:
    print(column)


print()
print("CATALOG RECORDS")
print("=" * 75)

display(
    missing_catalog
)

CATALOG COLUMNS
source_id
category
title
organization
country_context
source_url
filename
file_type
file_size_bytes
download_status
foreign_emergency_reference
notes

CATALOG RECORDS


,source_id,category,title,organization,country_context,source_url,filename,file_type,file_size_bytes,download_status,foreign_emergency_reference,notes
0,EXP_016,insect_bites_stings,Insect bites and stings,NHS (National Health Service),United Kingdom (England),https://www.nhs.uk/conditions/insect-bites-and...,source_documents/EXP_016_insect_bites_stings_n...,text/plain; charset=utf-8,9058,downloaded_and_validated,UK 999; NHS 111 / 111 online; GP appointments;...,Verified official nhs.uk source; readable main...
1,EXP_017,allergic_reaction,Anaphylaxis,NHS (National Health Service),United Kingdom (England),https://www.nhs.uk/conditions/anaphylaxis/,source_documents/EXP_017_anaphylaxis_nhs.txt,text/plain; charset=utf-8,5880,downloaded_and_validated,UK 999 ambulance; UK hospital/prescription/ref...,Verified official nhs.uk source; readable main...


In [126]:
new_document_records = []


for _, row in missing_catalog.iterrows():

    source_id = str(
        row["source_id"]
    ).strip()


    if source_id not in {
        "EXP_016",
        "EXP_017"
    }:
        continue


    category = str(
        row["category"]
    ).strip()


    filename = str(
        row["filename"]
    ).strip()


    local_path = (
        "research/rag/source_documents/"
        + filename
    )


    record = {
        "source_id":
            source_id,

        "title":
            str(
                row["title"]
            ).strip(),

        "organization":
            str(
                row["organization"]
            ).strip(),

        "categories":
            category,

        "source_url":
            str(
                row["source_url"]
            ).strip(),

        "file_type":
            str(
                row["file_type"]
            ).strip(),

        "local_path":
            local_path,

        "country_context":
            str(
                row["country_context"]
            ).strip()
    }


    new_document_records.append(
        record
    )


new_documents_df = pd.DataFrame(
    new_document_records
)


print("DOCUMENT RECORDS PREPARED FOR SQLITE")
print("=" * 75)

display(
    new_documents_df
)


assert len(new_documents_df) == 2

assert set(
    new_documents_df[
        "source_id"
    ]
) == {
    "EXP_016",
    "EXP_017"
}


print()
print(
    "Document metadata preparation: PASS"
)

DOCUMENT RECORDS PREPARED FOR SQLITE


,source_id,title,organization,categories,source_url,file_type,local_path,country_context
0,EXP_016,Insect bites and stings,NHS (National Health Service),insect_bites_stings,https://www.nhs.uk/conditions/insect-bites-and...,text/plain; charset=utf-8,research/rag/source_documents/source_documents...,United Kingdom (England)
1,EXP_017,Anaphylaxis,NHS (National Health Service),allergic_reaction,https://www.nhs.uk/conditions/anaphylaxis/,text/plain; charset=utf-8,research/rag/source_documents/source_documents...,United Kingdom (England)



Document metadata preparation: PASS


In [127]:
conn = sqlite3.connect(
    RAG_DB_PATH
)


existing_chunk_ids = pd.read_sql_query(
    """
    SELECT chunk_id
    FROM knowledge_chunks
    """,
    conn
)[
    "chunk_id"
].tolist()


conn.close()


existing_chunk_numbers = []


for chunk_id in existing_chunk_ids:

    match = re.search(
        r"(\d+)$",
        str(chunk_id)
    )

    if match:

        existing_chunk_numbers.append(
            int(
                match.group(1)
            )
        )


next_chunk_number = (
    max(existing_chunk_numbers)
    + 1
)


new_chunk_records = []


for source_id, chunks in (
    prepared_new_chunks.items()
):

    category = (
        new_source_definitions[
            source_id
        ]["category"]
    )


    for source_index, content in enumerate(
        chunks
    ):

        chunk_id = (
            f"CHK_{next_chunk_number:05d}"
        )

        next_chunk_number += 1


        new_chunk_records.append(
            {
                "chunk_id":
                    chunk_id,

                "source_id":
                    source_id,

                "source_chunk_index":
                    source_index,

                "categories":
                    category,

                "word_count":
                    len(
                        content.split()
                    ),

                "content":
                    content
            }
        )


new_chunks_df = pd.DataFrame(
    new_chunk_records
)


print("NEW CHUNK RECORDS")
print("=" * 75)

print(
    "Rows:",
    len(new_chunks_df)
)

print(
    "First chunk:",
    new_chunks_df.iloc[0][
        "chunk_id"
    ]
)

print(
    "Last chunk:",
    new_chunks_df.iloc[-1][
        "chunk_id"
    ]
)


display(
    new_chunks_df[
        [
            "chunk_id",
            "source_id",
            "source_chunk_index",
            "categories",
            "word_count"
        ]
    ]
)


assert (
    len(new_chunks_df)
    ==
    new_chunk_count
)


assert not set(
    new_chunks_df[
        "chunk_id"
    ]
).intersection(
    set(existing_chunk_ids)
)


print()
print(
    "Chunk-ID validation: PASS"
)

NEW CHUNK RECORDS
Rows: 16
First chunk: CHK_01796
Last chunk: CHK_01811


,chunk_id,source_id,source_chunk_index,categories,word_count
0,CHK_01796,EXP_016,0,insect_bites_stings,40
1,CHK_01797,EXP_016,1,insect_bites_stings,180
2,CHK_01798,EXP_016,2,insect_bites_stings,180
3,CHK_01799,EXP_016,3,insect_bites_stings,180
4,CHK_01800,EXP_016,4,insect_bites_stings,180
5,CHK_01801,EXP_016,5,insect_bites_stings,180
6,CHK_01802,EXP_016,6,insect_bites_stings,180
7,CHK_01803,EXP_016,7,insect_bites_stings,180
8,CHK_01804,EXP_016,8,insect_bites_stings,35
9,CHK_01805,EXP_017,0,allergic_reaction,37



Chunk-ID validation: PASS


In [128]:
conn = sqlite3.connect(
    RAG_DB_PATH
)


try:

    conn.execute(
        "BEGIN"
    )


    # -----------------------------------------
    # Insert document metadata
    # -----------------------------------------

    for _, row in (
        new_documents_df.iterrows()
    ):

        conn.execute(
            """
            INSERT INTO documents
            (
                source_id,
                title,
                organization,
                categories,
                source_url,
                file_type,
                local_path,
                country_context
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
            """,
            (
                row["source_id"],
                row["title"],
                row["organization"],
                row["categories"],
                row["source_url"],
                row["file_type"],
                row["local_path"],
                row["country_context"]
            )
        )


    # -----------------------------------------
    # Insert normal knowledge chunks
    # -----------------------------------------

    for _, row in (
        new_chunks_df.iterrows()
    ):

        conn.execute(
            """
            INSERT INTO knowledge_chunks
            (
                chunk_id,
                source_id,
                source_chunk_index,
                categories,
                word_count,
                content
            )
            VALUES (?, ?, ?, ?, ?, ?)
            """,
            (
                row["chunk_id"],
                row["source_id"],
                int(
                    row[
                        "source_chunk_index"
                    ]
                ),
                row["categories"],
                int(
                    row[
                        "word_count"
                    ]
                ),
                row["content"]
            )
        )


        # -------------------------------------
        # Insert corresponding FTS5 record
        # -------------------------------------

        conn.execute(
            """
            INSERT INTO knowledge_chunks_fts
            (
                chunk_id,
                source_id,
                categories,
                content
            )
            VALUES (?, ?, ?, ?)
            """,
            (
                row["chunk_id"],
                row["source_id"],
                row["categories"],
                row["content"]
            )
        )


    # -----------------------------------------
    # Validate before committing
    # -----------------------------------------

    document_count = conn.execute(
        """
        SELECT COUNT(*)
        FROM documents
        """
    ).fetchone()[0]


    chunk_count = conn.execute(
        """
        SELECT COUNT(*)
        FROM knowledge_chunks
        """
    ).fetchone()[0]


    fts_count = conn.execute(
        """
        SELECT COUNT(*)
        FROM knowledge_chunks_fts
        """
    ).fetchone()[0]


    exp_count = conn.execute(
        """
        SELECT COUNT(*)
        FROM documents
        WHERE source_id IN
        ('EXP_016', 'EXP_017')
        """
    ).fetchone()[0]


    assert (
        document_count
        ==
        expected_documents_after
    )


    assert (
        chunk_count
        ==
        expected_chunks_after
    )


    assert (
        fts_count
        ==
        expected_fts_after
    )


    assert exp_count == 2


    conn.commit()


    print(
        "TRANSACTION COMMITTED SUCCESSFULLY"
    )


except Exception as error:

    conn.rollback()

    print(
        "TRANSACTION ROLLED BACK"
    )

    print(
        "Reason:",
        repr(error)
    )

    raise


finally:

    conn.close()

TRANSACTION ROLLED BACK
Reason: IntegrityError('UNIQUE constraint failed: documents.source_id')


IntegrityError: UNIQUE constraint failed: documents.source_id

In [129]:
conn = sqlite3.connect(
    RAG_DB_PATH
)


final_documents = pd.read_sql_query(
    """
    SELECT COUNT(*) AS count
    FROM documents
    """,
    conn
).iloc[0]["count"]


final_chunks = pd.read_sql_query(
    """
    SELECT COUNT(*) AS count
    FROM knowledge_chunks
    """,
    conn
).iloc[0]["count"]


final_fts = pd.read_sql_query(
    """
    SELECT COUNT(*) AS count
    FROM knowledge_chunks_fts
    """,
    conn
).iloc[0]["count"]


new_source_counts = pd.read_sql_query(
    """
    SELECT
        source_id,
        COUNT(*) AS chunk_count
    FROM knowledge_chunks
    WHERE source_id IN
        ('EXP_016', 'EXP_017')
    GROUP BY source_id
    ORDER BY source_id
    """,
    conn
)


old_chunk_count = pd.read_sql_query(
    """
    SELECT COUNT(*) AS count
    FROM knowledge_chunks
    WHERE source_id NOT IN
        ('EXP_016', 'EXP_017')
    """,
    conn
).iloc[0]["count"]


orphan_count = pd.read_sql_query(
    """
    SELECT COUNT(*) AS count
    FROM knowledge_chunks kc

    LEFT JOIN documents d
        ON d.source_id = kc.source_id

    WHERE d.source_id IS NULL
    """,
    conn
).iloc[0]["count"]


conn.close()


print("POST-INSERTION DATABASE INTEGRITY")
print("=" * 75)

print(
    "Documents:",
    final_documents
)

print(
    "Knowledge chunks:",
    final_chunks
)

print(
    "FTS5 rows:",
    final_fts
)

print(
    "Original chunks preserved:",
    old_chunk_count
)

print(
    "Orphan chunks:",
    orphan_count
)


print()
print(
    "New source chunk counts:"
)

display(
    new_source_counts
)


integrity_pass = (
    final_documents
    ==
    expected_documents_after

    and

    final_chunks
    ==
    expected_chunks_after

    and

    final_fts
    ==
    expected_fts_after

    and

    old_chunk_count
    ==
    baseline_chunks

    and

    orphan_count
    == 0
)


print()
print(
    "DATABASE EXPANSION INTEGRITY:",
    integrity_pass
)

POST-INSERTION DATABASE INTEGRITY
Documents: 27
Knowledge chunks: 1795
FTS5 rows: 1795
Original chunks preserved: 1779
Orphan chunks: 0

New source chunk counts:


,source_id,chunk_count
0,EXP_016,9
1,EXP_017,7



DATABASE EXPANSION INTEGRITY: False


In [130]:
coverage_retests = [
    {
        "category":
            "insect_bites_stings",

        "question":
            (
                "What should I do for insect bites "
                "or stings while hiking?"
            )
    },

    {
        "category":
            "allergic_reaction",

        "question":
            (
                "What should I do if a hiker develops "
                "a serious allergic reaction?"
            )
    }
]


for case in coverage_retests:

    results = (
        retrieve_offline_knowledge_v2(
            question=case[
                "question"
            ],
            category=case[
                "category"
            ],
            limit=4
        )
    )


    print()
    print("=" * 75)

    print(
        case["category"].upper()
    )

    print("=" * 75)


    print(
        "Retrieved chunks:",
        len(results)
    )


    if not results.empty:

        print(
            "Sources:",
            results[
                "source_id"
            ].drop_duplicates().tolist()
        )

        print(
            "Countries:",
            results[
                "country_context"
            ].fillna(
                "Not specified"
            ).drop_duplicates().tolist()
        )


        display(
            results[
                [
                    "chunk_id",
                    "source_id",
                    "categories",
                    "country_context",
                    "rank"
                ]
            ]
        )


INSECT_BITES_STINGS
Retrieved chunks: 4
Sources: ['EXP_016']
Countries: ['United Kingdom (England)']


,chunk_id,source_id,categories,country_context,rank
0,CHK_01780,EXP_016,insect_bites_stings,United Kingdom (England),-19.914332
1,CHK_01781,EXP_016,insect_bites_stings,United Kingdom (England),-18.932659
2,CHK_01782,EXP_016,insect_bites_stings,United Kingdom (England),-13.195239
3,CHK_01783,EXP_016,insect_bites_stings,United Kingdom (England),-11.131358



ALLERGIC_REACTION
Retrieved chunks: 4
Sources: ['EXP_017']
Countries: ['United Kingdom (England)']


,chunk_id,source_id,categories,country_context,rank
0,CHK_01791,EXP_017,allergic_reaction,United Kingdom (England),-12.981621
1,CHK_01794,EXP_017,allergic_reaction,United Kingdom (England),-11.885205
2,CHK_01790,EXP_017,allergic_reaction,United Kingdom (England),-9.221939
3,CHK_01793,EXP_017,allergic_reaction,United Kingdom (England),-7.591401


## Retrieval Evaluation After Knowledge-Base Expansion

In [131]:
expanded_retrieval_rows = []


for case in rag_evaluation_cases:

    category = case[
        "category"
    ]

    question = case[
        "question"
    ]


    results = (
        retrieve_offline_knowledge_v2(
            question=question,
            category=category,
            limit=4
        )
    )


    retrieval_success = (
        not results.empty
    )


    if retrieval_success:

        category_consistent = (
            results[
                "categories"
            ]
            .fillna("")
            .apply(
                lambda value:
                    category
                    in [
                        item.strip()
                        for item
                        in str(value).split(",")
                    ]
            )
            .all()
        )


        source_ids = (
            results[
                "source_id"
            ]
            .drop_duplicates()
            .tolist()
        )


        countries = (
            results[
                "country_context"
            ]
            .fillna(
                "Not specified"
            )
            .drop_duplicates()
            .tolist()
        )

    else:

        category_consistent = False
        source_ids = []
        countries = []


    expanded_retrieval_rows.append(
        {
            "category":
                category,

            "retrieval_success":
                retrieval_success,

            "category_consistent":
                category_consistent,

            "retrieved_chunks":
                len(results),

            "unique_sources":
                len(source_ids),

            "source_ids":
                ", ".join(
                    source_ids
                ),

            "country_contexts":
                ", ".join(
                    countries
                )
        }
    )


expanded_retrieval_df = pd.DataFrame(
    expanded_retrieval_rows
)


display(
    expanded_retrieval_df
)

,category,retrieval_success,category_consistent,retrieved_chunks,unique_sources,source_ids,country_contexts
0,general_first_aid,True,True,3,1,SRC_016,Not specified
1,hypothermia,True,True,3,1,SRC_014,Not specified
2,heat_illness,True,True,4,1,SRC_015,Not specified
3,heavy_rain,True,True,2,1,SRC_006,Not specified
4,strong_wind,True,True,4,1,SRC_007,Not specified
5,water_safety,True,True,4,1,SRC_017,Not specified
6,remote_emergency,True,True,4,2,"SRC_010, SRC_009",Not specified
7,snakebite,True,True,4,2,"EXP_001, EXP_002","Sri Lanka, International"
8,insect_bites_stings,True,True,4,1,EXP_016,United Kingdom (England)
9,fracture_sprain,True,True,4,2,"EXP_004, EXP_005",United Kingdom


In [132]:
expanded_total = len(
    expanded_retrieval_df
)


expanded_successes = int(
    expanded_retrieval_df[
        "retrieval_success"
    ].sum()
)


expanded_consistent = int(
    expanded_retrieval_df[
        "category_consistent"
    ].sum()
)


expanded_success_rate = (
    expanded_successes
    /
    expanded_total
)


expanded_consistency_rate = (
    expanded_consistent
    /
    expanded_total
)


expanded_mean_chunks = (
    expanded_retrieval_df[
        "retrieved_chunks"
    ].mean()
)


expanded_mean_sources = (
    expanded_retrieval_df[
        "unique_sources"
    ].mean()
)


print(
    "EXPANDED OFFLINE RAG RETRIEVAL EVALUATION"
)

print(
    "=" * 75
)


print(
    "Evaluation cases:",
    expanded_total
)

print(
    "Successful retrievals:",
    f"{expanded_successes}/{expanded_total}"
)

print(
    "Retrieval success rate:",
    f"{expanded_success_rate:.2%}"
)

print(
    "Category-consistent cases:",
    f"{expanded_consistent}/{expanded_total}"
)

print(
    "Category consistency rate:",
    f"{expanded_consistency_rate:.2%}"
)

print(
    "Mean retrieved chunks:",
    f"{expanded_mean_chunks:.2f}"
)

print(
    "Mean unique sources:",
    f"{expanded_mean_sources:.2f}"
)


print()
print("BEFORE VS AFTER")
print("-" * 75)

print(
    "Before expansion:",
    "16/18 (88.89%)"
)

print(
    "After expansion:",
    f"{expanded_successes}/18 "
    f"({expanded_success_rate:.2%})"
)


remaining_failures = (
    expanded_retrieval_df[
        ~expanded_retrieval_df[
            "retrieval_success"
        ]
    ]
)


print()

if remaining_failures.empty:

    print(
        "Remaining retrieval failures: 0"
    )

else:

    print(
        "Remaining retrieval failures:"
    )

    display(
        remaining_failures
    )

EXPANDED OFFLINE RAG RETRIEVAL EVALUATION
Evaluation cases: 18
Successful retrievals: 18/18
Retrieval success rate: 100.00%
Category-consistent cases: 18/18
Category consistency rate: 100.00%
Mean retrieved chunks: 3.67
Mean unique sources: 1.22

BEFORE VS AFTER
---------------------------------------------------------------------------
Before expansion: 16/18 (88.89%)
After expansion: 18/18 (100.00%)

Remaining retrieval failures: 0


In [133]:
EVALUATION_DIR = (
    RAG_ROOT
    / "evaluation"
)


EVALUATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)


expanded_retrieval_path = (
    EVALUATION_DIR
    / "expanded_18_category_retrieval_evaluation.csv"
)


expanded_retrieval_df.to_csv(
    expanded_retrieval_path,
    index=False
)


print(
    "Saved:",
    expanded_retrieval_path
)

print(
    "Exists:",
    expanded_retrieval_path.exists()
)

print(
    "Rows:",
    len(
        pd.read_csv(
            expanded_retrieval_path
        )
    )
)

Saved: /Users/mirshad/Desktop/Decentralized-Hiking-AI/research/rag/evaluation/expanded_18_category_retrieval_evaluation.csv
Exists: True
Rows: 18


### Retrieval Evaluation Interpretation

## Country-Context Normalization


In [134]:
def normalize_country_context(country_context):

    if pd.isna(country_context):
        return "not specified"

    country = (
        str(country_context)
        .strip()
        .lower()
    )

    if not country:
        return "not specified"


    if "sri lanka" in country:
        return "sri lanka"


    if (
        "united kingdom" in country
        or country == "uk"
        or country.startswith("uk ")
    ):
        return "united kingdom"


    if (
        "united states" in country
        or country == "usa"
        or country == "us"
    ):
        return "united states"


    if "new zealand" in country:
        return "new zealand"


    if "international" in country:
        return "international"


    if "not specified" in country:
        return "not specified"


    return country


print(
    "Country-context normalization ready."
)

Country-context normalization ready.


In [135]:
country_normalization_tests = [
    "United Kingdom",
    "United Kingdom (England)",
    "United States",
    "New Zealand",
    "Sri Lanka",
    "International",
    "Not specified",
    None
]


print("COUNTRY NORMALIZATION TEST")
print("=" * 75)


for country in country_normalization_tests:

    normalized = (
        normalize_country_context(
            country
        )
    )

    print(
        f"{str(country):<30} -> "
        f"{normalized}"
    )

COUNTRY NORMALIZATION TEST
United Kingdom                 -> united kingdom
United Kingdom (England)       -> united kingdom
United States                  -> united states
New Zealand                    -> new zealand
Sri Lanka                      -> sri lanka
International                  -> international
Not specified                  -> not specified
None                           -> not specified


In [136]:
FOREIGN_LOCALIZATION_MARKERS = {

    "new zealand": [
        "new zealand",
        "linz",
        "topo50",
        "doc visitor",
        "mountain safety council"
    ],

    "united kingdom": [
        "united kingdom",
        "nhs",
        "met office",
        "mountain weather information service"
    ],

    "united states": [
        "united states",
        "national weather service",
        "nps.gov"
    ]
}


def contains_foreign_specific_reference(
    sentence,
    country_context
):

    normalized_country = (
        normalize_country_context(
            country_context
        )
    )


    markers = (
        FOREIGN_LOCALIZATION_MARKERS.get(
            normalized_country,
            []
        )
    )


    sentence_lower = (
        sentence.lower()
    )


    return any(
        marker in sentence_lower
        for marker in markers
    )


print(
    "Normalized foreign-reference detector ready."
)

Normalized foreign-reference detector ready.


In [137]:
def filter_foreign_evidence(
    content,
    country_context
):

    normalized_country = (
        normalize_country_context(
            country_context
        )
    )


    # Local or genuinely international evidence
    # does not require country-specific filtering.
    if normalized_country in {
        "sri lanka",
        "international",
        "not specified"
    }:

        return content.strip()


    sentences = re.split(
        r"(?<=[.!?])\s+",
        content
    )


    retained = []


    for sentence in sentences:

        sentence = sentence.strip()

        if not sentence:
            continue


        contains_foreign_number = any(
            re.search(
                rf"\b{re.escape(number)}\b",
                sentence
            )

            for number
            in FOREIGN_EMERGENCY_NUMBERS
        )


        if contains_foreign_number:
            continue


        if contains_foreign_specific_reference(
            sentence,
            normalized_country
        ):
            continue


        retained.append(
            sentence
        )


    return " ".join(
        retained
    ).strip()


print(
    "Normalized foreign-evidence filter ready."
)

Normalized foreign-evidence filter ready.


In [138]:
nhs_localization_tests = []


for source_id in [
    "EXP_016",
    "EXP_017"
]:

    conn = sqlite3.connect(
        RAG_DB_PATH
    )


    source_chunks = pd.read_sql_query(
        """
        SELECT
            kc.content,
            d.country_context
        FROM knowledge_chunks kc

        JOIN documents d
            ON d.source_id = kc.source_id

        WHERE kc.source_id = ?
        """,
        conn,
        params=[source_id]
    )


    conn.close()


    original_text = " ".join(
        source_chunks[
            "content"
        ].tolist()
    )


    country = (
        source_chunks.iloc[0][
            "country_context"
        ]
    )


    filtered_text = (
        filter_foreign_evidence(
            content=original_text,
            country_context=country
        )
    )


    audit = {
        "source_id":
            source_id,

        "original_country":
            country,

        "normalized_country":
            normalize_country_context(
                country
            ),

        "original_contains_999":
            bool(
                re.search(
                    r"\b999\b",
                    original_text
                )
            ),

        "filtered_contains_999":
            bool(
                re.search(
                    r"\b999\b",
                    filtered_text
                )
            ),

        "original_contains_111":
            bool(
                re.search(
                    r"\b111\b",
                    original_text
                )
            ),

        "filtered_contains_111":
            bool(
                re.search(
                    r"\b111\b",
                    filtered_text
                )
            ),

        "filtered_contains_nhs":
            "nhs" in filtered_text.lower(),

        "filtered_characters":
            len(filtered_text)
    }


    nhs_localization_tests.append(
        audit
    )


nhs_localization_df = pd.DataFrame(
    nhs_localization_tests
)


print(
    "NHS LOCALIZATION FILTER AUDIT"
)

print(
    "=" * 75
)


display(
    nhs_localization_df
)

NHS LOCALIZATION FILTER AUDIT


,source_id,original_country,normalized_country,original_contains_999,filtered_contains_999,original_contains_111,filtered_contains_111,filtered_contains_nhs,filtered_characters
0,EXP_016,United Kingdom (England),united kingdom,True,False,True,False,False,7091
1,EXP_017,United Kingdom (England),united kingdom,True,False,False,False,False,4585


In [139]:
nhs_filter_pass = (

    (
        ~nhs_localization_df[
            "filtered_contains_999"
        ]
    ).all()

    and

    (
        ~nhs_localization_df[
            "filtered_contains_111"
        ]
    ).all()

    and

    (
        ~nhs_localization_df[
            "filtered_contains_nhs"
        ]
    ).all()

    and

    (
        nhs_localization_df[
            "filtered_characters"
        ]
        > 0
    ).all()
)


print(
    "NHS FOREIGN-CONTEXT FILTER PASS:",
    nhs_filter_pass
)

NHS FOREIGN-CONTEXT FILTER PASS: True


## Systematic Hybrid Offline RAG Generation Evaluation

In [140]:
ANSWER_LOCALIZATION_TERMS = [
    "new zealand",
    "linz",
    "topo50",
    "doc visitor",
    "mountain safety council",
    "united kingdom",
    "nhs",
    "met office",
    "mountain weather information service",
    "united states",
    "national weather service",
    "nps.gov"
]


def audit_generated_answer(
    answer,
    category
):

    answer_text = str(
        answer
    )

    answer_lower = (
        answer_text.lower()
    )


    foreign_numbers = []

    for number in (
        FOREIGN_EMERGENCY_NUMBERS
    ):

        if re.search(
            rf"\b{re.escape(number)}\b",
            answer_text
        ):
            foreign_numbers.append(
                number
            )


    localization_terms = [
        term
        for term
        in ANSWER_LOCALIZATION_TERMS
        if term in answer_lower
    ]


    prompt_echo = (
        detect_prompt_echo(
            answer_text
        )
    )


    return {
        "foreign_numbers":
            foreign_numbers,

        "localization_terms":
            localization_terms,

        "prompt_echo":
            prompt_echo[
                "prompt_echo_detected"
            ],

        "contains_1990":
            bool(
                re.search(
                    r"\b1990\b",
                    answer_text
                )
            )
    }


print(
    "Answer-level safety auditor ready."
)

Answer-level safety auditor ready.


In [141]:
generation_sanity_cases = [
    {
        "category":
            "insect_bites_stings",

        "question":
            (
                "What should I do for insect bites "
                "or stings while hiking?"
            )
    },

    {
        "category":
            "allergic_reaction",

        "question":
            (
                "What should I do if a hiker develops "
                "a serious allergic reaction?"
            )
    },

    {
        "category":
            "sri_lanka_emergency",

        "question":
            (
                "What ambulance service can I contact "
                "for a medical emergency in Sri Lanka?"
            )
    }
]


generation_sanity_results = {}


for case in generation_sanity_cases:

    print()
    print("=" * 75)

    print(
        "TESTING:",
        case["category"]
    )

    print("=" * 75)


    result = (
        offline_hybrid_rag_answer(
            question=case[
                "question"
            ],
            category=case[
                "category"
            ],
            limit=4
        )
    )


    audit = (
        audit_generated_answer(
            answer=result[
                "answer"
            ],
            category=case[
                "category"
            ]
        )
    )


    generation_sanity_results[
        case["category"]
    ] = {
        "result":
            result,

        "audit":
            audit
    }


    print(
        "Success:",
        result.get(
            "success"
        )
    )

    print(
        "Answer mode:",
        result.get(
            "answer_mode"
        )
    )

    print(
        "Sources:",
        result.get(
            "source_ids"
        )
    )

    print(
        "Foreign numbers:",
        audit[
            "foreign_numbers"
        ]
    )

    print(
        "Localization terms:",
        audit[
            "localization_terms"
        ]
    )

    print(
        "Prompt echo:",
        audit[
            "prompt_echo"
        ]
    )

    print(
        "Contains 1990:",
        audit[
            "contains_1990"
        ]
    )

    print()
    print("ANSWER")
    print("-" * 75)

    print(
        result[
            "final_output"
        ]
    )


TESTING: insect_bites_stings
Success: True
Answer mode: local_llm_rag
Sources: ['EXP_016']
Foreign numbers: []
Localization terms: []
Prompt echo: False
Contains 1990: False

ANSWER
---------------------------------------------------------------------------
Based on the provided context, here is the practical instruction for handling insect bites or stings while hiking:

**Immediate Action:**
1.  **Remove the Stinger:** If a stinger is present, remove it carefully. For a bee, wasp, or hornet stinger, use a fingernail or the edge of a bank card to scrape or brush the stinger sideways. Do not use tweezers to pull out the stinger as this could squeeze poison out of it.
2.  **Wash the Area:** Wash the affected area with soap and water.
3.  **Monitor Symptoms:** Watch for signs of infection or allergic reaction. If the area becomes more swollen and painful, or if you notice a rash that looks like a bullseye, you may need to see a GP.

*Note: Do not

Sources: EXP_016

TESTING: allergic_reac

In [142]:
sanity_rows = []


for category, data in (
    generation_sanity_results.items()
):

    result = data[
        "result"
    ]

    audit = data[
        "audit"
    ]


    no_foreign_leakage = (
        len(
            audit[
                "foreign_numbers"
            ]
        )
        == 0

        and

        len(
            audit[
                "localization_terms"
            ]
        )
        == 0
    )


    if (
        category
        ==
        "sri_lanka_emergency"
    ):

        route_pass = (
            result.get(
                "answer_mode"
            )
            ==
            "verified_deterministic"

            and

            audit[
                "contains_1990"
            ]
        )

    else:

        route_pass = (
            result.get(
                "answer_mode"
            )
            ==
            "local_llm_rag"
        )


    sanity_rows.append(
        {
            "category":
                category,

            "success":
                result.get(
                    "success",
                    False
                ),

            "answer_mode":
                result.get(
                    "answer_mode"
                ),

            "no_foreign_leakage":
                no_foreign_leakage,

            "prompt_echo":
                audit[
                    "prompt_echo"
                ],

            "route_pass":
                route_pass
        }
    )


generation_sanity_df = pd.DataFrame(
    sanity_rows
)


display(
    generation_sanity_df
)


sanity_pass = (
    generation_sanity_df[
        "success"
    ].all()

    and

    generation_sanity_df[
        "no_foreign_leakage"
    ].all()

    and

    (
        ~generation_sanity_df[
            "prompt_echo"
        ]
    ).all()

    and

    generation_sanity_df[
        "route_pass"
    ].all()
)


print()
print(
    "GENERATION SANITY PASS:",
    sanity_pass
)

,category,success,answer_mode,no_foreign_leakage,prompt_echo,route_pass
0,insect_bites_stings,True,local_llm_rag,True,False,True
1,allergic_reaction,False,local_llm_rag,True,False,True
2,sri_lanka_emergency,True,verified_deterministic,True,False,True



GENERATION SANITY PASS: False


## Investigation of Allergic-Reaction Generation Failure

In [143]:
allergic_debug = (
    generation_sanity_results[
        "allergic_reaction"
    ][
        "result"
    ]
)


print(
    "ALLERGIC-REACTION RESULT OBJECT"
)

print(
    "=" * 75
)


print(
    "Available keys:"
)


for key in allergic_debug.keys():

    print(
        "-",
        key
    )


print()
print(
    "FULL RESULT"
)

print(
    "=" * 75
)


for key, value in (
    allergic_debug.items()
):

    print()
    print(
        f"{key}:"
    )

    print(
        value
    )

ALLERGIC-REACTION RESULT OBJECT
Available keys:
- success
- pipeline_success
- answer_generated
- fallback_used
- question
- category
- answer
- final_output
- source_ids
- retrieved_chunks
- country_contexts
- foreign_numbers_detected
- generation_time
- retrieval_results
- answer_mode

FULL RESULT

success:
False

pipeline_success:
True

answer_generated:
False

fallback_used:
True

question:
What should I do if a hiker develops a serious allergic reaction?

category:
allergic_reaction

answer:
The offline knowledge base does not contain enough information to answer this safely.

final_output:
The offline knowledge base does not contain enough information to answer this safely.

Sources: EXP_017

source_ids:
['EXP_017']

retrieved_chunks:
4

country_contexts:
['United Kingdom (England)']

foreign_numbers_detected:
[]

generation_time:
3.501359040999887

retrieval_results:
    chunk_id source_id                   organization        title  \
0  CHK_01791   EXP_017  NHS (National Healt

In [144]:
allergic_question = (
    "What should I do if a hiker develops "
    "a serious allergic reaction?"
)


allergic_retrieval = (
    retrieve_offline_knowledge_v2(
        question=allergic_question,
        category="allergic_reaction",
        limit=4
    )
)


print(
    "ALLERGIC-REACTION RETRIEVAL"
)

print(
    "=" * 75
)


print(
    "Retrieved chunks:",
    len(
        allergic_retrieval
    )
)


display(
    allergic_retrieval[
        [
            "chunk_id",
            "source_id",
            "categories",
            "country_context",
            "rank"
        ]
    ]
)

ALLERGIC-REACTION RETRIEVAL
Retrieved chunks: 4


,chunk_id,source_id,categories,country_context,rank
0,CHK_01791,EXP_017,allergic_reaction,United Kingdom (England),-12.981621
1,CHK_01794,EXP_017,allergic_reaction,United Kingdom (England),-11.885205
2,CHK_01790,EXP_017,allergic_reaction,United Kingdom (England),-9.221939
3,CHK_01793,EXP_017,allergic_reaction,United Kingdom (England),-7.591401


In [145]:
allergic_filtered_rows = []


for _, row in (
    allergic_retrieval.iterrows()
):

    original_content = (
        row[
            "content"
        ]
    )


    filtered_content = (
        filter_foreign_evidence(
            content=original_content,
            country_context=row[
                "country_context"
            ]
        )
    )


    allergic_filtered_rows.append(
        {
            "chunk_id":
                row[
                    "chunk_id"
                ],

            "source_id":
                row[
                    "source_id"
                ],

            "country_context":
                row[
                    "country_context"
                ],

            "original_chars":
                len(
                    original_content
                ),

            "filtered_chars":
                len(
                    filtered_content
                ),

            "filtered_content":
                filtered_content
        }
    )


allergic_filtered_df = pd.DataFrame(
    allergic_filtered_rows
)


display(
    allergic_filtered_df[
        [
            "chunk_id",
            "source_id",
            "original_chars",
            "filtered_chars"
        ]
    ]
)


print()
print(
    "TOTAL ORIGINAL CHARACTERS:",
    allergic_filtered_df[
        "original_chars"
    ].sum()
)

print(
    "TOTAL FILTERED CHARACTERS:",
    allergic_filtered_df[
        "filtered_chars"
    ].sum()
)

,chunk_id,source_id,original_chars,filtered_chars
0,CHK_01791,EXP_017,955,955
1,CHK_01794,EXP_017,1074,1074
2,CHK_01790,EXP_017,1045,864
3,CHK_01793,EXP_017,1040,1040



TOTAL ORIGINAL CHARACTERS: 4114
TOTAL FILTERED CHARACTERS: 3933


In [146]:
print(
    "SANITIZED ALLERGIC-REACTION EVIDENCE"
)

print(
    "=" * 75
)


for _, row in (
    allergic_filtered_df.iterrows()
):

    print()
    print(
        f"[{row['chunk_id']}]"
    )

    print(
        "-" * 75
    )

    print(
        row[
            "filtered_content"
        ]
    )

SANITIZED ALLERGIC-REACTION EVIDENCE

[CHK_01791]
---------------------------------------------------------------------------
you're breathing very fast or struggling to breathe (you may become very wheezy or feel like you're choking or gasping for air) - your throat feels tight or you're struggling to swallow - your skin, tongue or lips turn blue, grey or pale (if you have black or brown skin, this may be easier to see on the palms of your hands or soles of your feet) - you suddenly become very confused, drowsy or dizzy - someone faints and cannot be woken up - a child is limp, floppy or not responding like they normally do (their head may fall to the side, backwards or forwards, or they may find it difficult to lift their head or focus on your face) You or the person who's unwell may also have a rash that's swollen, raised or itchy. These can be signs of a serious allergic reaction and may need immediate treatment in hospital. What to do if you have anaphylaxis Follow these steps if 

In [147]:
allergic_sanitized_text = " ".join(
    allergic_filtered_df[
        "filtered_content"
    ].tolist()
)


diagnostic_terms = [
    "anaphylaxis",
    "allergic",
    "adrenaline",
    "auto-injector",
    "injector",
    "breathing",
    "swelling",
    "emergency"
]


allergic_term_audit = []


for term in diagnostic_terms:

    allergic_term_audit.append(
        {
            "term":
                term,

            "present":
                term
                in allergic_sanitized_text.lower()
        }
    )


allergic_term_df = pd.DataFrame(
    allergic_term_audit
)


display(
    allergic_term_df
)


print()
print(
    "Sanitized evidence characters:",
    len(
        allergic_sanitized_text
    )
)

,term,present
0,anaphylaxis,True
1,allergic,True
2,adrenaline,True
3,auto-injector,True
4,injector,True
5,breathing,True
6,swelling,True
7,emergency,True



Sanitized evidence characters: 3936


In [148]:
allergic_repeat_result = (
    offline_hybrid_rag_answer(
        question=allergic_question,
        category="allergic_reaction",
        limit=4
    )
)


print(
    "CONTROLLED ALLERGIC-REACTION REPEAT"
)

print(
    "=" * 75
)


print(
    "Success:",
    allergic_repeat_result.get(
        "success"
    )
)

print(
    "Answer mode:",
    allergic_repeat_result.get(
        "answer_mode"
    )
)

print(
    "Sources:",
    allergic_repeat_result.get(
        "source_ids"
    )
)


print()
print(
    "Answer:"
)

print(
    "-" * 75
)

print(
    allergic_repeat_result.get(
        "final_output"
    )
)


print()
print(
    "Raw answer:"
)

print(
    "-" * 75
)

print(
    allergic_repeat_result.get(
        "answer"
    )
)

CONTROLLED ALLERGIC-REACTION REPEAT
Success: False
Answer mode: local_llm_rag
Sources: ['EXP_017']

Answer:
---------------------------------------------------------------------------
The offline knowledge base does not contain enough information to answer this safely.

Sources: EXP_017

Raw answer:
---------------------------------------------------------------------------
The offline knowledge base does not contain enough information to answer this safely.


## Allergic-Reaction Failure Diagnosis

In [149]:
allergic_retrieval_7 = (
    retrieve_offline_knowledge_v2(
        question=allergic_question,
        category="allergic_reaction",
        limit=7
    )
)


print(
    "ALLERGIC-REACTION RETRIEVAL — LIMIT 7"
)

print("=" * 75)

print(
    "Retrieved chunks:",
    len(allergic_retrieval_7)
)


print()
print("Returned columns:")

print(
    allergic_retrieval_7.columns.tolist()
)


print()
display(
    allergic_retrieval_7[
        [
            "chunk_id",
            "source_id",
            "categories",
            "country_context",
            "word_count",
            "rank"
        ]
    ]
)

ALLERGIC-REACTION RETRIEVAL — LIMIT 7
Retrieved chunks: 6

Returned columns:
['chunk_id', 'source_id', 'organization', 'title', 'country_context', 'categories', 'content', 'word_count', 'rank']



,chunk_id,source_id,categories,country_context,word_count,rank
0,CHK_01791,EXP_017,allergic_reaction,United Kingdom (England),180,-12.981621
1,CHK_01794,EXP_017,allergic_reaction,United Kingdom (England),180,-11.885205
2,CHK_01790,EXP_017,allergic_reaction,United Kingdom (England),180,-9.221939
3,CHK_01793,EXP_017,allergic_reaction,United Kingdom (England),180,-7.591401
4,CHK_01795,EXP_017,allergic_reaction,United Kingdom (England),65,-7.223896
5,CHK_01792,EXP_017,allergic_reaction,United Kingdom (England),180,-3.261589


In [150]:
retrieved_chunk_ids = (
    allergic_retrieval_7[
        "chunk_id"
    ].tolist()
)


placeholders = ",".join(
    ["?"] * len(
        retrieved_chunk_ids
    )
)


conn = sqlite3.connect(
    RAG_DB_PATH
)


allergic_document_order = (
    pd.read_sql_query(
        f"""
        SELECT
            kc.chunk_id,
            kc.source_id,
            kc.source_chunk_index,
            kc.categories,
            kc.word_count,
            kc.content,
            d.country_context

        FROM knowledge_chunks kc

        JOIN documents d
            ON d.source_id = kc.source_id

        WHERE kc.chunk_id IN ({placeholders})

        ORDER BY
            kc.source_id,
            kc.source_chunk_index
        """,
        conn,
        params=retrieved_chunk_ids
    )
)


conn.close()


print(
    "EXP_017 — RETRIEVED CHUNKS IN DOCUMENT ORDER"
)

print("=" * 75)


display(
    allergic_document_order[
        [
            "chunk_id",
            "source_id",
            "source_chunk_index",
            "word_count"
        ]
    ]
)


for _, row in (
    allergic_document_order.iterrows()
):

    print()

    print(
        "Chunk:",
        row["chunk_id"],
        "| Source index:",
        row["source_chunk_index"]
    )

    print("-" * 75)

    print(
        row["content"]
    )

EXP_017 — RETRIEVED CHUNKS IN DOCUMENT ORDER


,chunk_id,source_id,source_chunk_index,word_count
0,CHK_01790,EXP_017,1,180
1,CHK_01791,EXP_017,2,180
2,CHK_01792,EXP_017,3,180
3,CHK_01793,EXP_017,4,180
4,CHK_01794,EXP_017,5,180
5,CHK_01795,EXP_017,6,65



Chunk: CHK_01790 | Source index: 1
---------------------------------------------------------------------------
Anaphylaxis Anaphylaxis is a life-threatening allergic reaction that happens very quickly. It can be caused by food, medicine or insect stings. Call 999 if you think you or someone else is having an anaphylactic reaction. Symptoms of anaphylaxis Symptoms of anaphylaxis happen very quickly. They usually start within minutes of coming into contact with something you're allergic to, such as a food, medicine or insect sting. Symptoms include: - swelling of your throat and tongue - difficulty breathing or breathing very fast - difficulty swallowing, tightness in your throat or a hoarse voice - wheezing, coughing or noisy breathing - feeling tired or confused - feeling faint, dizzy or fainting - skin that feels cold to the touch - blue, grey or pale skin, lips or tongue – if you have brown or black skin, this may be easier to see on the palms of your hands or soles of your feet You

In [151]:
allergic_filtered_7 = []


for _, row in (
    allergic_document_order.iterrows()
):

    filtered = (
        filter_foreign_evidence(
            content=row["content"],
            country_context=row[
                "country_context"
            ]
        )
    )


    if filtered.strip():

        allergic_filtered_7.append(
            {
                "chunk_id":
                    row["chunk_id"],

                "source_chunk_index":
                    row[
                        "source_chunk_index"
                    ],

                "filtered_content":
                    filtered
            }
        )


allergic_filtered_7_df = (
    pd.DataFrame(
        allergic_filtered_7
    )
)


allergic_context_7 = "\n\n".join(
    allergic_filtered_7_df[
        "filtered_content"
    ].tolist()
)


print(
    "SANITIZED LIMIT-7 EVIDENCE"
)

print("=" * 75)

print(
    "Retained chunks:",
    len(
        allergic_filtered_7_df
    )
)

print(
    "Sanitized characters:",
    len(
        allergic_context_7
    )
)

print(
    "Contains 999:",
    bool(
        re.search(
            r"\b999\b",
            allergic_context_7
        )
    )
)

print(
    "Contains 111:",
    bool(
        re.search(
            r"\b111\b",
            allergic_context_7
        )
    )
)

print(
    "Contains NHS:",
    "nhs"
    in allergic_context_7.lower()
)

SANITIZED LIMIT-7 EVIDENCE
Retained chunks: 6
Sanitized characters: 5270
Contains 999: False
Contains 111: False
Contains NHS: False


In [152]:
allergic_limit7_result = (
    offline_hybrid_rag_answer(
        question=allergic_question,
        category="allergic_reaction",
        limit=7
    )
)


allergic_limit7_audit = (
    audit_generated_answer(
        answer=allergic_limit7_result[
            "answer"
        ],
        category="allergic_reaction"
    )
)


print(
    "ALLERGIC-REACTION LIMIT-7 GENERATION TEST"
)

print("=" * 75)


print(
    "Success:",
    allergic_limit7_result.get(
        "success"
    )
)

print(
    "Answer mode:",
    allergic_limit7_result.get(
        "answer_mode"
    )
)

print(
    "Retrieved chunks:",
    allergic_limit7_result.get(
        "retrieved_chunks"
    )
)

print(
    "Sources:",
    allergic_limit7_result.get(
        "source_ids"
    )
)

print(
    "Foreign numbers:",
    allergic_limit7_audit[
        "foreign_numbers"
    ]
)

print(
    "Localization terms:",
    allergic_limit7_audit[
        "localization_terms"
    ]
)

print(
    "Prompt echo:",
    allergic_limit7_audit[
        "prompt_echo"
    ]
)


print()
print("ANSWER")
print("-" * 75)

print(
    allergic_limit7_result.get(
        "final_output"
    )
)

ALLERGIC-REACTION LIMIT-7 GENERATION TEST
Success: True
Answer mode: local_llm_rag
Retrieved chunks: 6
Sources: ['EXP_017']
Foreign numbers: []
Localization terms: []
Prompt echo: False

ANSWER
---------------------------------------------------------------------------
**FINAL ANSWER:**

If a hiker develops a serious allergic reaction, they should immediately use an adrenaline auto-injector (such as an EpiPen) to relieve symptoms. They should also avoid the specific food, medicine, or insect that caused the reaction. If the reaction does not improve within 5 minutes, they should use a second adrenaline auto-injector and remain still. Do not stand or walk at any time.

**PRINCIPAL ACTION:**
Use an adrenaline auto-injector immediately.

**PRINCIPAL ACTION:**
Avoid the specific food, medicine, or insect that caused the reaction.

**PRINCIPAL ACTION:**
If the reaction does not improve within 5 minutes, use a second adrenaline auto-injector and remain still.

**PRINCIPAL ACTION:**
Do not st

## Chunk-Continuity Improvement

In [153]:
def expand_with_adjacent_chunks(
    retrieval_results,
    neighbor_distance=1
):

    if retrieval_results.empty:
        return retrieval_results.copy()


    chunk_ids = (
        retrieval_results[
            "chunk_id"
        ]
        .drop_duplicates()
        .tolist()
    )


    placeholders = ",".join(
        ["?"] * len(chunk_ids)
    )


    conn = sqlite3.connect(
        RAG_DB_PATH
    )


    seed_chunks = pd.read_sql_query(
        f"""
        SELECT
            chunk_id,
            source_id,
            source_chunk_index
        FROM knowledge_chunks
        WHERE chunk_id IN ({placeholders})
        """,
        conn,
        params=chunk_ids
    )


    expanded_parts = []


    for source_id, group in (
        seed_chunks.groupby(
            "source_id"
        )
    ):

        indices = (
            group[
                "source_chunk_index"
            ]
            .astype(int)
            .tolist()
        )


        wanted_indices = set()


        for index in indices:

            for offset in range(
                -neighbor_distance,
                neighbor_distance + 1
            ):

                candidate = (
                    index + offset
                )

                if candidate >= 0:

                    wanted_indices.add(
                        candidate
                    )


        index_placeholders = ",".join(
            ["?"] * len(
                wanted_indices
            )
        )


        part = pd.read_sql_query(
            f"""
            SELECT
                kc.chunk_id,
                kc.source_id,
                d.organization,
                d.title,
                d.country_context,
                kc.categories,
                kc.content,
                kc.word_count,
                kc.source_chunk_index

            FROM knowledge_chunks kc

            JOIN documents d
                ON d.source_id = kc.source_id

            WHERE kc.source_id = ?
              AND kc.source_chunk_index
                  IN ({index_placeholders})

            ORDER BY
                kc.source_chunk_index
            """,
            conn,
            params=[
                source_id,
                *sorted(
                    wanted_indices
                )
            ]
        )


        expanded_parts.append(
            part
        )


    conn.close()


    if not expanded_parts:

        return pd.DataFrame()


    expanded = pd.concat(
        expanded_parts,
        ignore_index=True
    )


    expanded = (
        expanded
        .drop_duplicates(
            subset=[
                "chunk_id"
            ]
        )
        .sort_values(
            [
                "source_id",
                "source_chunk_index"
            ]
        )
        .reset_index(
            drop=True
        )
    )


    return expanded

In [154]:
allergic_top4 = (
    retrieve_offline_knowledge_v2(
        question=allergic_question,
        category="allergic_reaction",
        limit=4
    )
)


allergic_top4_expanded = (
    expand_with_adjacent_chunks(
        allergic_top4,
        neighbor_distance=1
    )
)


print(
    "TOP-4 + ADJACENT-CHUNK EXPANSION"
)

print("=" * 75)


print(
    "Original retrieved chunks:",
    len(
        allergic_top4
    )
)

print(
    "Expanded chunks:",
    len(
        allergic_top4_expanded
    )
)


display(
    allergic_top4_expanded[
        [
            "chunk_id",
            "source_id",
            "source_chunk_index",
            "word_count"
        ]
    ]
)

TOP-4 + ADJACENT-CHUNK EXPANSION
Original retrieved chunks: 4
Expanded chunks: 7


,chunk_id,source_id,source_chunk_index,word_count
0,CHK_01789,EXP_017,0,37
1,CHK_01790,EXP_017,1,180
2,CHK_01791,EXP_017,2,180
3,CHK_01792,EXP_017,3,180
4,CHK_01793,EXP_017,4,180
5,CHK_01794,EXP_017,5,180
6,CHK_01795,EXP_017,6,65


In [155]:
continuation_recovered = (
    "CHK_01792"
    in allergic_top4_expanded[
        "chunk_id"
    ].tolist()
)


print(
    "Critical adjacent continuation recovered:",
    continuation_recovered
)


assert continuation_recovered


print(
    "Adjacent-chunk continuity test: PASS"
)

Critical adjacent continuation recovered: True
Adjacent-chunk continuity test: PASS


In [156]:
continuity_audit_rows = []


for case in rag_evaluation_cases:

    base_results = (
        retrieve_offline_knowledge_v2(
            question=case[
                "question"
            ],
            category=case[
                "category"
            ],
            limit=4
        )
    )


    expanded_results = (
        expand_with_adjacent_chunks(
            base_results,
            neighbor_distance=1
        )
    )


    continuity_audit_rows.append(
        {
            "category":
                case[
                    "category"
                ],

            "base_chunks":
                len(
                    base_results
                ),

            "expanded_chunks":
                len(
                    expanded_results
                ),

            "additional_chunks":
                (
                    len(
                        expanded_results
                    )
                    -
                    len(
                        base_results
                    )
                ),

            "expanded_words":
                (
                    expanded_results[
                        "word_count"
                    ].sum()
                    if not
                    expanded_results.empty
                    else 0
                )
        }
    )


continuity_audit_df = (
    pd.DataFrame(
        continuity_audit_rows
    )
)


display(
    continuity_audit_df
)


print()
print(
    "Mean base chunks:",
    f"{continuity_audit_df['base_chunks'].mean():.2f}"
)

print(
    "Mean expanded chunks:",
    f"{continuity_audit_df['expanded_chunks'].mean():.2f}"
)

print(
    "Maximum expanded chunks:",
    continuity_audit_df[
        "expanded_chunks"
    ].max()
)

print(
    "Maximum expanded words:",
    continuity_audit_df[
        "expanded_words"
    ].max()
)

,category,base_chunks,expanded_chunks,additional_chunks,expanded_words
0,general_first_aid,3,4,1,720
1,hypothermia,3,3,0,505
2,heat_illness,4,5,1,823
3,heavy_rain,2,2,0,307
4,strong_wind,4,4,0,604
5,water_safety,4,7,3,1217
6,remote_emergency,4,12,8,2145
7,snakebite,4,7,3,1246
8,insect_bites_stings,4,5,1,760
9,fracture_sprain,4,9,5,1602



Mean base chunks: 3.67
Mean expanded chunks: 5.67
Maximum expanded chunks: 12
Maximum expanded words: 2145


## Bounded Adjacent-Chunk Context Assembly


In [157]:
def build_bounded_continuity_context(
    retrieval_results,
    neighbor_distance=1,
    max_words=1200
):

    if retrieval_results.empty:

        return pd.DataFrame()


    expanded = (
        expand_with_adjacent_chunks(
            retrieval_results,
            neighbor_distance=neighbor_distance
        )
    )


    if expanded.empty:

        return expanded


    seed_ids = set(
        retrieval_results[
            "chunk_id"
        ].tolist()
    )


    rank_map = dict(
        zip(
            retrieval_results[
                "chunk_id"
            ],
            retrieval_results[
                "rank"
            ]
        )
    )


    expanded = (
        expanded.copy()
    )


    expanded[
        "is_seed"
    ] = (
        expanded[
            "chunk_id"
        ].isin(
            seed_ids
        )
    )


    expanded[
        "retrieval_rank"
    ] = (
        expanded[
            "chunk_id"
        ].map(
            rank_map
        )
    )


    # -------------------------------------------------
    # First include all original retrieved evidence.
    # -------------------------------------------------

    selected_ids = []

    total_words = 0


    seed_rows = (
        expanded[
            expanded[
                "is_seed"
            ]
        ]
        .sort_values(
            "retrieval_rank"
        )
    )


    for _, row in (
        seed_rows.iterrows()
    ):

        words = int(
            row[
                "word_count"
            ]
        )


        if (
            total_words + words
            <= max_words
        ):

            selected_ids.append(
                row[
                    "chunk_id"
                ]
            )

            total_words += words


    # -------------------------------------------------
    # Then prioritize adjacent chunks nearest to seeds.
    # -------------------------------------------------

    neighbor_candidates = (
        expanded[
            ~expanded[
                "is_seed"
            ]
        ].copy()
    )


    seed_positions = (
        expanded[
            expanded[
                "is_seed"
            ]
        ][
            [
                "source_id",
                "source_chunk_index"
            ]
        ]
    )


    def nearest_seed_distance(row):

        same_source = (
            seed_positions[
                seed_positions[
                    "source_id"
                ]
                ==
                row[
                    "source_id"
                ]
            ]
        )


        if same_source.empty:

            return 999


        return min(
            abs(
                int(
                    row[
                        "source_chunk_index"
                    ]
                )
                -
                int(seed_index)
            )

            for seed_index
            in same_source[
                "source_chunk_index"
            ]
        )


    if not neighbor_candidates.empty:

        neighbor_candidates[
            "seed_distance"
        ] = (
            neighbor_candidates.apply(
                nearest_seed_distance,
                axis=1
            )
        )


        neighbor_candidates = (
            neighbor_candidates
            .sort_values(
                [
                    "seed_distance",
                    "source_id",
                    "source_chunk_index"
                ]
            )
        )


        for _, row in (
            neighbor_candidates.iterrows()
        ):

            words = int(
                row[
                    "word_count"
                ]
            )


            if (
                total_words + words
                <= max_words
            ):

                selected_ids.append(
                    row[
                        "chunk_id"
                    ]
                )

                total_words += words


    # -------------------------------------------------
    # Return selected evidence in document order.
    # -------------------------------------------------

    selected = (
        expanded[
            expanded[
                "chunk_id"
            ].isin(
                selected_ids
            )
        ]
        .sort_values(
            [
                "source_id",
                "source_chunk_index"
            ]
        )
        .reset_index(
            drop=True
        )
    )


    return selected

In [158]:
allergic_bounded_context = (
    build_bounded_continuity_context(
        allergic_top4,
        neighbor_distance=1,
        max_words=1200
    )
)


print(
    "ALLERGIC-REACTION BOUNDED CONTEXT"
)

print("=" * 75)


print(
    "Chunks:",
    len(
        allergic_bounded_context
    )
)

print(
    "Words:",
    allergic_bounded_context[
        "word_count"
    ].sum()
)


display(
    allergic_bounded_context[
        [
            "chunk_id",
            "source_id",
            "source_chunk_index",
            "word_count",
            "is_seed"
        ]
    ]
)


print()
print(
    "Critical continuation included:",
    "CHK_01792"
    in allergic_bounded_context[
        "chunk_id"
    ].tolist()
)

ALLERGIC-REACTION BOUNDED CONTEXT
Chunks: 7
Words: 1002


,chunk_id,source_id,source_chunk_index,word_count,is_seed
0,CHK_01789,EXP_017,0,37,False
1,CHK_01790,EXP_017,1,180,True
2,CHK_01791,EXP_017,2,180,True
3,CHK_01792,EXP_017,3,180,False
4,CHK_01793,EXP_017,4,180,True
5,CHK_01794,EXP_017,5,180,True
6,CHK_01795,EXP_017,6,65,False



Critical continuation included: True


In [159]:
bounded_context_rows = []


for case in rag_evaluation_cases:

    base_results = (
        retrieve_offline_knowledge_v2(
            question=case[
                "question"
            ],
            category=case[
                "category"
            ],
            limit=4
        )
    )


    bounded_results = (
        build_bounded_continuity_context(
            base_results,
            neighbor_distance=1,
            max_words=1200
        )
    )


    bounded_context_rows.append(
        {
            "category":
                case[
                    "category"
                ],

            "retrieved_chunks":
                len(
                    base_results
                ),

            "context_chunks":
                len(
                    bounded_results
                ),

            "context_words":
                int(
                    bounded_results[
                        "word_count"
                    ].sum()
                )
                if not
                bounded_results.empty
                else 0,

            "all_seed_chunks_preserved":
                set(
                    base_results[
                        "chunk_id"
                    ]
                ).issubset(
                    set(
                        bounded_results[
                            "chunk_id"
                        ]
                    )
                )
        }
    )


bounded_context_df = (
    pd.DataFrame(
        bounded_context_rows
    )
)


display(
    bounded_context_df
)


print()
print(
    "Mean context chunks:",
    f"{bounded_context_df['context_chunks'].mean():.2f}"
)

print(
    "Maximum context chunks:",
    bounded_context_df[
        "context_chunks"
    ].max()
)

print(
    "Mean context words:",
    f"{bounded_context_df['context_words'].mean():.2f}"
)

print(
    "Maximum context words:",
    bounded_context_df[
        "context_words"
    ].max()
)

print(
    "All original retrieval chunks preserved:",
    bounded_context_df[
        "all_seed_chunks_preserved"
    ].all()
)

,category,retrieved_chunks,context_chunks,context_words,all_seed_chunks_preserved
0,general_first_aid,3,4,720,True
1,hypothermia,3,3,505,True
2,heat_illness,4,5,823,True
3,heavy_rain,2,2,307,True
4,strong_wind,4,4,604,True
5,water_safety,4,6,1037,True
6,remote_emergency,4,6,1080,True
7,snakebite,4,6,1066,True
8,insect_bites_stings,4,5,760,True
9,fracture_sprain,4,6,1080,True



Mean context chunks: 4.83
Maximum context chunks: 7
Mean context words: 813.72
Maximum context words: 1080
All original retrieval chunks preserved: True


In [160]:
bounded_allergic_filtered = []


for _, row in (
    allergic_bounded_context.iterrows()
):

    filtered = (
        filter_foreign_evidence(
            content=row[
                "content"
            ],
            country_context=row[
                "country_context"
            ]
        )
    )


    if filtered.strip():

        bounded_allergic_filtered.append(
            filtered
        )


bounded_allergic_text = "\n\n".join(
    bounded_allergic_filtered
)


print(
    "BOUNDED ALLERGIC CONTEXT LOCALIZATION"
)

print("=" * 75)


print(
    "Characters:",
    len(
        bounded_allergic_text
    )
)

print(
    "Contains 999:",
    bool(
        re.search(
            r"\b999\b",
            bounded_allergic_text
        )
    )
)

print(
    "Contains 111:",
    bool(
        re.search(
            r"\b111\b",
            bounded_allergic_text
        )
    )
)

print(
    "Contains NHS:",
    "nhs"
    in bounded_allergic_text.lower()
)

print(
    "Contains adrenaline:",
    "adrenaline"
    in bounded_allergic_text.lower()
)

print(
    "Contains auto-injector:",
    "auto-injector"
    in bounded_allergic_text.lower()
)

BOUNDED ALLERGIC CONTEXT LOCALIZATION
Characters: 5312
Contains 999: False
Contains 111: False
Contains NHS: False
Contains adrenaline: True
Contains auto-injector: True


## Integration of Bounded Context Continuity

In [161]:
def prepare_bounded_localized_context(
    retrieval_results,
    neighbor_distance=1,
    max_words=1200
):

    bounded_results = (
        build_bounded_continuity_context(
            retrieval_results,
            neighbor_distance=neighbor_distance,
            max_words=max_words
        )
    )


    localized_parts = []


    for _, row in (
        bounded_results.iterrows()
    ):

        filtered_content = (
            filter_foreign_evidence(
                content=row[
                    "content"
                ],
                country_context=row[
                    "country_context"
                ]
            )
        )


        if filtered_content.strip():

            localized_parts.append(
                {
                    "chunk_id":
                        row[
                            "chunk_id"
                        ],

                    "source_id":
                        row[
                            "source_id"
                        ],

                    "country_context":
                        row[
                            "country_context"
                        ],

                    "content":
                        filtered_content
                }
            )


    localized_df = pd.DataFrame(
        localized_parts
    )


    if localized_df.empty:

        localized_context = ""

    else:

        localized_context = "\n\n".join(
            localized_df[
                "content"
            ].tolist()
        )


    return {
        "bounded_results":
            bounded_results,

        "localized_results":
            localized_df,

        "localized_context":
            localized_context
    }


print(
    "Bounded localized-context preparation ready."
)

Bounded localized-context preparation ready.


In [162]:
allergic_prepared_context = (
    prepare_bounded_localized_context(
        allergic_top4,
        neighbor_distance=1,
        max_words=1200
    )
)


print(
    "PREPARED ALLERGIC-REACTION CONTEXT"
)

print("=" * 75)


print(
    "Bounded chunks:",
    len(
        allergic_prepared_context[
            "bounded_results"
        ]
    )
)

print(
    "Localized chunks:",
    len(
        allergic_prepared_context[
            "localized_results"
        ]
    )
)

print(
    "Context characters:",
    len(
        allergic_prepared_context[
            "localized_context"
        ]
    )
)


print(
    "CHK_01792 included:",
    "CHK_01792"
    in allergic_prepared_context[
        "bounded_results"
    ][
        "chunk_id"
    ].tolist()
)


context_lower = (
    allergic_prepared_context[
        "localized_context"
    ].lower()
)


print(
    "Contains 999:",
    bool(
        re.search(
            r"\b999\b",
            context_lower
        )
    )
)

print(
    "Contains 111:",
    bool(
        re.search(
            r"\b111\b",
            context_lower
        )
    )
)

print(
    "Contains NHS:",
    "nhs"
    in context_lower
)


print(
    "Contains adrenaline:",
    "adrenaline"
    in context_lower
)

PREPARED ALLERGIC-REACTION CONTEXT
Bounded chunks: 7
Localized chunks: 7
Context characters: 5312
CHK_01792 included: True
Contains 999: False
Contains 111: False
Contains NHS: False
Contains adrenaline: True


In [163]:
import inspect


current_rag_function_source = (
    inspect.getsource(
        offline_rag_answer_final
    )
)


print(
    current_rag_function_source
)

def offline_rag_answer_final(
    question,
    category,
    limit=4
):

    results = retrieve_offline_knowledge_v2(
        question=question,
        category=category,
        limit=limit
    )


    if results.empty:

        fallback = (
            "The offline knowledge base does not contain "
            "enough information to answer this safely."
        )

        return {
            "success": False,
            "pipeline_success": True,
            "answer_generated": False,
            "fallback_used": True,
            "question": question,
            "category": category,
            "answer": fallback,
            "final_output": fallback,
            "source_ids": [],
            "retrieved_chunks": 0,
            "country_contexts": [],
            "foreign_numbers_detected": [],
            "generation_time": 0
        }


    context, source_ids = (
        prepare_final_rag_context(
            results
        )
    )


    if not context.strip():

        fall

## Integration of Bounded Context Continuity

In [164]:
def prepare_bounded_localized_context(
    retrieval_results,
    neighbor_distance=1,
    max_words=1200
):
    bounded_results = build_bounded_continuity_context(
        retrieval_results,
        neighbor_distance=neighbor_distance,
        max_words=max_words
    )

    localized_parts = []

    for _, row in bounded_results.iterrows():

        filtered_content = filter_foreign_evidence(
            content=row["content"],
            country_context=row["country_context"]
        )

        if filtered_content.strip():
            localized_parts.append(
                {
                    "chunk_id": row["chunk_id"],
                    "source_id": row["source_id"],
                    "country_context": row["country_context"],
                    "content": filtered_content
                }
            )

    localized_df = pd.DataFrame(localized_parts)

    if localized_df.empty:
        localized_context = ""
    else:
        localized_context = "\n\n".join(
            localized_df["content"].tolist()
        )

    return {
        "bounded_results": bounded_results,
        "localized_results": localized_df,
        "localized_context": localized_context
    }


print("Bounded localized-context preparation ready.")

Bounded localized-context preparation ready.


In [165]:
allergic_prepared_context = prepare_bounded_localized_context(
    allergic_top4,
    neighbor_distance=1,
    max_words=1200
)

bounded_results = allergic_prepared_context["bounded_results"]
localized_results = allergic_prepared_context["localized_results"]
localized_context = allergic_prepared_context["localized_context"]

print("PREPARED ALLERGIC-REACTION CONTEXT")
print("=" * 75)

print("Bounded chunks:", len(bounded_results))
print("Localized chunks:", len(localized_results))
print("Context characters:", len(localized_context))

print(
    "CHK_01792 included:",
    "CHK_01792" in bounded_results["chunk_id"].tolist()
)

print(
    "Context words:",
    int(bounded_results["word_count"].sum())
)

print(
    "Within 1200-word budget:",
    int(bounded_results["word_count"].sum()) <= 1200
)

context_lower = localized_context.lower()

print(
    "Contains 999:",
    bool(re.search(r"\b999\b", localized_context))
)

print(
    "Contains 111:",
    bool(re.search(r"\b111\b", localized_context))
)

print(
    "Contains NHS:",
    "nhs" in context_lower
)

print(
    "Contains adrenaline:",
    "adrenaline" in context_lower
)

print(
    "Contains auto-injector:",
    "auto-injector" in context_lower
)

PREPARED ALLERGIC-REACTION CONTEXT
Bounded chunks: 7
Localized chunks: 7
Context characters: 5312
CHK_01792 included: True
Context words: 1002
Within 1200-word budget: True
Contains 999: False
Contains 111: False
Contains NHS: False
Contains adrenaline: True
Contains auto-injector: True


In [166]:
import inspect

print(
    inspect.getsource(
        offline_rag_answer_final
    )
)

def offline_rag_answer_final(
    question,
    category,
    limit=4
):

    results = retrieve_offline_knowledge_v2(
        question=question,
        category=category,
        limit=limit
    )


    if results.empty:

        fallback = (
            "The offline knowledge base does not contain "
            "enough information to answer this safely."
        )

        return {
            "success": False,
            "pipeline_success": True,
            "answer_generated": False,
            "fallback_used": True,
            "question": question,
            "category": category,
            "answer": fallback,
            "final_output": fallback,
            "source_ids": [],
            "retrieved_chunks": 0,
            "country_contexts": [],
            "foreign_numbers_detected": [],
            "generation_time": 0
        }


    context, source_ids = (
        prepare_final_rag_context(
            results
        )
    )


    if not context.strip():

        fall

In [167]:
source_lines = inspect.getsource(
    offline_rag_answer_final
).splitlines()

print(
    "offline_rag_answer_final lines:",
    len(source_lines)
)

print("=" * 80)

for line_number, line in enumerate(
    source_lines,
    start=1
):
    print(
        f"{line_number:03d}: {line}"
    )

offline_rag_answer_final lines: 190
001: def offline_rag_answer_final(
002:     question,
003:     category,
004:     limit=4
005: ):
006: 
007:     results = retrieve_offline_knowledge_v2(
008:         question=question,
009:         category=category,
010:         limit=limit
011:     )
012: 
013: 
014:     if results.empty:
015: 
016:         fallback = (
017:             "The offline knowledge base does not contain "
018:             "enough information to answer this safely."
019:         )
020: 
021:         return {
022:             "success": False,
023:             "pipeline_success": True,
024:             "answer_generated": False,
025:             "fallback_used": True,
026:             "question": question,
027:             "category": category,
028:             "answer": fallback,
029:             "final_output": fallback,
030:             "source_ids": [],
031:             "retrieved_chunks": 0,
032:             "country_contexts": [],
033:             "foreign_numbers_d

In [168]:
print("PART 1")
print("=" * 80)

for line_number, line in enumerate(
    source_lines[:60],
    start=1
):
    print(
        f"{line_number:03d}: {line}"
    )

PART 1
001: def offline_rag_answer_final(
002:     question,
003:     category,
004:     limit=4
005: ):
006: 
007:     results = retrieve_offline_knowledge_v2(
008:         question=question,
009:         category=category,
010:         limit=limit
011:     )
012: 
013: 
014:     if results.empty:
015: 
016:         fallback = (
017:             "The offline knowledge base does not contain "
018:             "enough information to answer this safely."
019:         )
020: 
021:         return {
022:             "success": False,
023:             "pipeline_success": True,
024:             "answer_generated": False,
025:             "fallback_used": True,
026:             "question": question,
027:             "category": category,
028:             "answer": fallback,
029:             "final_output": fallback,
030:             "source_ids": [],
031:             "retrieved_chunks": 0,
032:             "country_contexts": [],
033:             "foreign_numbers_detected": [],
034:           

In [169]:
print("PART 2")
print("=" * 80)

for line_number, line in enumerate(
    source_lines[60:120],
    start=61
):
    print(
        f"{line_number:03d}: {line}"
    )

PART 2
061:                 fallback
062:                 + "\n\nSources: "
063:                 + ", ".join(source_ids)
064:             ),
065:             "source_ids": source_ids,
066:             "retrieved_chunks": len(results),
067:             "country_contexts": (
068:                 results["country_context"]
069:                 .fillna("Not specified")
070:                 .unique()
071:                 .tolist()
072:             ),
073:             "foreign_numbers_detected": [],
074:             "generation_time": 0
075:         }
076: 
077: 
078:     prompt = build_localized_grounded_prompt(
079:         question=question,
080:         context=context
081:     )
082: 
083: 
084:     generation = generate_with_local_server(
085:         prompt=prompt,
086:         max_tokens=180,
087:         temperature=0.0,
088:         timeout_seconds=60
089:     )
090: 
091: 
092:     if not generation["success"]:
093: 
094:         return {
095:             "success": False,
096:   

In [170]:
print("PART 3")
print("=" * 80)

for line_number, line in enumerate(
    source_lines[120:],
    start=121
):
    print(
        f"{line_number:03d}: {line}"
    )

PART 3
121:         flags=(
122:             re.DOTALL
123:             |
124:             re.IGNORECASE
125:         )
126:     ).strip()
127: 
128: 
129:     fallback_used = (
130:         "INSUFFICIENT_OFFLINE_EVIDENCE"
131:         in answer
132:     )
133: 
134: 
135:     if fallback_used:
136: 
137:         answer = (
138:             "The offline knowledge base does not contain "
139:             "enough information to answer this safely."
140:         )
141: 
142: 
143:     guarded = apply_output_guardrails(
144:         answer=answer,
145:         retrieved_source_ids=source_ids,
146:         category=category
147:     )
148: 
149: 
150:     answer = guarded["answer"]
151: 
152: 
153:     answer_generated = (
154:         bool(answer.strip())
155:         and
156:         not fallback_used
157:     )
158: 
159: 
160:     final_output = (
161:         answer
162:         + "\n\nSources: "
163:         + ", ".join(source_ids)
164:     )
165: 
166: 
167:     return {
168:        

## Integrated Bounded-Continuity RAG Candidate

In [171]:
for start, end, title in [
    (25, 80, "CONTEXT / LOCALIZATION"),
    (75, 120, "PROMPT / GENERATION"),
    (125, 191, "GUARDRAILS / RETURN")
]:
    print()
    print(title)
    print("=" * 80)

    for line_number in range(start, min(end, len(source_lines) + 1)):
        print(
            f"{line_number:03d}: "
            f"{source_lines[line_number - 1]}"
        )


CONTEXT / LOCALIZATION
025:             "fallback_used": True,
026:             "question": question,
027:             "category": category,
028:             "answer": fallback,
029:             "final_output": fallback,
030:             "source_ids": [],
031:             "retrieved_chunks": 0,
032:             "country_contexts": [],
033:             "foreign_numbers_detected": [],
034:             "generation_time": 0
035:         }
036: 
037: 
038:     context, source_ids = (
039:         prepare_final_rag_context(
040:             results
041:         )
042:     )
043: 
044: 
045:     if not context.strip():
046: 
047:         fallback = (
048:             "The offline knowledge base does not contain "
049:             "enough localized information to answer this safely."
050:         )
051: 
052:         return {
053:             "success": False,
054:             "pipeline_success": True,
055:             "answer_generated": False,
056:             "fallback_used": True,
057:   

In [172]:
hybrid_source_lines = inspect.getsource(
    offline_hybrid_rag_answer
).splitlines()

print(
    "offline_hybrid_rag_answer lines:",
    len(hybrid_source_lines)
)

print("=" * 80)

for line_number, line in enumerate(
    hybrid_source_lines,
    start=1
):
    print(
        f"{line_number:03d}: {line}"
    )

offline_hybrid_rag_answer lines: 99
001: def offline_hybrid_rag_answer(
002:     question,
003:     category,
004:     limit=4
005: ):
006: 
007:     # -----------------------------------------
008:     # 1. Retrieve authoritative offline evidence
009:     # -----------------------------------------
010: 
011:     results = retrieve_offline_knowledge_v2(
012:         question=question,
013:         category=category,
014:         limit=limit
015:     )
016: 
017: 
018:     if results.empty:
019: 
020:         fallback = (
021:             "The offline knowledge base does not contain "
022:             "enough information to answer this safely."
023:         )
024: 
025:         return {
026:             "success": False,
027:             "answer": fallback,
028:             "final_output": fallback,
029:             "answer_mode": "fallback",
030:             "source_ids": [],
031:             "retrieved_chunks": 0
032:         }
033: 
034: 
035:     source_ids = (
036:         results

In [173]:
prepare_final_rag_context_original = prepare_final_rag_context

print(
    "Original prepare_final_rag_context saved as "
    "prepare_final_rag_context_original."
)

Original prepare_final_rag_context saved as prepare_final_rag_context_original.


In [174]:
prepare_context_source = inspect.getsource(
    prepare_final_rag_context
)

print(prepare_context_source)

def prepare_final_rag_context(
    results
):

    context_parts = []
    source_ids = []

    for _, row in results.iterrows():

        source_id = row["source_id"]

        if source_id not in source_ids:
            source_ids.append(source_id)


        country = row["country_context"]

        if (
            pd.isna(country)
            or
            not str(country).strip()
        ):
            country = "Not specified"


        content = str(
            row["content"]
        ).strip()


        # Country metadata is used internally to decide
        # whether deterministic localization filtering
        # is required.
        if (
            str(country).strip().lower()
            not in {
                "sri lanka",
                "international",
                "not specified"
            }
        ):

            content = filter_foreign_evidence(
                content=content,
                country_context=country
            )


        if not content.stri

In [175]:
def prepare_final_rag_context_bounded(
    results,
    neighbor_distance=1,
    max_words=1200
):
    """
    Prepare localized generation context using bounded
    adjacent-chunk continuity.

    The original FTS retrieval results remain the primary
    retrieval evidence. Adjacent chunks are added only to
    improve document continuity for generation.
    """

    prepared = prepare_bounded_localized_context(
        retrieval_results=results,
        neighbor_distance=neighbor_distance,
        max_words=max_words
    )

    localized_results = prepared["localized_results"]
    localized_context = prepared["localized_context"]

    if localized_results.empty:
        return "", []

    source_ids = (
        localized_results["source_id"]
        .dropna()
        .drop_duplicates()
        .tolist()
    )

    return localized_context, source_ids


print("Bounded final-context candidate ready.")

Bounded final-context candidate ready.


In [176]:
old_context, old_sources = (
    prepare_final_rag_context(
        allergic_top4
    )
)

bounded_context, bounded_sources = (
    prepare_final_rag_context_bounded(
        allergic_top4,
        neighbor_distance=1,
        max_words=1200
    )
)

print("ALLERGIC CONTEXT COMPARISON")
print("=" * 75)

print("Primary FTS chunks:", len(allergic_top4))
print("Old context characters:", len(old_context))
print("Bounded context characters:", len(bounded_context))
print("Old sources:", old_sources)
print("Bounded sources:", bounded_sources)

print()

print(
    "Bounded contains 999:",
    bool(re.search(r"\b999\b", bounded_context))
)

print(
    "Bounded contains 111:",
    bool(re.search(r"\b111\b", bounded_context))
)

print(
    "Bounded contains NHS:",
    "nhs" in bounded_context.lower()
)

print(
    "Bounded contains adrenaline:",
    "adrenaline" in bounded_context.lower()
)

print(
    "Bounded contains auto-injector:",
    "auto-injector" in bounded_context.lower()
)

ALLERGIC CONTEXT COMPARISON
Primary FTS chunks: 4
Old context characters: 4106
Bounded context characters: 5312
Old sources: ['EXP_017']
Bounded sources: ['EXP_017']

Bounded contains 999: False
Bounded contains 111: False
Bounded contains NHS: False
Bounded contains adrenaline: True
Bounded contains auto-injector: True


In [177]:
prepare_final_rag_context_original = (
    prepare_final_rag_context
)

prepare_final_rag_context = (
    prepare_final_rag_context_bounded
)

print(
    "Bounded context preparation activated."
)

Bounded context preparation activated.


In [178]:
allergic_bounded_result = (
    offline_hybrid_rag_answer(
        question=allergic_question,
        category="allergic_reaction",
        limit=4
    )
)

allergic_bounded_audit = (
    audit_generated_answer(
        answer=allergic_bounded_result["answer"],
        category="allergic_reaction"
    )
)

print(
    "ALLERGIC-REACTION TOP-4 + BOUNDED CONTINUITY TEST"
)

print("=" * 75)

print(
    "Success:",
    allergic_bounded_result.get("success")
)

print(
    "Answer mode:",
    allergic_bounded_result.get("answer_mode")
)

print(
    "Primary FTS retrieved chunks:",
    allergic_bounded_result.get("retrieved_chunks")
)

print(
    "Sources:",
    allergic_bounded_result.get("source_ids")
)

print(
    "Foreign numbers:",
    allergic_bounded_audit["foreign_numbers"]
)

print(
    "Localization terms:",
    allergic_bounded_audit["localization_terms"]
)

print(
    "Prompt echo:",
    allergic_bounded_audit["prompt_echo"]
)

print()
print("ANSWER")
print("-" * 75)

print(
    allergic_bounded_result.get("final_output")
)

ALLERGIC-REACTION TOP-4 + BOUNDED CONTINUITY TEST
Success: False
Answer mode: local_llm_rag
Primary FTS retrieved chunks: 4
Sources: ['EXP_017']
Foreign numbers: []
Localization terms: []
Prompt echo: False

ANSWER
---------------------------------------------------------------------------
The offline knowledge base does not contain enough information to answer this safely.

Sources: EXP_017


## Diagnostic: allergic-reaction bounded-context generation

This section is diagnostic only. It retains top-4 FTS retrieval, uses the existing bounded localized context and existing generation helper, and performs at most one local-server request.

In [182]:
allergic_context_for_generation = bounded_context

search_terms = [
    "What to do if you have anaphylaxis",
    "Use an",
    "adrenaline auto-injector",
    "symptoms have not improved"
]

print("CRITICAL EVIDENCE INSPECTION")
print("=" * 75)

for term in search_terms:

    position = allergic_context_for_generation.lower().find(
        term.lower()
    )

    print()
    print("Search term:", repr(term))
    print("Position:", position)

    if position >= 0:

        start = max(
            0,
            position - 250
        )

        end = min(
            len(allergic_context_for_generation),
            position + 700
        )

        print("-" * 75)
        print(
            allergic_context_for_generation[
                start:end
            ]
        )

CRITICAL EVIDENCE INSPECTION

Search term: 'What to do if you have anaphylaxis'
Position: 1725
---------------------------------------------------------------------------
they may find it difficult to lift their head or focus on your face) You or the person who's unwell may also have a rash that's swollen, raised or itchy. These can be signs of a serious allergic reaction and may need immediate treatment in hospital. What to do if you have anaphylaxis Follow these steps if you think you or someone you're with is having an anaphylactic reaction: - Use an

adrenaline auto-injector (such as an EpiPen) if you have one – instructions are included on the side of the injector. - Lie down – you can raise your legs, and if you're struggling to breathe, raise your shoulders or sit up slowly (if you're pregnant, lie on your left side). - If you have been stung by an insect, try to remove the sting if it's still in the skin. - If your symptoms have not improved after 5 minutes, use a 2nd adrenalin

In [183]:
allergic_exact_prompt = (
    build_localized_grounded_prompt(
        question=allergic_question,
        context=allergic_context_for_generation
    )
)

print("ALLERGIC EXACT PROMPT DIAGNOSTIC")
print("=" * 75)

print(
    "Prompt characters:",
    len(allergic_exact_prompt)
)

print(
    "Prompt words:",
    len(allergic_exact_prompt.split())
)

print(
    "Contains question:",
    allergic_question.lower()
    in allergic_exact_prompt.lower()
)

print(
    "Contains adrenaline:",
    "adrenaline"
    in allergic_exact_prompt.lower()
)

print(
    "Contains auto-injector:",
    "auto-injector"
    in allergic_exact_prompt.lower()
)

print(
    "Contains 999:",
    bool(
        re.search(
            r"\b999\b",
            allergic_exact_prompt
        )
    )
)

print(
    "Contains 111:",
    bool(
        re.search(
            r"\b111\b",
            allergic_exact_prompt
        )
    )
)

ALLERGIC EXACT PROMPT DIAGNOSTIC
Prompt characters: 6709
Prompt words: 1128
Contains question: True
Contains adrenaline: True
Contains auto-injector: True
Contains 999: False
Contains 111: False


In [184]:
generation_helper_candidates = []

for name, obj in list(globals().items()):

    if callable(obj):

        lowered = name.lower()

        if any(
            keyword in lowered
            for keyword in [
                "llama",
                "generate",
                "generation"
            ]
        ):

            generation_helper_candidates.append(
                name
            )

print(
    "GENERATION HELPER CANDIDATES"
)

print("=" * 75)

for name in sorted(
    generation_helper_candidates
):
    print(name)

GENERATION HELPER CANDIDATES
audit_generated_answer
generate_offline_answer
generate_with_local_server
retrieve_for_generation
sanitize_context_for_generation


In [185]:
print(
    inspect.getsource(
        generate_with_local_server
    )
)

def generate_with_local_server(
    prompt,
    max_tokens=220,
    temperature=0.1,
    timeout_seconds=120
):

    endpoint = (
        LLAMA_SERVER_URL
        + "/completion"
    )

    payload = {
        "prompt": prompt,
        "n_predict": max_tokens,
        "temperature": temperature,
        "stream": False
    }

    request = urllib.request.Request(
        endpoint,
        data=json.dumps(
            payload
        ).encode("utf-8"),
        headers={
            "Content-Type":
                "application/json"
        },
        method="POST"
    )

    start_time = (
        time.perf_counter()
    )

    try:

        with urllib.request.urlopen(
            request,
            timeout=timeout_seconds
        ) as response:

            response_body = (
                response
                .read()
                .decode("utf-8")
            )

            data = json.loads(
                response_body
            )

        elapsed = (
            time.p

In [186]:
for line_number in range(
    75,
    118
):
    print(
        f"{line_number:03d}: "
        f"{source_lines[line_number - 1]}"
    )

075:         }
076: 
077: 
078:     prompt = build_localized_grounded_prompt(
079:         question=question,
080:         context=context
081:     )
082: 
083: 
084:     generation = generate_with_local_server(
085:         prompt=prompt,
086:         max_tokens=180,
087:         temperature=0.0,
088:         timeout_seconds=60
089:     )
090: 
091: 
092:     if not generation["success"]:
093: 
094:         return {
095:             "success": False,
096:             "pipeline_success": False,
097:             "answer_generated": False,
098:             "fallback_used": False,
099:             "question": question,
100:             "category": category,
101:             "answer": "Local generation failed.",
102:             "final_output": "Local generation failed.",
103:             "source_ids": source_ids,
104:             "retrieved_chunks": len(results),
105:             "country_contexts": (
106:                 results["country_context"]
107:                 .fillna("Not specif

In [187]:
print(
    inspect.getsource(
        apply_output_guardrails
    )
)

def apply_output_guardrails(
    answer,
    retrieved_source_ids,
    category
):

    cleaned = answer.strip()

    # Remove generated source-ID claims.
    # Provenance is attached separately by the application.
    cleaned = re.sub(
        r"\b(?:SRC|EXP)_\d{3}\b",
        "[retrieved source]",
        cleaned,
        flags=re.IGNORECASE
    )

    # Remove Qwen thinking blocks if present.
    cleaned = re.sub(
        r"<think>.*?</think>",
        "",
        cleaned,
        flags=(
            re.DOTALL
            |
            re.IGNORECASE
        )
    ).strip()

    foreign_numbers_found = []

    # Foreign emergency numbers must not be presented
    # as local Sri Lankan emergency instructions.
    if category != "sri_lanka_emergency":

        for number in (
            FOREIGN_EMERGENCY_NUMBERS
        ):

            if re.search(
                rf"\b{re.escape(number)}\b",
                cleaned
            ):

                foreign_numbers_found.append(
      

## Final Offline RAG Prototype Evaluation

In [188]:
final_rag_test_cases = [
    {
        "category": "hypothermia",
        "question": (
            "What should I do if a hiker shows "
            "signs of hypothermia?"
        )
    },
    {
        "category": "heat_illness",
        "question": (
            "What should I do if a hiker shows "
            "signs of heat illness?"
        )
    },
    {
        "category": "water_safety",
        "question": (
            "What should a hiker do to reduce "
            "water-related safety risks?"
        )
    },
    {
        "category": "lost_navigation",
        "question": (
            "What should I do if I become lost "
            "while hiking?"
        )
    },
    {
        "category": "insect_bites_stings",
        "question": (
            "What should I do if a hiker is "
            "bitten or stung by an insect?"
        )
    },
    {
        "category": "allergic_reaction",
        "question": (
            "What should I do if a hiker develops "
            "a serious allergic reaction?"
        )
    },
    {
        "category": "sri_lanka_emergency",
        "question": (
            "How can I request an emergency "
            "ambulance in Sri Lanka?"
        )
    }
]

print(
    "Final evaluation cases:",
    len(final_rag_test_cases)
)

Final evaluation cases: 7


In [189]:
final_rag_results = []

for case in final_rag_test_cases:

    category = case["category"]
    question = case["question"]

    print()
    print("=" * 75)
    print("CATEGORY:", category)
    print("QUESTION:", question)
    print("=" * 75)

    result = offline_hybrid_rag_answer(
        question=question,
        category=category,
        limit=4
    )

    answer = result.get(
        "final_output",
        result.get("answer", "")
    )

    audit = audit_generated_answer(
        answer=answer,
        category=category
    )

    row = {
        "category": category,
        "question": question,

        "retrieval_success": (
            result.get(
                "retrieved_chunks",
                0
            ) > 0
        ),

        "retrieved_chunks": result.get(
            "retrieved_chunks",
            0
        ),

        "source_ids": ", ".join(
            result.get(
                "source_ids",
                []
            )
        ),

        "answer_mode": result.get(
            "answer_mode",
            "unknown"
        ),

        "answer_generated": result.get(
            "answer_generated",
            result.get(
                "success",
                False
            )
        ),

        "fallback_used": result.get(
            "fallback_used",
            False
        ),

        "foreign_numbers_detected": (
            ", ".join(
                map(
                    str,
                    audit.get(
                        "foreign_numbers",
                        []
                    )
                )
            )
        ),

        "prompt_echo": audit.get(
            "prompt_echo",
            False
        ),

        "generation_time_seconds": result.get(
            "generation_time",
            0
        ),

        "final_output": answer
    }

    final_rag_results.append(row)

    print(
        "Mode:",
        row["answer_mode"]
    )

    print(
        "Retrieved:",
        row["retrieved_chunks"]
    )

    print(
        "Sources:",
        row["source_ids"]
    )

    print(
        "Generated:",
        row["answer_generated"]
    )

    print(
        "Fallback:",
        row["fallback_used"]
    )

    print(
        "Foreign numbers:",
        row[
            "foreign_numbers_detected"
        ]
    )

    print(
        "Prompt echo:",
        row["prompt_echo"]
    )


CATEGORY: hypothermia
QUESTION: What should I do if a hiker shows signs of hypothermia?
Mode: local_llm_rag
Retrieved: 3
Sources: SRC_014
Generated: True
Fallback: False
Foreign numbers: 
Prompt echo: False

CATEGORY: heat_illness
QUESTION: What should I do if a hiker shows signs of heat illness?
Mode: local_llm_rag
Retrieved: 4
Sources: SRC_015
Generated: True
Fallback: False
Foreign numbers: 
Prompt echo: False

CATEGORY: water_safety
QUESTION: What should a hiker do to reduce water-related safety risks?
Mode: local_llm_rag
Retrieved: 4
Sources: SRC_017
Generated: True
Fallback: False
Foreign numbers: 
Prompt echo: False

CATEGORY: lost_navigation
QUESTION: What should I do if I become lost while hiking?
Mode: local_llm_rag
Retrieved: 3
Sources: EXP_011
Generated: True
Fallback: False
Foreign numbers: 
Prompt echo: False

CATEGORY: insect_bites_stings
QUESTION: What should I do if a hiker is bitten or stung by an insect?
Mode: local_llm_rag
Retrieved: 4
Sources: EXP_016
Generated: T

In [190]:
final_rag_df = pd.DataFrame(
    final_rag_results
)

display(
    final_rag_df[
        [
            "category",
            "retrieval_success",
            "retrieved_chunks",
            "source_ids",
            "answer_mode",
            "answer_generated",
            "fallback_used",
            "foreign_numbers_detected",
            "prompt_echo"
        ]
    ]
)

,category,retrieval_success,retrieved_chunks,source_ids,answer_mode,answer_generated,fallback_used,foreign_numbers_detected,prompt_echo
0,hypothermia,True,3,SRC_014,local_llm_rag,True,False,,False
1,heat_illness,True,4,SRC_015,local_llm_rag,True,False,,False
2,water_safety,True,4,SRC_017,local_llm_rag,True,False,,False
3,lost_navigation,True,3,EXP_011,local_llm_rag,True,False,,False
4,insect_bites_stings,True,4,EXP_016,local_llm_rag,True,False,,False
5,allergic_reaction,True,4,EXP_017,local_llm_rag,False,True,,False
6,sri_lanka_emergency,True,4,EXP_015,verified_deterministic,True,False,,False


In [191]:
total_cases = len(final_rag_df)

retrieval_successes = int(
    final_rag_df[
        "retrieval_success"
    ].sum()
)

generated_answers = int(
    final_rag_df[
        "answer_generated"
    ].sum()
)

safe_fallbacks = int(
    final_rag_df[
        "fallback_used"
    ].sum()
)

deterministic_answers = int(
    (
        final_rag_df["answer_mode"]
        ==
        "verified_deterministic"
    ).sum()
)

local_llm_cases = int(
    (
        final_rag_df["answer_mode"]
        ==
        "local_llm_rag"
    ).sum()
)

foreign_leakage_cases = int(
    (
        final_rag_df[
            "foreign_numbers_detected"
        ].str.len() > 0
    ).sum()
)

prompt_echo_cases = int(
    final_rag_df[
        "prompt_echo"
    ].sum()
)


print("FINAL OFFLINE RAG EVALUATION")
print("=" * 75)

print(
    "Total cases:",
    total_cases
)

print(
    "Retrieval successes:",
    f"{retrieval_successes}/{total_cases}"
)

print(
    "Generated answers:",
    f"{generated_answers}/{total_cases}"
)

print(
    "Safe fallbacks:",
    safe_fallbacks
)

print(
    "Verified deterministic answers:",
    deterministic_answers
)

print(
    "Local LLM routed cases:",
    local_llm_cases
)

print(
    "Foreign-number leakage cases:",
    foreign_leakage_cases
)

print(
    "Prompt-echo cases:",
    prompt_echo_cases
)

FINAL OFFLINE RAG EVALUATION
Total cases: 7
Retrieval successes: 7/7
Generated answers: 6/7
Safe fallbacks: 1
Verified deterministic answers: 1
Local LLM routed cases: 6
Foreign-number leakage cases: 0
Prompt-echo cases: 0


In [192]:
from pathlib import Path

final_eval_path = Path(
    "../../rag/evaluation/"
    "final_offline_rag_generation_evaluation.csv"
)

final_eval_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

final_rag_df.to_csv(
    final_eval_path,
    index=False
)

print(
    "Saved:",
    final_eval_path
)

print(
    "Rows:",
    len(final_rag_df)
)

Saved: ../../rag/evaluation/final_offline_rag_generation_evaluation.csv
Rows: 7


## Runtime Module Preparation

In [193]:
runtime_function_names = [
    # Retrieval
    "retrieve_offline_knowledge_v2",

    # Context / localization
    "normalize_country_context",
    "contains_foreign_specific_reference",
    "filter_foreign_evidence",
    "expand_with_adjacent_chunks",
    "build_bounded_continuity_context",
    "prepare_bounded_localized_context",
    "prepare_final_rag_context_bounded",

    # Generation
    "build_localized_grounded_prompt",
    "generate_with_local_server",
    "apply_output_guardrails",

    # Critical deterministic route
    "verify_sri_lanka_1990_evidence",
    "build_verified_emergency_answer",

    # Hybrid
    "offline_rag_answer_final",
    "offline_hybrid_rag_answer",
]

print("RUNTIME FUNCTION CHECK")
print("=" * 70)

for name in runtime_function_names:

    obj = globals().get(name)

    print(
        f"{name:<42}",
        "FOUND" if callable(obj) else "MISSING"
    )

RUNTIME FUNCTION CHECK
retrieve_offline_knowledge_v2              FOUND
normalize_country_context                  FOUND
contains_foreign_specific_reference        FOUND
filter_foreign_evidence                    FOUND
expand_with_adjacent_chunks                FOUND
build_bounded_continuity_context           FOUND
prepare_bounded_localized_context          FOUND
prepare_final_rag_context_bounded          FOUND
build_localized_grounded_prompt            FOUND
generate_with_local_server                 FOUND
apply_output_guardrails                    FOUND
verify_sri_lanka_1990_evidence             FOUND
build_verified_emergency_answer            FOUND
offline_rag_answer_final                   FOUND
offline_hybrid_rag_answer                  FOUND
